# Testing the ATAS with meta-llama/Llama-3.2-1B-Instruct

This project focuses on testing and validating the **effectiveness of ATAS**.

## Requirements

- **pysteer** library


# Importing libraries

In [1]:
import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # nvidia-smi GPU 0 with PCI ordering.
TARGET_GPU_NAME = "Quadro RTX 8000"
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("TRANSFORMERS_NO_TF", "1")
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", "python")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")

CACHE_BASE = os.path.abspath(os.environ.get("ATAS_CACHE_DIR", "./.atas_cache"))
HF_ROOT = os.path.abspath(os.environ.get("ATAS_HF_CACHE_DIR", os.path.join(CACHE_BASE, "hf_cache")))
os.environ.setdefault("HF_HOME", HF_ROOT)
os.environ.setdefault("HF_HUB_CACHE", os.path.join(HF_ROOT, "hub"))
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", os.path.join(HF_ROOT, "hub"))
os.environ.setdefault("HF_DATASETS_CACHE", os.path.join(HF_ROOT, "datasets"))
os.environ.setdefault("HF_XET_CACHE", os.path.join(HF_ROOT, "xet"))
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

PYSTEER_PATH = os.environ.get("PYSTEER_PATH", "pysteer/")
HF_TOKEN = ""
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
MODEL_REVISION = "9213176726f574b556790deb65791e0c5aa438b6"
CLASSIFIER_MODEL_NAME = "distilbert/distilbert-base-uncased"
CLASSIFIER_MODEL_REVISION = "12040accade4e8a0f71eabdb258fecc2e7e948be"
TASK_ROUTER_MODEL_NAME = "google-bert/bert-base-uncased"
TASK_ROUTER_MODEL_REVISION = "86b5e0934494bd15c9632b12f734a8a67f723594"
BERTSCORE_MODEL_NAME = "distilbert/distilroberta-base"
BERTSCORE_MODEL_REVISION = "fb53ab8802853c8e4fbdbcd0529f21fc6f459b2b"
ATAS_USE_LAST_TOKEN_FOR_PROMPT = True
LLAMA_ROBUSTNESS_LAYER = 8
LAYERS_TO_EXTRACT = sorted({LLAMA_ROBUSTNESS_LAYER - 1, LLAMA_ROBUSTNESS_LAYER, LLAMA_ROBUSTNESS_LAYER + 1})

PYPI_PINS = {
    "numpy": "1.26.4",
    "scipy": "1.13.1",
    "scikit-learn": "1.4.2",
    "pandas": "2.2.2",
    "matplotlib": "3.8.4",
    "numexpr": "2.10.2",
    "pyarrow": "25.0.1",
    "huggingface-hub": "0.36.0",
    "datasets": "4.5.0",
    "transformers": "4.57.6",
    "peft": "0.17.1",
    "accelerate": "1.14.0",
    "tokenizers": "0.22.2",
    "safetensors": "0.8.0",
    "protobuf": "5.29.6",
    "sacrebleu": "2.6.0",
    "bert-score": "0.3.13",
    "tqdm": "4.67.3",
}
TORCH_PINS = {"torch": "2.5.1", "torchvision": "0.20.1", "torchaudio": "2.5.1"}
TORCH_INDEX_URL = "https://download.pytorch.org/whl/cu121"

def _installed_version(distribution: str) -> str | None:
    try:
        return version(distribution)
    except PackageNotFoundError:
        return None

def _base_version(value: str | None) -> str | None:
    return None if value is None else value.split("+", 1)[0]

torch_to_install = [
    f"{name}=={wanted}"
    for name, wanted in TORCH_PINS.items()
    if (
        _base_version(_installed_version(name)) != wanted
        or "+cu121" not in str(_installed_version(name))
    )
]
if torch_to_install:
    cmd = [sys.executable, "-m", "pip", "install", "--no-cache-dir", "--upgrade",
           "--index-url", TORCH_INDEX_URL, *torch_to_install]
    print("RUN:", " ".join(cmd))
    subprocess.check_call(cmd)

pypi_to_install = [
    f"{name}=={wanted}"
    for name, wanted in PYPI_PINS.items()
    if _installed_version(name) != wanted
]
if pypi_to_install:
    cmd = [sys.executable, "-m", "pip", "install", "--no-cache-dir", "--upgrade", *pypi_to_install]
    print("RUN:", " ".join(cmd))
    subprocess.check_call(cmd)

if not torch_to_install and not pypi_to_install:
    print("Dependencies already match the pinned environment.")

Dependencies already match the pinned environment.


In [2]:
import contextlib
import csv
import gc
import hashlib
import inspect
import io
import json
import math
import os
import pickle
import random
import subprocess
import sys
import time
import urllib.request
import zipfile
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Audio, load_dataset
from matplotlib.lines import Line2D
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import get_last_checkpoint
from transformers.utils.notebook import NotebookProgressCallback

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "300")
os.environ.setdefault("HF_HOME", os.path.abspath("./.hf"))
os.environ.setdefault("TRANSFORMERS_NO_TF", "1")
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", "python")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")

os.environ["HF_HUB_ETAG_TIMEOUT"] = "300"

def detect_torch_flavor() -> str:
    if "TORCH_FLAVOR" in os.environ:
        return os.environ["TORCH_FLAVOR"].lower()

    try:
        subprocess.check_output(["nvidia-smi"], stderr=subprocess.STDOUT)
    except Exception as exc:
        raise RuntimeError("A CUDA-capable GPU is required to run this notebook.") from exc
    flavor = "cu121"

    os.environ["TORCH_FLAVOR"] = flavor
    return flavor

TORCH_FLAVOR = detect_torch_flavor()
print("[deps] TORCH_FLAVOR:", TORCH_FLAVOR)

PIN_TORCH = {
    "cu121": {
        "torch": "2.5.1",
        "torchvision": "0.20.1",
        "torchaudio": "2.5.1",
        "index_url": "https://download.pytorch.org/whl/cu121",
    },
}

PIN_HF = {
    "huggingface_hub": "0.36.0",
    "datasets": "4.5.0",
    "transformers": "4.57.6",
    "peft": "0.17.1",
    "accelerate": "1.14.0",
    "tokenizers": "0.22.2",
    "safetensors": "0.8.0",
}

PROTOBUF_SPEC = "protobuf==5.29.6"
_RESTART_REQUIRED = False
SEED = 42
# Unified generation budget across Llama, OLMo and Qwen: 1024 prompt + 512 completion.
MAX_SEQUENCE_TOKENS = 1536
MAX_NEW_TOKENS = 512
# Lower bound for the dynamic max_new_tokens budget; generation may still stop earlier at EOS.
MIN_GENERATION_TOKENS = 128
GENERATION_LENGTH_RATIO = 1.10
MAX_PROMPT_TOKENS = 1024
EVAL_GENERATION_BATCH_SIZE = 16
CLASSIFIER_MAX_LENGTH = 512
STORE_N_PREVIEW = 5

def _run(cmd: List[str]) -> None:
    print(">>", " ".join(cmd))
    subprocess.check_call(cmd)

def _pip_install(
    pkgs: Iterable[str],
    *,
    upgrade: bool = True,
    force_reinstall: bool = False,
    index_url: Optional[str] = None,
    quiet: bool = True,
) -> None:
    cmd = [sys.executable, "-m", "pip"]
    if quiet:
        cmd.append("-q")
    cmd += ["install", "--no-cache-dir"]
    if upgrade:
        cmd.append("--upgrade")
    if force_reinstall:
        cmd.append("--force-reinstall")
    if index_url:
        cmd += ["--index-url", index_url]
    cmd += list(pkgs)
    _run(cmd)

def _pip_uninstall(pkgs: Iterable[str], quiet: bool = True) -> None:
    cmd = [sys.executable, "-m", "pip"]
    if quiet:
        cmd.append("-q")
    cmd += ["uninstall", "-y", *list(pkgs)]
    _run(cmd)

def _torch_ok() -> bool:
    try:
        import torch
        import torch.distributed.tensor

        is_cuda_build = torch.version.cuda is not None
        has_device = torch.cuda.is_available()
        if not (is_cuda_build and has_device):
            print(
                "[deps] Torch present but CUDA not usable:",
                "torch.version.cuda=", torch.version.cuda,
                "cuda_available=", torch.cuda.is_available(),
            )
            return False
        return True
    except Exception as e:
        print("[deps] Torch DTensor import check failed:", repr(e))
        return False

def _sanity_check_imports() -> bool:
    try:
        import datasets
        import huggingface_hub
        import tokenizers
        import transformers
        return True
    except Exception as e:
        print("[deps] Sanity import failed:", repr(e))
        return False

def _repair_torch_stack() -> None:
    global _RESTART_REQUIRED

    spec = PIN_TORCH[TORCH_FLAVOR]
    print(f"[deps] Repairing torch stack using flavor={TORCH_FLAVOR} -> {spec}")

    try:
        _pip_uninstall(["torch", "torchvision", "torchaudio"], quiet=False)
    except Exception as e:
        print("[deps] Uninstall torch stack warning:", repr(e))

    _pip_install(
        [
            f"torch=={spec['torch']}",
            f"torchvision=={spec['torchvision']}",
            f"torchaudio=={spec['torchaudio']}",
        ],
        upgrade=False,
        force_reinstall=False,
        index_url=spec["index_url"],
        quiet=False,
    )
    _RESTART_REQUIRED = True

def _repair_hf_stack() -> None:
    global _RESTART_REQUIRED

    print("[deps] Repairing HF stack (hub/datasets/transformers/peft/accelerate)...")

    try:
        _pip_uninstall(
            ["transformers", "tokenizers", "datasets", "huggingface_hub", "peft", "accelerate"],
            quiet=False,
        )
    except Exception as e:
        print("[deps] Uninstall HF stack warning:", repr(e))

    pkgs = [
        PROTOBUF_SPEC,
        f"huggingface_hub=={PIN_HF['huggingface_hub']}",
        f"datasets=={PIN_HF['datasets']}",
        f"transformers=={PIN_HF['transformers']}",
        f"peft=={PIN_HF['peft']}",
        f"accelerate=={PIN_HF['accelerate']}",
        f"safetensors=={PIN_HF['safetensors']}",
        f"tokenizers=={PIN_HF['tokenizers']}",
    ]
    _pip_install(pkgs, upgrade=True, force_reinstall=False, quiet=False)
    _RESTART_REQUIRED = True

if not _torch_ok():
    _repair_torch_stack()

if not _sanity_check_imports():
    _repair_hf_stack()

if _RESTART_REQUIRED:
    print("\n[deps] RESTART REQUIRED: I installed/uninstalled packages.")
    print("[deps] On Kaggle: Run -> Restart Session, then rerun the notebook from the beginning.\n")
    raise SystemExit("Restart Session required to avoid 'mixed' HF modules in memory.")

try:
    from bert_score import BERTScorer
    _HAS_BERTSCORE = True
except Exception as e:
    print("[deps] BERTScore not available:", repr(e))
    BERTScorer = None
    _HAS_BERTSCORE = False

print("[deps] OK")
print("[deps] torch:", torch.__version__, "cuda:", torch.version.cuda, "cuda_available:", torch.cuda.is_available())

try:
    from huggingface_hub import login, snapshot_download
    _HAS_HF_LOGIN = True
except Exception:
    _HAS_HF_LOGIN = False

if _HAS_HF_LOGIN and HF_TOKEN and str(HF_TOKEN).strip() not in {"", "---"}:
    login(token=HF_TOKEN)
else:
    print("HF login skipped (HF_TOKEN missing/placeholder or huggingface_hub not available).")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)
if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

def resolve_pysteer_path(path: Optional[str]) -> str:
    if path and os.path.isdir(path):
        return path

    candidates = [str(Path("/kaggle/working/pysteer"))]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        candidates.extend(str(p) for p in sorted(kaggle_input.glob("*/pysteer")) if p.is_dir())
    candidates.extend(
        str(Path("/") / d / "pysteer")
        for d in sorted(os.listdir("/"))
        if (Path("/") / d / "pysteer").is_dir()
    )

    candidates = [candidate for candidate in candidates if os.path.isdir(candidate)]
    if candidates:
        found = candidates[0]
        print("PYSTEER_PATH auto-detected:", found)
        return found

    raise FileNotFoundError(
        f"Can't find pysteer at: {path}\n"
        "Update PYSTEER_PATH to your path (or set the PYSTEER_PATH env var)."
    )

PYSTEER_PATH = resolve_pysteer_path(PYSTEER_PATH)

if PYSTEER_PATH not in sys.path:
    sys.path.insert(0, PYSTEER_PATH)

def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def _sha256_python_tree(root: Path) -> str:
    digest = hashlib.sha256()
    for path in sorted(root.rglob("*.py")):
        digest.update(path.relative_to(root).as_posix().encode("utf-8"))
        digest.update(path.read_bytes())
    return digest.hexdigest()

PYSTEER_SOURCE_SHA256 = _sha256_python_tree(Path(PYSTEER_PATH))
PROMPT_REGISTRY_SHA256 = _sha256_file(Path("task_prompt_variants.py"))
print("pysteer source SHA256:", PYSTEER_SOURCE_SHA256)
print("prompt registry SHA256:", PROMPT_REGISTRY_SHA256)

from enums.ApplyFromModeEnum import ApplyFromModeEnum
from enums.MediatorTypeEnum import MediatorTypeEnum
from enums.ModelTypeEnum import ModelTypeEnum
from enums.TaskTypeEnum import TaskTypeEnum
from activation_manager.SteeredModelWrapper import ATAS_REFUSAL_TEXT
from executor import Executor
from prompt_generator.Llama3Point1PromptGenerator import Llama3Point1PromptGenerator
from utils.ModelUtils import ModelUtils

def cleanup_memory(cuda: bool = True) -> None:
    gc.collect()
    if cuda and torch.cuda.is_available():
        torch.cuda.empty_cache()
        if hasattr(torch.cuda, "ipc_collect"):
            torch.cuda.ipc_collect()

def prune_eval_result(res: dict, keep_n: int = STORE_N_PREVIEW) -> dict:
    out = dict(res)
    if keep_n and isinstance(out.get("rewrites"), list):
        out["rewrites_preview"] = out["rewrites"][:keep_n]
    for k in ("rewrites", "pred_orig", "pred_rewrite"):
        out.pop(k, None)
    return out

[deps] TORCH_FLAVOR: cu121
[deps] OK
[deps] torch: 2.5.1+cu121 cuda: 12.1 cuda_available: True
HF login skipped (HF_TOKEN missing/placeholder or huggingface_hub not available).
pysteer source SHA256: 5cc7e92cdb3405f35c0ac20a907502d822100eab1f5be754064898de6467a03b
prompt registry SHA256: 95148c5aff0e95a3d1477976357a4a46dbe294e29dc452565a10f22ace56de46


In [3]:
import warnings
warnings.filterwarnings("ignore")

from transformers import logging
logging.set_verbosity_error()

# Functions

In [4]:
def _count_forward_hooks(model) -> int:
    tot = 0
    for m in model.modules():
        tot += len(getattr(m, "_forward_hooks", {}))
        tot += len(getattr(m, "_forward_pre_hooks", {}))
    return tot

def _assert_no_forward_hooks(model, where: str = "") -> None:
    n = _count_forward_hooks(model)
    if n != 0:
        raise RuntimeError(
            f"[HOOK LEAK] Found {n} forward/pre-forward hooks on base model {where}. "
            "This can contaminate baseline generations."
        )

In [5]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='binary', zero_division=0)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    return {"acc": acc, "f1": f1, "precision": p, "recall": r}

_BERTSCORER_CACHE: Dict[Tuple[str, str], Any] = {}
_BERTSCORE_MODEL_PATH: Optional[str] = None

def compute_bertscore(preds: List[str], refs: List[str]) -> float:
    global _BERTSCORE_MODEL_PATH
    scorer_cls = BERTScorer
    if (not preds) or (not _HAS_BERTSCORE) or (scorer_cls is None):
        return float("nan")

    lang = "en"

    if torch.cuda.is_available():
        device = "cuda"
        bs = 64
    elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        device = "mps"
        bs = 32
    else:
        device = "cpu"
        bs = 8

    if _BERTSCORE_MODEL_PATH is None:
        _BERTSCORE_MODEL_PATH = snapshot_download(
            repo_id=BERTSCORE_MODEL_NAME,
            revision=BERTSCORE_MODEL_REVISION,
        )

    cache_key = (_BERTSCORE_MODEL_PATH, device)
    scorer = _BERTSCORER_CACHE.get(cache_key)
    if scorer is None:
        scorer = scorer_cls(
            lang=lang,
            model_type=_BERTSCORE_MODEL_PATH,
            num_layers=6,
            rescale_with_baseline=False,
            device=device,
            batch_size=bs,
        )
        _BERTSCORER_CACHE[cache_key] = scorer

    _, _, F1 = scorer.score(
        cands=[p if isinstance(p, str) else "" for p in preds],
        refs=[r if isinstance(r, str) else "" for r in refs],
        batch_size=bs,
    )
    return float(F1.mean().item())

prompt_generator = Llama3Point1PromptGenerator(ModelTypeEnum.INSTRUCT)

def build_llama32_prompt(system: str, user: str, who_you_are: str = "") -> str:
    return prompt_generator.generate_prompt(
        system=system,
        user=user,
        who_you_are=who_you_are,
    )

In [6]:
GENERATION_KWARGS = dict(
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=False,
    num_beams=1,
)

GENERATION_KWARGS.pop("temperature", None)
GENERATION_KWARGS.pop("top_p", None)

In [7]:
def get_model_device(m: torch.nn.Module) -> torch.device:
    base = getattr(m, "base_model", m)

    try:
        if hasattr(base, "get_input_embeddings"):
            emb = base.get_input_embeddings()
            if emb is not None:
                return next(emb.parameters()).device
    except Exception:
        pass

    dm = getattr(base, "hf_device_map", None)
    if isinstance(dm, dict) and dm:
        preferred = ("embed_tokens", "wte", "embeddings", "tok_embeddings", "word_embeddings")
        for k, v in dm.items():
            if any(s in str(k) for s in preferred):
                try:
                    if isinstance(v, str):
                        return torch.device(v)
                    if isinstance(v, int):
                        return torch.device(f"cuda:{v}") if torch.cuda.is_available() else torch.device("cpu")
                except Exception:
                    break

        for v in dm.values():
            try:
                dv = torch.device(v) if isinstance(v, str) else torch.device(f"cuda:{v}")
                if dv.type == "cuda":
                    return dv
            except Exception:
                continue

    return next(base.parameters()).device

def batched_generate(
        model_obj,
        tokenizer,
        prompts,
        batch_size: int = EVAL_GENERATION_BATCH_SIZE,
        max_new_tokens: int = MAX_NEW_TOKENS,
        do_sample: bool = False,
        top_p: float = 0.9,
        temperature: float = 0.8,
        max_prompt_tokens: int | None = MAX_PROMPT_TOKENS,
        empty_cache_every: int = 0,
        gc_every: int = 50,
        generation_kwargs: Optional[Dict[str, Any]] = None,
        forced_task_ids: Optional[List[int]] = None,
        **extra_generate_kwargs,
) -> List[str]:
    if not prompts:
        return []
    all_outputs: List[str] = []
    if forced_task_ids is not None and len(forced_task_ids) != len(prompts):
        raise ValueError("forced_task_ids must have the same length as prompts.")

    original_prompts = list(prompts)
    length_kwargs = dict(add_special_tokens=False, truncation=True, padding=False)
    if max_prompt_tokens is not None:
        length_kwargs["max_length"] = int(max_prompt_tokens)
    length_enc = tokenizer(original_prompts, **length_kwargs)
    prompt_lengths = [len(ids) for ids in length_enc["input_ids"]]
    sort_order = sorted(range(len(original_prompts)), key=lambda idx: prompt_lengths[idx])
    prompts = [original_prompts[idx] for idx in sort_order]
    if forced_task_ids is not None:
        forced_task_ids = [forced_task_ids[idx] for idx in sort_order]
    del length_enc, prompt_lengths

    device = get_model_device(model_obj)
    base = {}
    if generation_kwargs is not None:
        base.update(dict(generation_kwargs))
    else:
        base.update(dict(GENERATION_KWARGS))

    base["max_new_tokens"] = int(max_new_tokens)
    base["do_sample"] = bool(do_sample)
    base.update(extra_generate_kwargs)

    if base["do_sample"]:
        base.setdefault("top_p", float(top_p))
        base.setdefault("temperature", float(temperature))
    else:
        base.pop("top_p", None)
        base.pop("temperature", None)

    base.update(extra_generate_kwargs)

    tok_kwargs = dict(return_tensors="pt", padding=True, truncation=True, add_special_tokens=False)
    if max_prompt_tokens is not None:
        tok_kwargs["max_length"] = int(max_prompt_tokens)

    for batch_idx, i in enumerate(range(0, len(prompts), batch_size), start=1):
        batch = prompts[i:i + batch_size]

        enc = tokenizer(batch, **tok_kwargs)
        enc = {k: v.to(device) for k, v in enc.items()}
        seq_len = enc["input_ids"].shape[1]
        prompt_len = int(enc["attention_mask"].sum(dim=1).max().item())
        available_new_tokens = MAX_SEQUENCE_TOKENS - seq_len
        dynamic_max_new_tokens = min(
            int(base["max_new_tokens"]),
            max(MIN_GENERATION_TOKENS, int(math.ceil(prompt_len * GENERATION_LENGTH_RATIO))),
            int(available_new_tokens),
        )
        if dynamic_max_new_tokens <= 0:
            raise ValueError(
                f"Prompt batch length {seq_len} leaves no room in the {MAX_SEQUENCE_TOKENS}-token context."
            )
        batch_generation_kwargs = dict(base)
        batch_generation_kwargs["max_new_tokens"] = dynamic_max_new_tokens

        strategy = getattr(model_obj, "strategy", None)
        forced_batch = None if forced_task_ids is None else forced_task_ids[i:i + len(batch)]
        if forced_batch is not None:
            if strategy is None or not hasattr(strategy, "set_forced_task_ids"):
                raise TypeError("The active steering strategy does not support forced task routing.")
            strategy.set_forced_task_ids(forced_batch)

        generation_oom = None
        try:
            try:
                with torch.inference_mode():
                    out = model_obj.generate(**enc, **batch_generation_kwargs)
            except RuntimeError as exc:
                if "out of memory" not in str(exc).lower():
                    raise
                generation_oom = exc
        finally:
            if forced_batch is not None:
                strategy.clear_forced_task_ids()

        if generation_oom is not None:
            del enc
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            if len(batch) <= 1:
                raise generation_oom
            fallback_batch_size = max(1, len(batch) // 2)
            print(f"[generation] CUDA OOM at batch={len(batch)}; retrying with batch={fallback_batch_size}.")
            remaining_forced_ids = None if forced_task_ids is None else forced_task_ids[i:]
            fallback_outputs = batched_generate(
                model_obj=model_obj,
                tokenizer=tokenizer,
                prompts=prompts[i:],
                batch_size=fallback_batch_size,
                max_new_tokens=int(base["max_new_tokens"]),
                do_sample=bool(base["do_sample"]),
                max_prompt_tokens=max_prompt_tokens,
                empty_cache_every=empty_cache_every,
                gc_every=gc_every,
                generation_kwargs=base,
                forced_task_ids=remaining_forced_ids,
            )
            all_outputs.extend(fallback_outputs)
            break

        completion_ids = out[:, seq_len:]
        decoded = tokenizer.batch_decode(completion_ids, skip_special_tokens=True)
        all_outputs.extend(decoded)

        del out, completion_ids, decoded, enc, batch

        if torch.cuda.is_available() and empty_cache_every and (batch_idx % empty_cache_every == 0):
            torch.cuda.empty_cache()
            if hasattr(torch.cuda, "ipc_collect"):
                torch.cuda.ipc_collect()

        if gc_every and (batch_idx % gc_every == 0):
            gc.collect()

    restored_outputs = [""] * len(all_outputs)
    for sorted_pos, original_idx in enumerate(sort_order):
        restored_outputs[original_idx] = all_outputs[sorted_pos]
    return restored_outputs

# Datasets


In [8]:
TASK_IDS = {
    "libritts_text_original_to_normalized": 0,
    "paradetox_non_toxic": 1,
    "imdb_positive_sentiment": 2,
    "stanford_politeness_polite": 3,
    "authormix_woolf_style": 4,
    "setfit_subj_objective": 5,
}


def _pick_writable_base():
    candidates = [
        os.environ.get("SCRATCH"),
        os.environ.get("LOCAL_SCRATCH"),
        os.environ.get("TMPDIR"),
        f"/scratch/{os.environ.get('USER', '')}",
        f"/tmp/{os.environ.get('USER', '')}",
        str(Path.home() / ".cache"),
    ]
    for base in candidates:
        if not base:
            continue
        try:
            os.makedirs(base, exist_ok=True)
            test = os.path.join(base, ".write_test")
            with open(test, "w", encoding="utf-8") as f:
                f.write("ok")
            os.remove(test)
            return base
        except Exception:
            pass
    raise RuntimeError("No writable directory found for caches.")


CACHE_BASE = os.path.abspath(os.environ.get("ATAS_CACHE_DIR", "./.atas_cache"))
HF_ROOT = os.path.abspath(os.environ.get("ATAS_HF_CACHE_DIR", os.path.join(CACHE_BASE, "hf_cache")))
LIBRITTS_ROOT = os.path.join(CACHE_BASE, "libritts_r_slim_2cols")
ARTIFACT_DIR = Path(CACHE_BASE) / "atas_artifacts"
CLF_CACHE_ROOT = ARTIFACT_DIR / "classifiers"

for p in [
    HF_ROOT,
    os.path.join(HF_ROOT, "hub"),
    os.path.join(HF_ROOT, "datasets"),
    os.path.join(HF_ROOT, "xet"),
    LIBRITTS_ROOT,
    str(ARTIFACT_DIR),
    str(CLF_CACHE_ROOT),
]:
    os.makedirs(p, exist_ok=True)

os.environ["HF_HOME"] = HF_ROOT
os.environ["HF_HUB_CACHE"] = os.path.join(HF_ROOT, "hub")
OS_HUB = os.path.join(HF_ROOT, "hub")
os.environ["HUGGINGFACE_HUB_CACHE"] = OS_HUB
os.environ["HF_DATASETS_CACHE"] = os.path.join(HF_ROOT, "datasets")
os.environ["HF_XET_CACHE"] = os.path.join(HF_ROOT, "xet")
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["LIBRITTS_SLIM_CACHE_DIR"] = LIBRITTS_ROOT

print("CACHE_BASE =", CACHE_BASE)
print("HF_DATASETS_CACHE =", os.environ["HF_DATASETS_CACHE"])
print("LIBRITTS_SLIM_CACHE_DIR =", os.environ["LIBRITTS_SLIM_CACHE_DIR"])
print("ARTIFACT_DIR =", ARTIFACT_DIR)

CODE_VERSION = "atas-effectiveness-v6-three-layer-patience4"
STEERING_TRAIN_PER_CLASS = 50


CACHE_BASE = /home/ubuntu/ATAS/.atas_cache
HF_DATASETS_CACHE = /home/ubuntu/ATAS/.atas_cache/hf_cache/datasets
LIBRITTS_SLIM_CACHE_DIR = /home/ubuntu/ATAS/.atas_cache/libritts_r_slim_2cols
ARTIFACT_DIR = /home/ubuntu/ATAS/.atas_cache/atas_artifacts


In [9]:
import pandas as pd
from itertools import islice

SEED = 42

REWRITE_TARGET_LABEL = 1

LIBRITTS_SOURCE_SPLITS = {
    "train": ["train.clean.100", "train.clean.360", "train.other.500"],
    "validation": ["dev.clean", "dev.other"],
    "test": ["test.clean", "test.other"],
}
LIBRITTS_SLIM_CACHE_VERSION = 1

LIBRITTS_DATASET_NAME = "mythicinfinity/libritts_r"
IMDB_DATASET_NAME = "stanfordnlp/imdb"
STANFORD_POLITENESS_DATASET_NAME = "Cleanlab/stanford-politeness"
PARADETOX_TOXICITY_DATASET_NAME = "s-nlp/en_paradetox_toxicity"
AUTHORMIX_DATASET_NAME = "hallisky/AuthorMix"
SETFIT_SUBJ_DATASET_NAME = "SetFit/subj"
DATASET_REVISIONS = {
    LIBRITTS_DATASET_NAME: "0d1718db351207d5a493135a62d5ed8cc4733830",
    IMDB_DATASET_NAME: "e6281661ce1c48d982bc483cf8a173c1bbeb5d31",
    STANFORD_POLITENESS_DATASET_NAME: "9fdefb9b4206e12647be56a5ceb48ca898c958a2",
    PARADETOX_TOXICITY_DATASET_NAME: "01c05c8bb326b3c166648723d073bb0724ef458d",
    AUTHORMIX_DATASET_NAME: "b8cabb806a1ee4ce5c07bb31b7702664ff91e891",
    SETFIT_SUBJ_DATASET_NAME: "f3c1162e678417f664d76b21864fdb87b0615fcf",
}

ORIGINAL_SPLIT_ORDER = ("train", "validation", "test")
CLASSIFIER_VAL_COUNT = 400
CLASSIFIER_TEST_FRACTION = 0.20
PAIRED_EVAL_TASKS = {"libritts_text_original_to_normalized"}

ACTIVE_TASK_IDS = dict(TASK_IDS)
SKIPPED_TASK_IDS = {}


def _truncate(s: str, max_chars: int = 4000) -> str:
    s = "" if s is None else str(s)
    s = " ".join(s.split())
    return s[:max_chars]

def _first_present(d: dict, keys: List[str]) -> Optional[str]:
    for k in keys:
        if k in d and d[k] is not None:
            return k
    return None

def _shuffle_df(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy().reset_index(drop=True)
    return df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

def _empty_binary_text_df() -> pd.DataFrame:
    return pd.DataFrame(columns=["text", "label"])

def _empty_paired_rewrite_df() -> pd.DataFrame:
    return pd.DataFrame(
        columns=["raw_text", "target_text", "dataset", "task_id", "gold_label", "target_label", "reference"]
    )

def _prepare_binary_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["text"] = df["text"].astype(str).map(_truncate)
    df["label"] = df["label"].astype(int)
    df = df[df["text"].str.len() > 0].reset_index(drop=True)
    df = df.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)

    parts = []
    for lab in (0, 1):
        part = df[df["label"] == lab].copy()
        if not part.empty:
            parts.append(part)

    if not parts:
        return _empty_binary_text_df()

    out = pd.concat(parts, ignore_index=True)
    return _shuffle_df(out)

def _stream_binary_df(
    ds,
    emit: Callable[[dict], Iterable[Tuple[str, Optional[int]]]],
) -> pd.DataFrame:
    rows: List[dict] = []

    for ex in ds:
        for text, lab in emit(ex):
            if lab is None:
                continue
            lab = int(lab)
            if lab not in (0, 1):
                continue

            text = _truncate(text)
            if not text:
                continue

            rows.append({"text": text, "label": lab})

    if not rows:
        return _empty_binary_text_df()

    return _prepare_binary_df(pd.DataFrame(rows))

def _iter_dataset_streams(streams):
    for stream in streams:
        for ex in stream:
            yield ex

def _compute_split_counts(
    total_count: int,
    *,
    val_count: int = CLASSIFIER_VAL_COUNT,
    test_frac: float = CLASSIFIER_TEST_FRACTION,
) -> Tuple[int, int, int]:
    total_count = int(total_count)
    val_count = int(val_count)

    if total_count <= 0:
        return 0, 0, 0

    if val_count < 0:
        raise ValueError("val_count must be >= 0")
    if not (0.0 <= float(test_frac) < 1.0):
        raise ValueError("test_frac must be in [0, 1)")

    n_val = min(val_count, total_count)

    n_test = int(math.floor(total_count * float(test_frac)))

    max_test_allowed = max(0, total_count - n_val)
    n_test = min(n_test, max_test_allowed)

    n_train = total_count - n_val - n_test

    if n_train < 0:
        raise ValueError("Invalid split sizes: train became negative.")

    return n_train, n_val, n_test

def _split_binary_pool_random(
    df: pd.DataFrame,
    *,
    val_count: int = CLASSIFIER_VAL_COUNT,
    test_frac: float = CLASSIFIER_TEST_FRACTION,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    df = _prepare_binary_df(df)
    if df.empty:
        empty = _empty_binary_text_df()
        return empty, empty, empty

    df = _shuffle_df(df)
    n_train, n_val, n_test = _compute_split_counts(
        len(df),
        val_count=val_count,
        test_frac=test_frac,
    )

    train_df = df.iloc[:n_train].reset_index(drop=True)
    val_df = df.iloc[n_train:n_train + n_val].reset_index(drop=True)
    test_df = df.iloc[n_train + n_val:n_train + n_val + n_test].reset_index(drop=True)

    for split_name, split_df in (("train", train_df), ("validation", val_df), ("test", test_df)):
        labels = set(split_df["label"].astype(int).unique().tolist())
        if labels != {0, 1}:
            raise RuntimeError(
                f"Invalid random split for binary task: {split_name} must contain both classes 0 and 1, "
                f"found {sorted(labels)}."
            )

    return train_df, val_df, test_df

def _split_paired_pool(
    pair_df: pd.DataFrame,
    *,
    val_count: int = CLASSIFIER_VAL_COUNT,
    test_frac: float = CLASSIFIER_TEST_FRACTION,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    pair_df = pair_df.copy().reset_index(drop=True)
    if pair_df.empty:
        empty = _empty_paired_rewrite_df()
        return empty, empty, empty

    pair_df = _shuffle_df(pair_df)
    n_train, n_val, n_test = _compute_split_counts(
        len(pair_df),
        val_count=val_count,
        test_frac=test_frac,
    )

    train_df = pair_df.iloc[:n_train].reset_index(drop=True)
    val_df = pair_df.iloc[n_train:n_train + n_val].reset_index(drop=True)
    test_df = pair_df.iloc[n_train + n_val:n_train + n_val + n_test].reset_index(drop=True)
    return train_df, val_df, test_df

def _expand_paired_split_to_binary_df(pair_df: pd.DataFrame) -> pd.DataFrame:
    if pair_df is None or pair_df.empty:
        return _empty_binary_text_df()

    part0 = pd.DataFrame({
        "text": pair_df["raw_text"].astype(str),
        "label": np.zeros(len(pair_df), dtype=int),
    })
    part1 = pd.DataFrame({
        "text": pair_df["target_text"].astype(str),
        "label": np.ones(len(pair_df), dtype=int),
    })

    return _prepare_binary_df(pd.concat([part0, part1], ignore_index=True))

def _default_libritts_slim_cache_dir() -> str:
    explicit = (
            globals().get("LIBRITTS_SLIM_CACHE_DIR")
            or os.environ.get("LIBRITTS_SLIM_CACHE_DIR")
            or os.environ.get("PYSTEER_LIBRITTS_SLIM_CACHE_DIR")
    )
    if explicit:
        return os.path.abspath(os.path.expanduser(str(explicit)))

    candidates = []

    for env_name in ("SCRATCH", "LOCAL_SCRATCH", "TMPDIR"):
        val = os.environ.get(env_name)
        if val:
            candidates.append(os.path.abspath(os.path.expanduser(val)))

    user_name = os.environ.get("USER") or os.path.basename(os.path.expanduser("~"))
    candidates.extend([
        f"/scratch/{user_name}",
        f"/tmp/{user_name}",
        os.path.expanduser("~/.cache/pysteer"),
    ])

    for base in candidates:
        try:
            os.makedirs(base, exist_ok=True)
            probe = os.path.join(base, ".pysteer_write_test")
            with open(probe, "w", encoding="utf-8") as f:
                f.write("ok")
            os.remove(probe)
            return os.path.join(base, "libritts_r_slim_2cols")
        except Exception:
            continue

    return os.path.expanduser("~/libritts_r_slim_2cols")


LIBRITTS_SLIM_CACHE_DIR = _default_libritts_slim_cache_dir()
LIBRITTS_DATASET_NAME = "mythicinfinity/libritts_r"
IMDB_DATASET_NAME = "stanfordnlp/imdb"
STANFORD_POLITENESS_DATASET_NAME = "Cleanlab/stanford-politeness"
PARADETOX_TOXICITY_DATASET_NAME = "s-nlp/en_paradetox_toxicity"
AUTHORMIX_DATASET_NAME = "hallisky/AuthorMix"
SETFIT_SUBJ_DATASET_NAME = "SetFit/subj"
ALPACA_DATASET_NAME = "tatsu-lab/alpaca"

ORIGINAL_SPLIT_ORDER = ("train", "validation", "test")
CLASSIFIER_VAL_COUNT = 400
CLASSIFIER_TEST_FRACTION = 0.20
PAIRED_EVAL_TASKS = {"libritts_text_original_to_normalized"}

ACTIVE_TASK_IDS = dict(TASK_IDS)
SKIPPED_TASK_IDS = {}

LIBRITTS_SLIM_CACHE_DIR = _default_libritts_slim_cache_dir()

def _libritts_config_for_hf_split(hf_split: str) -> str:
    if ".clean" in hf_split:
        return "clean"
    if ".other" in hf_split:
        return "other"
    return "all"

def _libritts_hf_cache_dir() -> str:
    return os.path.join(LIBRITTS_SLIM_CACHE_DIR, "_hf_stream_meta")

def _ensure_libritts_cache_root() -> None:
    os.makedirs(LIBRITTS_SLIM_CACHE_DIR, exist_ok=True)
    os.makedirs(_libritts_hf_cache_dir(), exist_ok=True)

def _libritts_expected_manifest() -> Dict[str, object]:
    return {
        "format_version": LIBRITTS_SLIM_CACHE_VERSION,
        "dataset_name": LIBRITTS_DATASET_NAME,
        "dataset_revision": DATASET_REVISIONS[LIBRITTS_DATASET_NAME],
        "split_names": list(LIBRITTS_SOURCE_SPLITS.keys()),
        "columns": ["source_text", "target_text"],
        "source_splits": LIBRITTS_SOURCE_SPLITS,
    }

def _libritts_manifest_path() -> str:
    return os.path.join(LIBRITTS_SLIM_CACHE_DIR, "manifest.json")

def _load_libritts_manifest() -> Optional[dict]:
    path = _libritts_manifest_path()
    if not os.path.exists(path):
        return None
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None

def _write_libritts_manifest(rows_by_split: Dict[str, int]) -> None:
    manifest = dict(_libritts_expected_manifest())
    manifest["rows_by_split"] = {k: int(v) for k, v in rows_by_split.items()}
    manifest["files"] = {k: os.path.basename(_libritts_split_parquet_path(k)) for k in LIBRITTS_SOURCE_SPLITS.keys()}

    with open(_libritts_manifest_path(), "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

def _libritts_split_parquet_path(split_name: str) -> str:
    return os.path.join(LIBRITTS_SLIM_CACHE_DIR, f"{split_name}.parquet")

def _stream_libritts_hf_source_split(hf_split: str):
    config = _libritts_config_for_hf_split(hf_split)

    ds = load_dataset(
        LIBRITTS_DATASET_NAME,
        config,
        split=hf_split,
        revision=DATASET_REVISIONS[LIBRITTS_DATASET_NAME],
        streaming=True,
        cache_dir=_libritts_hf_cache_dir(),
    )

    try:
        ds = ds.decode(False)
    except Exception:
        try:
            ds = ds.cast_column("audio", Audio(decode=False))
        except Exception:
            pass

    keep_cols = [c for c in ["text_original", "text_normalized"] if c in getattr(ds, "column_names", [])]
    if keep_cols and hasattr(ds, "select_columns"):
        ds = ds.select_columns(keep_cols)

    return ds

def _iter_libritts_batch(iterable, batch_size: int):
    it = iter(iterable)
    while True:
        batch = list(islice(it, batch_size))
        if not batch:
            break
        yield batch

def _build_libritts_slim_cache(
        *,
        batch_size: int = 16384,
        force_rebuild: bool = False,
) -> str:
    _ensure_libritts_cache_root()

    import pyarrow as pa
    import pyarrow.parquet as pq

    schema = pa.schema([
        ("source_text", pa.string()),
        ("target_text", pa.string()),
    ])

    def _valid_split_file(path: str) -> Optional[int]:
        if not os.path.exists(path) or os.path.getsize(path) <= 0:
            return None
        try:
            pf = pq.ParquetFile(path)
            cols = set(pf.schema.names)
            if cols != {"source_text", "target_text"}:
                return None
            return int(pf.metadata.num_rows or 0)
        except Exception:
            return None

    rows_by_split = {}
    manifest = _load_libritts_manifest() or {}
    expected_manifest = _libritts_expected_manifest()
    manifest_matches = all(manifest.get(key) == value for key, value in expected_manifest.items())
    if manifest and not manifest_matches:
        print("[DATA] LibriTTS cache metadata changed; rebuilding pinned source data.")

    for split_name in LIBRITTS_SOURCE_SPLITS.keys():
        out_path = _libritts_split_parquet_path(split_name)

        if not force_rebuild and manifest_matches:
            existing_rows = _valid_split_file(out_path)
            if existing_rows is not None:
                rows_by_split[split_name] = existing_rows
                print(f"[DATA] LibriTTS slim cache reuse -> {split_name}: {existing_rows} rows")
                continue

        tmp_path = f"{out_path}.tmp"
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

        writer = None
        rows_written = 0
        next_report = 100_000

        try:
            for hf_split in LIBRITTS_SOURCE_SPLITS[split_name]:
                print(f"[DATA] LibriTTS slim build -> {split_name} / {hf_split}")
                ds = _stream_libritts_hf_source_split(hf_split)

                for batch in _iter_libritts_batch(ds, batch_size):
                    source_batch = []
                    target_batch = []

                    for ex in batch:
                        src = _truncate(ex.get("text_original", ""))
                        tgt = _truncate(ex.get("text_normalized", ""))

                        if not src or not tgt:
                            continue

                        source_batch.append(src)
                        target_batch.append(tgt)

                    if not source_batch:
                        continue

                    table = pa.table(
                        {
                            "source_text": source_batch,
                            "target_text": target_batch,
                        },
                        schema=schema,
                    )

                    if writer is None:
                        writer = pq.ParquetWriter(
                            tmp_path,
                            schema=schema,
                            compression="zstd",
                        )

                    writer.write_table(table)
                    rows_written += len(source_batch)

                    if rows_written >= next_report:
                        print(f"[DATA] LibriTTS progress -> {split_name}: {rows_written:,} rows")
                        next_report += 100_000

        finally:
            if writer is not None:
                writer.close()

        if rows_written <= 0:
            raise RuntimeError(f"[LibriTTS] no row written for split '{split_name}'")

        os.replace(tmp_path, out_path)
        rows_by_split[split_name] = int(rows_written)
        print(f"[DATA] LibriTTS slim done -> {split_name}: {rows_written} rows")

    _write_libritts_manifest(rows_by_split)
    print(f"[DATA] LibriTTS slim cache ready -> {LIBRITTS_SLIM_CACHE_DIR}")
    return LIBRITTS_SLIM_CACHE_DIR

def _load_libritts_slim_split_df(split_name: str) -> pd.DataFrame:
    _build_libritts_slim_cache(force_rebuild=False)
    path = _libritts_split_parquet_path(split_name)

    df = pd.read_parquet(path, columns=["source_text", "target_text"])
    if df.empty:
        return pd.DataFrame(columns=["source_text", "target_text"])

    df["source_text"] = df["source_text"].astype(str).map(_truncate)
    df["target_text"] = df["target_text"].astype(str).map(_truncate)
    df = df[(df["source_text"].str.len() > 0) & (df["target_text"].str.len() > 0)].reset_index(drop=True)
    return df

def _build_libritts_views_from_slim_df(df: pd.DataFrame):
    if df is None or df.empty:
        return _empty_binary_text_df(), _empty_paired_rewrite_df()

    base = df.copy().reset_index(drop=True)
    base["source_text"] = base["source_text"].astype(str)
    base["target_text"] = base["target_text"].astype(str)

    base = base[
        (base["source_text"].str.len() > 0) &
        (base["target_text"].str.len() > 0) &
        (base["source_text"] != base["target_text"])
        ].drop_duplicates(subset=["source_text", "target_text"]).reset_index(drop=True)

    if base.empty:
        return _empty_binary_text_df(), _empty_paired_rewrite_df()

    paired_df = pd.DataFrame({
        "raw_text": base["source_text"],
        "target_text": base["target_text"],
        "dataset": "libritts_text_original_to_normalized",
        "task_id": int(TASK_IDS["libritts_text_original_to_normalized"]),
        "gold_label": 0,
        "target_label": int(REWRITE_TARGET_LABEL),
        "reference": int(REWRITE_TARGET_LABEL),
    })
    paired_df = _shuffle_df(paired_df)

    binary_df = _expand_paired_split_to_binary_df(paired_df)

    return binary_df.reset_index(drop=True), paired_df.reset_index(drop=True)

def _load_libritts_stream(split_name: str):
    df = _load_libritts_slim_split_df(split_name)
    for ex in df.to_dict(orient="records"):
        yield ex

def _build_libritts_binary_split(split_name: str) -> pd.DataFrame:
    pair_df = _load_libritts_slim_split_df(split_name)
    if pair_df.empty:
        return _empty_binary_text_df()

    pair_df = pair_df[pair_df["source_text"] != pair_df["target_text"]].reset_index(drop=True)
    if pair_df.empty:
        return _empty_binary_text_df()

    part0 = pd.DataFrame({
        "text": pair_df["source_text"].astype(str),
        "label": np.zeros(len(pair_df), dtype=int),
    })
    part1 = pd.DataFrame({
        "text": pair_df["target_text"].astype(str),
        "label": np.ones(len(pair_df), dtype=int),
    })

    return _prepare_binary_df(pd.concat([part0, part1], ignore_index=True))

def _build_libritts_paired_split(split_name: str) -> pd.DataFrame:
    df = _load_libritts_slim_split_df(split_name)
    if df.empty:
        return _empty_paired_rewrite_df()

    df = df[df["source_text"] != df["target_text"]].copy().reset_index(drop=True)
    if df.empty:
        return _empty_paired_rewrite_df()

    df = df.drop_duplicates(subset=["source_text", "target_text"]).reset_index(drop=True)

    out = pd.DataFrame({
        "raw_text": df["source_text"].astype(str),
        "target_text": df["target_text"].astype(str),
        "dataset": "libritts_text_original_to_normalized",
        "task_id": int(TASK_IDS["libritts_text_original_to_normalized"]),
        "gold_label": 0,
        "target_label": int(REWRITE_TARGET_LABEL),
        "reference": int(REWRITE_TARGET_LABEL),
    })

    return _shuffle_df(out)

def _cap_df(df: pd.DataFrame, max_samples: Optional[int]) -> pd.DataFrame:
    if max_samples is None:
        return df.reset_index(drop=True)
    n = min(int(max_samples), len(df))
    return df.sample(n=n, random_state=SEED).reset_index(drop=True)
    
def _prepare_libritts_shared_task_data():
    _build_libritts_slim_cache(force_rebuild=False)

    base_by_split = {
        split_name: _load_libritts_slim_split_df(split_name)
        for split_name in ("train", "validation", "test")
    }

    merged_base_df = pd.concat(
        [df for df in base_by_split.values() if not df.empty],
        ignore_index=True,
    ) if any(not df.empty for df in base_by_split.values()) else pd.DataFrame(columns=["source_text", "target_text"])

    if merged_base_df.empty:
        raise RuntimeError("No LibriTTS-R slim split loaded correctly.")

    _, paired_merged_df = _build_libritts_views_from_slim_df(merged_base_df)
    if paired_merged_df.empty:
        raise RuntimeError("No valid LibriTTS raw->target pairs available after merge.")

    paired_train_df, paired_val_df, paired_test_df = _split_paired_pool(
        paired_merged_df,
        val_count=CLASSIFIER_VAL_COUNT,
        test_frac=CLASSIFIER_TEST_FRACTION,
    )

    train_df = _expand_paired_split_to_binary_df(paired_train_df)
    val_df = _expand_paired_split_to_binary_df(paired_val_df)
    test_df = _expand_paired_split_to_binary_df(paired_test_df)

    merged_df = _expand_paired_split_to_binary_df(paired_merged_df)
    available_splits = [split_name for split_name, df_split in base_by_split.items() if not df_split.empty]

    return {
        "df_train": train_df.reset_index(drop=True),
        "df_val": val_df.reset_index(drop=True),
        "df_test": test_df.reset_index(drop=True),
        "df_merged": merged_df.reset_index(drop=True),
        "available_original_splits": available_splits,
        "df_paired_train": paired_train_df.reset_index(drop=True),
        "df_paired_val": paired_val_df.reset_index(drop=True),
        "df_paired_test": paired_test_df.reset_index(drop=True),
        "df_paired_merged": paired_merged_df.reset_index(drop=True),
        "libritts_slim_cache_dir": LIBRITTS_SLIM_CACHE_DIR,
    }



SHARED_TASK_DATA = None
ALPACA_SHARED_DATA = None

@lru_cache(maxsize=8)
def _build_imdb_df(split_name: str) -> pd.DataFrame:
    ds = load_dataset(
        IMDB_DATASET_NAME, split=split_name, revision=DATASET_REVISIONS[IMDB_DATASET_NAME], streaming=True
    ).shuffle(seed=SEED, buffer_size=20_000)

    def emit(ex):
        text = ex.get("text", "")
        raw = ex.get("label")
        if raw is None:
            return []
        try:
            lab = int(raw)
        except Exception:
            return []
        return [(text, lab)]

    return _stream_binary_df(ds, emit)

@lru_cache(maxsize=8)
def _build_setfit_subj_df(split_name: str) -> pd.DataFrame:
    ds = load_dataset(
        SETFIT_SUBJ_DATASET_NAME, split=split_name, revision=DATASET_REVISIONS[SETFIT_SUBJ_DATASET_NAME], streaming=True
    ).shuffle(seed=SEED, buffer_size=20_000)

    def emit(ex):
        text = ex.get("text", "")
        raw = ex.get("label")
        if raw is None:
            return []
        try:
            lab = int(raw)
        except Exception:
            return []
        if lab not in (0, 1):
            return []
        lab = 1 - lab
        return [(text, lab)]

    return _stream_binary_df(ds, emit)

@lru_cache(maxsize=8)
def _load_stanford_politeness_split_df(split_name: str) -> pd.DataFrame:
    from huggingface_hub import hf_hub_download

    split_name = split_name.lower()
    filename_by_split = {
        "train": "fine-tuning/train_full.csv",
        "test": "fine-tuning/test.csv",
    }

    filename = filename_by_split.get(split_name)
    if filename is None:
        raise ValueError(
            f"Split not available for {STANFORD_POLITENESS_DATASET_NAME}: {split_name}"
        )

    csv_path = hf_hub_download(
        repo_id=STANFORD_POLITENESS_DATASET_NAME,
        filename=filename,
        repo_type="dataset",
        revision=DATASET_REVISIONS[STANFORD_POLITENESS_DATASET_NAME],
    )
    return pd.read_csv(csv_path)

def _normalize_stanford_politeness_label(raw) -> Optional[int]:
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return None

    val = str(raw).strip().lower()
    if val == "polite":
        return 1
    if val == "impolite":
        return 0
    return None


def _build_stanford_politeness_df(split_name: str) -> pd.DataFrame:
    df = _load_stanford_politeness_split_df(split_name).copy()

    if {"prompt", "completion"}.issubset(df.columns):
        text_series = df["prompt"]
        label_series = df["completion"]
    elif "text" in df.columns and ("label_cat" in df.columns or "label" in df.columns):
        text_series = df["text"]
        label_series = df["label_cat"] if "label_cat" in df.columns else df["label"]
    else:
        raise KeyError(
            "Cleanlab/stanford-politeness: unsupported schema. "
            f"Available columns: {sorted(df.columns.tolist())}"
        )

    out = pd.DataFrame({
        "text": text_series.astype(str).map(_truncate),
        "label": pd.Series(label_series).map(_normalize_stanford_politeness_label),
    })
    out = out.dropna(subset=["label"]).reset_index(drop=True)
    out["label"] = out["label"].astype(int)

    return _prepare_binary_df(out[["text", "label"]])

def _coerce_toxic_binary_label(raw) -> Optional[int]:
    if raw is None:
        return None
    if isinstance(raw, (bool, np.bool_)):
        return int(bool(raw))
    if isinstance(raw, str):
        val = raw.strip().lower()
        if val in {"1", "true", "toxic", "yes"}:
            return 1
        if val in {"0", "false", "non-toxic", "nontoxic", "non_toxic", "no"}:
            return 0
    try:
        return int(round(float(raw)))
    except Exception:
        return None


def _build_hf_binary_toxicity_df(
    dataset_name: str,
    split_name: str,
    *,
    text_keys: List[str],
    label_keys: List[str],
) -> pd.DataFrame:
    if split_name != "train":
        return _empty_binary_text_df()

    ds = load_dataset(
        dataset_name, split="train", revision=DATASET_REVISIONS[dataset_name], streaming=True
    ).shuffle(seed=SEED, buffer_size=20_000)

    def emit(ex):
        text_key = _first_present(ex, text_keys)
        label_key = _first_present(ex, label_keys)
        if text_key is None or label_key is None:
            return []

        text = ex.get(text_key, "")
        toxic_raw = ex.get(label_key)
        toxic_label = _coerce_toxic_binary_label(toxic_raw)
        if toxic_label is None:
            return []

        lab = 0 if int(toxic_label) == 1 else 1
        return [(text, lab)]

    return _stream_binary_df(ds, emit)

def _build_paradetox_df(split_name: str) -> pd.DataFrame:
    return _build_hf_binary_toxicity_df(
        PARADETOX_TOXICITY_DATASET_NAME,
        split_name,
        text_keys=["comment", "text", "comment_text"],
        label_keys=["toxic", "label"],
    )

@lru_cache(maxsize=8)
def _build_authormix_woolf_df(split_name: str) -> pd.DataFrame:
    ds = load_dataset(
        AUTHORMIX_DATASET_NAME, split=split_name,
        revision=DATASET_REVISIONS[AUTHORMIX_DATASET_NAME], streaming=True,
    )
    ds = ds.shuffle(seed=SEED, buffer_size=20_000)

    def emit(ex):
        text = ex.get("text", "")
        style = str(ex.get("style", "")).strip().lower()
        if style == "woolf":
            return [(text, 1)]
        if style in {"trump", "obama"}:
            return [(text, 0)]
        return []

    return _stream_binary_df(ds, emit)

TASK_BINARY_BUILDERS = {
    "libritts_text_original_to_normalized": _build_libritts_binary_split,
    "paradetox_non_toxic": _build_paradetox_df,
    "imdb_positive_sentiment": _build_imdb_df,
    "stanford_politeness_polite": _build_stanford_politeness_df,
    "authormix_woolf_style": _build_authormix_woolf_df,
    "setfit_subj_objective": _build_setfit_subj_df,
}

def _load_binary_task_split(task_name: str, split_name: str) -> pd.DataFrame:
    if task_name not in TASK_BINARY_BUILDERS:
        raise KeyError(f"Task not supported in the active test: {task_name}")

    builder_fn = TASK_BINARY_BUILDERS[task_name]
    try:
        df = builder_fn(split_name)
    except Exception as exc:
        print(f"[DATA] {task_name}: original split '{split_name}' not available, skip in merge ({exc!r})")
        return _empty_binary_text_df()

    if df is None or df.empty:
        return _empty_binary_text_df()

    required_cols = {"text", "label"}
    missing = required_cols.difference(df.columns)
    if missing:
        raise KeyError(f"{task_name}/{split_name}: missing columns {sorted(missing)}")

    return _prepare_binary_df(df[["text", "label"]])

def _merge_available_binary_splits(task_name: str) -> Tuple[pd.DataFrame, List[str]]:
    parts: List[pd.DataFrame] = []
    available_splits: List[str] = []

    for split_name in ORIGINAL_SPLIT_ORDER:
        split_df = _load_binary_task_split(task_name, split_name)
        if split_df.empty:
            continue

        split_df = split_df.copy().reset_index(drop=True)
        split_df["original_split"] = split_name
        parts.append(split_df)
        available_splits.append(split_name)

    if not parts:
        raise RuntimeError(f"[{task_name}] no original splits available for merging.")

    merged_df = pd.concat(parts, ignore_index=True)
    merged_df = _prepare_binary_df(merged_df[["text", "label"]])

    label_values = set(merged_df["label"].astype(int).unique().tolist())
    if label_values != {0, 1}:
        raise RuntimeError(f"[{task_name}] the merged dataset must contain both classes 0 and 1, found {sorted(label_values)}")

    return merged_df, available_splits

def _prepare_libritts_shared_task_data():
    _build_libritts_slim_cache(force_rebuild=False)

    base_by_split = {
        split_name: _load_libritts_slim_split_df(split_name)
        for split_name in ("train", "validation", "test")
    }

    merged_base_df = pd.concat(
        [df for df in base_by_split.values() if not df.empty],
        ignore_index=True,
    ) if any(not df.empty for df in base_by_split.values()) else pd.DataFrame(columns=["source_text", "target_text"])

    if merged_base_df.empty:
        raise RuntimeError("No LibriTTS-R slim split loaded correctly.")

    _, paired_merged_df = _build_libritts_views_from_slim_df(merged_base_df)
    if paired_merged_df.empty:
        raise RuntimeError("No valid LibriTTS raw->target pairs available after merge.")

    paired_train_df, paired_val_df, paired_test_df = _split_paired_pool(
        paired_merged_df,
        val_count=CLASSIFIER_VAL_COUNT,
        test_frac=CLASSIFIER_TEST_FRACTION,
    )

    train_df = _expand_paired_split_to_binary_df(paired_train_df)
    val_df = _expand_paired_split_to_binary_df(paired_val_df)
    test_df = _expand_paired_split_to_binary_df(paired_test_df)

    merged_df = _expand_paired_split_to_binary_df(paired_merged_df)
    available_splits = [split_name for split_name, df_split in base_by_split.items() if not df_split.empty]

    return {
        "df_train": train_df.reset_index(drop=True),
        "df_val": val_df.reset_index(drop=True),
        "df_test": test_df.reset_index(drop=True),
        "df_merged": merged_df.reset_index(drop=True),
        "available_original_splits": available_splits,
        "df_paired_train": paired_train_df.reset_index(drop=True),
        "df_paired_val": paired_val_df.reset_index(drop=True),
        "df_paired_test": paired_test_df.reset_index(drop=True),
        "df_paired_merged": paired_merged_df.reset_index(drop=True),
        "libritts_slim_cache_dir": LIBRITTS_SLIM_CACHE_DIR,
    }



def prepare_shared_task_data() -> Dict[str, Dict[str, object]]:
    from datasets import Dataset, DatasetDict

    task_data_by_name: Dict[str, Dict[str, object]] = {}
    task_names = list(ACTIVE_TASK_IDS.keys())

    for task_name in task_names:
        print(f"[DATA] build merged pool -> {task_name}")

        if task_name == "libritts_text_original_to_normalized":
            task_data = _prepare_libritts_shared_task_data()
            train_df = task_data["df_train"]
            val_df = task_data["df_val"]
            test_df = task_data["df_test"]
            merged_df = task_data["df_merged"]
            available_splits = task_data["available_original_splits"]
        else:
            merged_df, available_splits = _merge_available_binary_splits(task_name)
            train_df, val_df, test_df = _split_binary_pool_random(
                merged_df,
                val_count=CLASSIFIER_VAL_COUNT,
                test_frac=CLASSIFIER_TEST_FRACTION,
            )
            task_data = {
                "df_train": train_df,
                "df_val": val_df,
                "df_test": test_df,
                "df_merged": merged_df,
                "available_original_splits": available_splits,
            }

        train_hf = Dataset.from_pandas(train_df[["text", "label"]].reset_index(drop=True), preserve_index=False)
        val_hf = Dataset.from_pandas(val_df[["text", "label"]].reset_index(drop=True), preserve_index=False)
        test_hf = Dataset.from_pandas(test_df[["text", "label"]].reset_index(drop=True), preserve_index=False)
        task_data["hf"] = DatasetDict(train=train_hf, validation=val_hf, test=test_hf)
        task_data_by_name[task_name] = task_data

        if task_name == "libritts_text_original_to_normalized":
            print(
                f"[DATA] {task_name}: paired merged={len(task_data['df_paired_merged'])} from {available_splits} | "
                f"paired train={len(task_data['df_paired_train'])} | "
                f"paired validation={len(task_data['df_paired_val'])} | "
                f"paired test={len(task_data['df_paired_test'])}"
            )
        else:
            print(
                f"[DATA] {task_name}: merged={len(merged_df)} from {available_splits} | "
                f"train={len(train_df)} (label0={int((train_df.label == 0).sum())}, label1={int((train_df.label == 1).sum())}) | "
                f"validation={len(val_df)} (label0={int((val_df.label == 0).sum())}, label1={int((val_df.label == 1).sum())}) | "
                f"test={len(test_df)} (label0={int((test_df.label == 0).sum())}, label1={int((test_df.label == 1).sum())})"
            )

    return task_data_by_name

SHARED_TASK_DATA = None


# Classifiers

In [10]:
CHECKPOINT_CONFIG = {
    "schema_version": 2,
    "code_version": CODE_VERSION,
    "seed": SEED,
    "model_name": MODEL_NAME,
    "model_revision": MODEL_REVISION,
    "dataset_revisions": DATASET_REVISIONS,
    "task_ids": TASK_IDS,
    "generation_kwargs": GENERATION_KWARGS,
    "steering_layers": {
        "layers_to_extract": LAYERS_TO_EXTRACT,
        "robustness_selected_layer": LLAMA_ROBUSTNESS_LAYER,
    },
    "execution_policy": {
        "target_gpu_name": TARGET_GPU_NAME,
        "cuda_device_order": os.environ["CUDA_DEVICE_ORDER"],
        "visible_device": os.environ["CUDA_VISIBLE_DEVICES"],
        "eval_generation_batch_size": EVAL_GENERATION_BATCH_SIZE,
        "length_bucketing": True,
        "oom_batch_fallback": True,
        "sdpa_policy": "flash_if_capability_ge_8_else_memory_efficient_with_math_fallback",
        "empty_cache_every": 0,
        "gc_every": 50,
    },
    "token_budgets": {
        "max_sequence_tokens": MAX_SEQUENCE_TOKENS,
        "max_prompt_tokens": MAX_PROMPT_TOKENS,
        "max_new_tokens": MAX_NEW_TOKENS,
        "min_dynamic_new_token_budget": MIN_GENERATION_TOKENS,
        "generation_length_ratio": GENERATION_LENGTH_RATIO,
    },
    "pysteer_source_sha256": PYSTEER_SOURCE_SHA256,
    "prompt_registry_sha256": PROMPT_REGISTRY_SHA256,
}
CHECKPOINT_RUN_ID = hashlib.sha256(
    json.dumps(CHECKPOINT_CONFIG, sort_keys=True, ensure_ascii=False).encode("utf-8")
).hexdigest()[:16]
CHECKPOINT_DIR = ARTIFACT_DIR / "experiment_checkpoints" / CHECKPOINT_RUN_ID
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("Checkpoint directory:", CHECKPOINT_DIR)

def _checkpoint_path(name: str) -> Path:
    safe_name = "".join(ch if ch.isalnum() or ch in {"-", "_"} else "_" for ch in str(name))
    if not safe_name:
        raise ValueError("Checkpoint name cannot be empty.")
    return CHECKPOINT_DIR / f"{safe_name}.pkl"

def save_experiment_checkpoint(name: str, payload: Any, signature: Any = None) -> Path:
    path = _checkpoint_path(name)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    envelope = {
        "run_id": CHECKPOINT_RUN_ID,
        "signature": signature,
        "payload": payload,
    }
    with tmp_path.open("wb") as handle:
        pickle.dump(envelope, handle, protocol=pickle.HIGHEST_PROTOCOL)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(tmp_path, path)
    print(f"[checkpoint] saved {name} -> {path}")
    return path

def write_json_atomic(path: Path, payload: Any) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    with tmp_path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(tmp_path, path)
    return path

def find_resumable_trainer_checkpoint(output_dir: Path) -> Optional[str]:
    output_dir = Path(output_dir)
    if not output_dir.exists():
        return None
    latest = get_last_checkpoint(str(output_dir))
    if latest is None:
        return None
    candidates = [
        path for path in output_dir.glob("checkpoint-*")
        if path.name.rsplit("-", 1)[-1].isdigit()
    ]
    candidates.sort(key=lambda path: int(path.name.rsplit("-", 1)[-1]), reverse=True)
    for candidate in candidates:
        model_exists = any(
            (candidate / filename).exists()
            for filename in ("model.safetensors", "pytorch_model.bin")
        )
        state_exists = all(
            (candidate / filename).exists()
            for filename in ("trainer_state.json", "optimizer.pt", "scheduler.pt", "rng_state.pth")
        )
        if model_exists and state_exists:
            return str(candidate)
        print(f"[checkpoint] ignoring incomplete Trainer checkpoint: {candidate}")
    return None

def load_experiment_checkpoint(name: str, default: Any, signature: Any = None) -> Any:
    path = _checkpoint_path(name)
    if not path.exists():
        return default
    try:
        with path.open("rb") as handle:
            envelope = pickle.load(handle)
        if envelope.get("run_id") != CHECKPOINT_RUN_ID:
            raise ValueError("run fingerprint mismatch")
        if envelope.get("signature") != signature:
            raise ValueError("section signature mismatch")
        print(f"[checkpoint] loaded {name} <- {path}")
        return envelope["payload"]
    except Exception as exc:
        print(f"[checkpoint] ignored unreadable or incompatible {name}: {exc}")
        return default

@dataclass
class ClassifierSpec:
    base_model_name: str
    max_length: int

classifier_specs: Dict[str, ClassifierSpec] = {
    "libritts_text_original_to_normalized": ClassifierSpec(CLASSIFIER_MODEL_NAME, CLASSIFIER_MAX_LENGTH),
    "paradetox_non_toxic": ClassifierSpec(CLASSIFIER_MODEL_NAME, CLASSIFIER_MAX_LENGTH),
    "imdb_positive_sentiment": ClassifierSpec(CLASSIFIER_MODEL_NAME, CLASSIFIER_MAX_LENGTH),
    "stanford_politeness_polite": ClassifierSpec(CLASSIFIER_MODEL_NAME, CLASSIFIER_MAX_LENGTH),
    "authormix_woolf_style": ClassifierSpec(CLASSIFIER_MODEL_NAME, CLASSIFIER_MAX_LENGTH),
    "setfit_subj_objective": ClassifierSpec(CLASSIFIER_MODEL_NAME, CLASSIFIER_MAX_LENGTH),
}

classifier_specs = {
    name: spec for name, spec in classifier_specs.items() if name in ACTIVE_TASK_IDS
}

print("[config] Active classifier tasks:", list(classifier_specs.keys()))

local_classifiers: Dict[str, Dict[str, Any]] = {}

TRAIN_CFG = dict(
    learning_rate=3e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=128,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.10,
    lr_scheduler_type="linear",
    max_grad_norm=1.0,
    eval_strategy="epoch",
    save_strategy="epoch",
    greater_is_better=True,
    load_best_model_at_end=True,
    group_by_length=True,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    dataloader_persistent_workers=True,
    dataloader_prefetch_factor=4,
    optim="adamw_torch_fused",
)

EARLY_STOPPING = dict(
    patience=4,
    threshold=1e-4,
)

def _fingerprint(payload: dict) -> str:
    blob = json.dumps(payload, sort_keys=True, ensure_ascii=False).encode("utf-8")
    return hashlib.sha256(blob).hexdigest()[:16]

def _classifier_payload(dataset_name: str, spec: ClassifierSpec) -> dict:
    payload = {
        "dataset_name": dataset_name,
        "base_model_name": spec.base_model_name,
        "base_model_revision": CLASSIFIER_MODEL_REVISION,
        "dataset_revision": DATASET_REVISIONS.get({
            "libritts_text_original_to_normalized": LIBRITTS_DATASET_NAME,
            "paradetox_non_toxic": PARADETOX_TOXICITY_DATASET_NAME,
            "imdb_positive_sentiment": IMDB_DATASET_NAME,
            "stanford_politeness_polite": STANFORD_POLITENESS_DATASET_NAME,
            "authormix_woolf_style": AUTHORMIX_DATASET_NAME,
            "setfit_subj_objective": SETFIT_SUBJ_DATASET_NAME,
        }[dataset_name]),
        "max_length": spec.max_length,
        "seed": SEED,
        "train_cfg": TRAIN_CFG,
        "early_stopping": EARLY_STOPPING,
        "code_version": CODE_VERSION,
    }
    return payload

def _classifier_dir(dataset_name: str, spec: ClassifierSpec) -> Path:
    fp = _fingerprint(_classifier_payload(dataset_name, spec))
    return CLF_CACHE_ROOT / dataset_name / fp

def _make_training_args(output_dir: str) -> TrainingArguments:
    sig = inspect.signature(TrainingArguments.__init__)

    kwargs = dict(
        **TRAIN_CFG,
        output_dir=output_dir,
        seed=SEED,
        data_seed=SEED,
        full_determinism=True,
        logging_steps=20,
        report_to="none",
    )

    kwargs.update(
        load_best_model_at_end=True,
        save_total_limit=1,
        metric_for_best_model="eval_acc",
        greater_is_better=True,
    )

    if "save_safetensors" in sig.parameters:
        kwargs["save_safetensors"] = True

    if torch.cuda.is_available() and "fp16" in sig.parameters:
        kwargs["fp16"] = True

    return TrainingArguments(**kwargs)

def _build_trainer_callbacks(args: TrainingArguments):
    callbacks = []
    if getattr(args, "load_best_model_at_end", True):
        callbacks.append(
            EarlyStoppingCallback(
                early_stopping_patience=EARLY_STOPPING["patience"],
                early_stopping_threshold=EARLY_STOPPING["threshold"],
            )
        )
    return callbacks

all_results = []

def train_single_classifier(dataset_name: str, spec: ClassifierSpec) -> Dict[str, Any]:
    save_dir = _classifier_dir(dataset_name, spec)
    completion_marker = save_dir / "meta.json"
    if completion_marker.exists():
        try:
            metadata = json.loads(completion_marker.read_text(encoding="utf-8"))
            cached_config = metadata.get("config", metadata)
            if cached_config != _classifier_payload(dataset_name, spec):
                raise ValueError("classifier cache configuration mismatch")
            print(f"\n=== Loaded cached classifier: {dataset_name} ({save_dir}) ===")
            model = AutoModelForSequenceClassification.from_pretrained(str(save_dir))
            tok = AutoTokenizer.from_pretrained(str(save_dir), use_fast=True)
            return {
                "model": model,
                "tokenizer": tok,
                "max_length": spec.max_length,
                "artifact_dir": str(save_dir),
                "test_metrics": metadata.get("test_metrics"),
            }
        except Exception as e:
            print(f"[cache] incomplete or unreadable final classifier for {dataset_name} ({e}); resuming training when possible.")

    print(f"\n=== Training classifier: {dataset_name} ===")

    global SHARED_TASK_DATA
    if SHARED_TASK_DATA is None:
        SHARED_TASK_DATA = prepare_shared_task_data()

    hf_ds = SHARED_TASK_DATA[dataset_name]["hf"]
    tok = AutoTokenizer.from_pretrained(
        spec.base_model_name, revision=CLASSIFIER_MODEL_REVISION, use_fast=True
    )
    clf = AutoModelForSequenceClassification.from_pretrained(
        spec.base_model_name, revision=CLASSIFIER_MODEL_REVISION, num_labels=2
    )

    def tok_fn(batch):
        return tok(batch["text"], truncation=True, max_length=spec.max_length)

    hf_ds = hf_ds.map(
        tok_fn, batched=True, num_proc=min(8, max(1, os.cpu_count() or 1)),
        desc=f"Tokenizing {dataset_name}",
    )
    keep_cols = {"input_ids", "attention_mask", "label"}
    hf_ds = hf_ds.remove_columns([c for c in hf_ds["train"].column_names if c not in keep_cols])
    hf_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

    save_dir.mkdir(parents=True, exist_ok=True)
    args = _make_training_args(output_dir=str(save_dir))

    data_collator = DataCollatorWithPadding(
        tokenizer=tok,
        pad_to_multiple_of=8 if torch.cuda.is_available() else None,
    )

    trainer = Trainer(
        model=clf,
        args=args,
        train_dataset=hf_ds["train"],
        eval_dataset=hf_ds["validation"],
        compute_metrics=compute_metrics,
        data_collator=data_collator,
        callbacks=_build_trainer_callbacks(args),
    )
    trainer.add_callback(NotebookProgressCallback)
    resume_checkpoint = find_resumable_trainer_checkpoint(save_dir)
    if resume_checkpoint is not None:
        print(f"[checkpoint] resuming classifier training from {resume_checkpoint}")
    trainer.train(resume_from_checkpoint=resume_checkpoint)

    final_test_metrics = trainer.evaluate(
        eval_dataset=hf_ds["test"],
        metric_key_prefix="test",
    )

    print(f"[{dataset_name}] final holdout test: {final_test_metrics}")

    trained = trainer.model
    try:
        trained.save_pretrained(str(save_dir), safe_serialization=True)
    except TypeError:
        trained.save_pretrained(str(save_dir))
    tok.save_pretrained(str(save_dir))

    write_json_atomic(
        completion_marker,
        {
            "config": _classifier_payload(dataset_name, spec),
            "test_metrics": final_test_metrics,
        },
    )

    return {
        "model": trained,
        "tokenizer": tok,
        "max_length": spec.max_length,
        "artifact_dir": str(save_dir),
        "test_metrics": final_test_metrics,
    }

def train_all_classifiers():
    for name, spec in classifier_specs.items():
        local_classifiers[name] = train_single_classifier(name, spec)

        metrics = local_classifiers[name]["test_metrics"]
        if metrics is not None:
            all_results.append({"dataset": name, **metrics})

Checkpoint directory: /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/2143919937f89e4b
[config] Active classifier tasks: ['libritts_text_original_to_normalized', 'paradetox_non_toxic', 'imdb_positive_sentiment', 'stanford_politeness_polite', 'authormix_woolf_style', 'setfit_subj_objective']


In [11]:
train_all_classifiers()

for name, pack in local_classifiers.items():
    pack["model"].to("cpu")
    pack["model"].eval()

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

results_df = pd.DataFrame(all_results)
display(results_df)


=== Loaded cached classifier: libritts_text_original_to_normalized (/home/ubuntu/ATAS/.atas_cache/atas_artifacts/classifiers/libritts_text_original_to_normalized/8df3d924215a0992) ===

=== Loaded cached classifier: paradetox_non_toxic (/home/ubuntu/ATAS/.atas_cache/atas_artifacts/classifiers/paradetox_non_toxic/286d61264b5d34c8) ===



=== Loaded cached classifier: imdb_positive_sentiment (/home/ubuntu/ATAS/.atas_cache/atas_artifacts/classifiers/imdb_positive_sentiment/2245af26904d857c) ===

=== Loaded cached classifier: stanford_politeness_polite (/home/ubuntu/ATAS/.atas_cache/atas_artifacts/classifiers/stanford_politeness_polite/fd6d5be5e1ee9ac0) ===

=== Loaded cached classifier: authormix_woolf_style (/home/ubuntu/ATAS/.atas_cache/atas_artifacts/classifiers/authormix_woolf_style/16555cd883c0967e) ===



=== Loaded cached classifier: setfit_subj_objective (/home/ubuntu/ATAS/.atas_cache/atas_artifacts/classifiers/setfit_subj_objective/caeed8825b5e467b) ===


,dataset,test_loss,test_acc,test_f1,test_precision,test_recall,test_runtime,test_samples_per_second,test_steps_per_second,epoch
0,libritts_text_original_to_normalized,0.083721,0.97375,0.973978,0.965602,0.982500,0.3245,2465.318,21.572,5.0
1,paradetox_non_toxic,0.170239,0.95250,0.973501,0.980337,0.966759,0.1347,2969.072,29.691,5.0
2,imdb_positive_sentiment,0.228244,0.93000,0.928571,0.914573,0.943005,0.6734,594.028,5.940,5.0
3,stanford_politeness_polite,0.305649,0.87750,0.803213,0.800000,0.806452,0.2349,1702.648,17.026,5.0
4,authormix_woolf_style,0.015014,0.99750,0.996610,0.993243,1.000000,0.3006,1330.685,13.307,5.0
5,setfit_subj_objective,0.135662,0.96500,0.963351,0.968421,0.958333,0.1508,2652.604,26.526,5.0


## Supervised BERT task router baseline

This baseline trains BERT on the complete formatted model input prompt for every example in the classifier training splits. Validation and test examples are never used for fitting. During steering, the predicted task selects both the corresponding ATAS correction vector and that task's validation-selected alpha. Centroid-similarity routing and rejection are bypassed.

In [12]:
if SHARED_TASK_DATA is None:
    SHARED_TASK_DATA = prepare_shared_task_data()

TASK_ROUTER_MAX_LENGTH = CLASSIFIER_MAX_LENGTH
TASK_ROUTER_CACHE_ROOT = ARTIFACT_DIR / "bert_task_router"
TASK_ROUTER_TRAIN_CFG = {
    **TRAIN_CFG,
    "num_train_epochs": 5,
}
if sorted(int(task_id) for task_id in TASK_IDS.values()) != list(range(len(TASK_IDS))):
    raise ValueError("BERT task-router labels must be contiguous integers starting at zero.")

def _build_task_router_frame(split_name: str) -> pd.DataFrame:
    rows = []
    frame_key = {"train": "df_train", "validation": "df_val", "test": "df_test"}[split_name]
    for task_name, task_id in TASK_IDS.items():
        source = SHARED_TASK_DATA[task_name][frame_key]
        part = source[["text"]].copy().reset_index(drop=True)
        part["raw_text"] = part["text"].astype(str)
        part = _assign_prompt_variants(part, task_name, split=split_name)
        part["text"] = part["prompt"].astype(str)
        part["label"] = int(task_id)
        part["task_name"] = task_name
        rows.append(part)
    out = pd.concat(rows, ignore_index=True)
    return out.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

def compute_task_router_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "acc": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
    }

def _task_router_payload() -> dict:
    return {
        "model_name": TASK_ROUTER_MODEL_NAME,
        "model_revision": TASK_ROUTER_MODEL_REVISION,
        "dataset_revisions": DATASET_REVISIONS,
        "task_ids": TASK_IDS,
        "train_rows_by_task": task_router_train_counts,
        "max_length": TASK_ROUTER_MAX_LENGTH,
        "seed": SEED,
        "train_cfg": TASK_ROUTER_TRAIN_CFG,
        "early_stopping": EARLY_STOPPING,
        "input_representation": "full_formatted_prompt",
        "code_version": CODE_VERSION,
    }

def _make_task_router_args(output_dir: str) -> TrainingArguments:
    kwargs = dict(
        **TASK_ROUTER_TRAIN_CFG,
        output_dir=output_dir,
        seed=SEED,
        data_seed=SEED,
        full_determinism=True,
        logging_steps=20,
        report_to="none",
        save_total_limit=1,
        metric_for_best_model="eval_f1_macro",
    )
    sig = inspect.signature(TrainingArguments.__init__)
    if "save_safetensors" in sig.parameters:
        kwargs["save_safetensors"] = True
    if torch.cuda.is_available() and "fp16" in sig.parameters:
        kwargs["fp16"] = True
    return TrainingArguments(**kwargs)

def train_or_load_task_router() -> Dict[str, Any]:
    payload = _task_router_payload()
    save_dir = TASK_ROUTER_CACHE_ROOT / _fingerprint(payload)
    completion_marker = save_dir / "meta.json"
    if completion_marker.exists():
        try:
            metadata = json.loads(completion_marker.read_text(encoding="utf-8"))
            cached_config = metadata.get("config", metadata)
            if cached_config != payload:
                raise ValueError("task-router cache configuration mismatch")
            print(f"Loaded cached BERT task router: {save_dir}")
            return {
                "model": AutoModelForSequenceClassification.from_pretrained(str(save_dir)),
                "tokenizer": AutoTokenizer.from_pretrained(str(save_dir), use_fast=True),
                "max_length": TASK_ROUTER_MAX_LENGTH,
                "artifact_dir": str(save_dir),
            }
        except Exception as exc:
            print(f"Incomplete or unreadable final BERT task router ({exc}); resuming training when possible.")

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    tokenizer_router = AutoTokenizer.from_pretrained(
        TASK_ROUTER_MODEL_NAME, revision=TASK_ROUTER_MODEL_REVISION, use_fast=True
    )
    model_router = AutoModelForSequenceClassification.from_pretrained(
        TASK_ROUTER_MODEL_NAME,
        revision=TASK_ROUTER_MODEL_REVISION,
        num_labels=len(TASK_IDS),
        id2label={int(task_id): task_name for task_name, task_id in TASK_IDS.items()},
        label2id={task_name: int(task_id) for task_name, task_id in TASK_IDS.items()},
    )

    from datasets import Dataset
    train_hf = Dataset.from_pandas(task_router_train_df[["text", "label"]], preserve_index=False)
    val_hf = Dataset.from_pandas(task_router_val_df[["text", "label"]], preserve_index=False)

    def tokenize_router(batch):
        return tokenizer_router(batch["text"], truncation=True, max_length=TASK_ROUTER_MAX_LENGTH)

    tokenize_num_proc = min(8, max(1, os.cpu_count() or 1))
    train_hf = train_hf.map(
        tokenize_router, batched=True, remove_columns=["text"],
        num_proc=tokenize_num_proc, desc="Tokenizing task-router train split",
    )
    val_hf = val_hf.map(
        tokenize_router, batched=True, remove_columns=["text"],
        num_proc=tokenize_num_proc, desc="Tokenizing task-router validation split",
    )
    train_hf.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    val_hf.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

    save_dir.mkdir(parents=True, exist_ok=True)
    router_args = _make_task_router_args(str(save_dir))
    trainer = Trainer(
        model=model_router,
        args=router_args,
        train_dataset=train_hf,
        eval_dataset=val_hf,
        compute_metrics=compute_task_router_metrics,
        data_collator=DataCollatorWithPadding(
            tokenizer=tokenizer_router,
            pad_to_multiple_of=8 if torch.cuda.is_available() else None,
        ),
        callbacks=_build_trainer_callbacks(router_args),
    )
    trainer.add_callback(NotebookProgressCallback)
    resume_checkpoint = find_resumable_trainer_checkpoint(save_dir)
    if resume_checkpoint is not None:
        print(f"[checkpoint] resuming BERT task-router training from {resume_checkpoint}")
    trainer.train(resume_from_checkpoint=resume_checkpoint)
    trained = trainer.model
    trained.save_pretrained(str(save_dir), safe_serialization=True)
    tokenizer_router.save_pretrained(str(save_dir))
    write_json_atomic(completion_marker, {"config": payload})
    return {
        "model": trained,
        "tokenizer": tokenizer_router,
        "max_length": TASK_ROUTER_MAX_LENGTH,
        "artifact_dir": str(save_dir),
    }

def predict_task_ids(texts, batch_size: int = 128, return_conf: bool = False):
    router_model = task_router["model"]
    router_tokenizer = task_router["tokenizer"]
    router_model.eval()
    device = get_model_device(router_model)
    predictions, confidences = [], []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        encoded = router_tokenizer(
            batch, return_tensors="pt", padding=True, truncation=True,
            max_length=task_router["max_length"],
        )
        encoded = {key: value.to(device) for key, value in encoded.items()}
        with torch.inference_mode():
            probabilities = F.softmax(router_model(**encoded).logits, dim=-1)
        predictions.extend(int(value) for value in probabilities.argmax(dim=-1).cpu().tolist())
        confidences.extend(float(value) for value in probabilities.max(dim=-1).values.cpu().tolist())
    return (predictions, confidences) if return_conf else predictions

# Training and evaluation are intentionally executed after prompt construction.

[DATA] build merged pool -> libritts_text_original_to_normalized
[DATA] LibriTTS slim cache reuse -> train: 354693 rows
[DATA] LibriTTS slim cache reuse -> validation: 10349 rows
[DATA] LibriTTS slim cache reuse -> test: 9957 rows
[DATA] LibriTTS slim cache ready -> /home/ubuntu/ATAS/.atas_cache/libritts_r_slim_2cols
[DATA] LibriTTS slim cache reuse -> train: 354693 rows
[DATA] LibriTTS slim cache reuse -> validation: 10349 rows
[DATA] LibriTTS slim cache reuse -> test: 9957 rows
[DATA] LibriTTS slim cache ready -> /home/ubuntu/ATAS/.atas_cache/libritts_r_slim_2cols


[DATA] LibriTTS slim cache reuse -> train: 354693 rows
[DATA] LibriTTS slim cache reuse -> validation: 10349 rows
[DATA] LibriTTS slim cache reuse -> test: 9957 rows
[DATA] LibriTTS slim cache ready -> /home/ubuntu/ATAS/.atas_cache/libritts_r_slim_2cols
[DATA] LibriTTS slim cache reuse -> train: 354693 rows
[DATA] LibriTTS slim cache reuse -> validation: 10349 rows
[DATA] LibriTTS slim cache reuse -> test: 9957 rows
[DATA] LibriTTS slim cache ready -> /home/ubuntu/ATAS/.atas_cache/libritts_r_slim_2cols


[DATA] libritts_text_original_to_normalized: paired merged=63361 from ['train', 'validation', 'test'] | paired train=50289 | paired validation=400 | paired test=12672
[DATA] build merged pool -> paradetox_non_toxic


[DATA] paradetox_non_toxic: merged=24242 from ['train'] | train=18994 (label0=2878, label1=16116) | validation=400 (label0=39, label1=361) | test=4848 (label0=754, label1=4094)
[DATA] build merged pool -> imdb_positive_sentiment


[DATA] imdb_positive_sentiment: original split 'validation' not available, skip in merge (ValueError("Bad split: validation. Available splits: ['train', 'test', 'unsupervised']"))


...

Repo card metadata block was not found. Setting CardData to empty.


Repo card metadata block was not found. Setting CardData to empty.


[DATA] setfit_subj_objective: original split 'validation' not available, skip in merge (ValueError("Bad split: validation. Available splits: ['train', 'test']"))


Repo card metadata block was not found. Setting CardData to empty.


[DATA] setfit_subj_objective: merged=10000 from ['train', 'test'] | train=7600 (label0=3845, label1=3755) | validation=400 (label0=208, label1=192) | test=2000 (label0=947, label1=1053)


In [13]:
def predict_local_labels(texts, clf_key, batch_size=128, return_conf=False):
    info = local_classifiers[clf_key]
    clf, tok, max_len = info["model"], info["tokenizer"], info["max_length"]
    clf.eval()

    device = get_model_device(clf)

    preds, confs = [], []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=max_len)
        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.inference_mode():
            logits = clf(**enc).logits
            probs = F.softmax(logits, dim=-1)
            pred = probs.argmax(dim=-1).cpu().tolist()
            conf = probs.max(dim=-1).values.cpu().tolist()

        preds.extend([int(x) for x in pred])
        confs.extend([float(x) for x in conf])

    return (preds, confs) if return_conf else preds

def _sample_per_class(df: pd.DataFrame, per_class: int) -> pd.DataFrame:
    df = df.copy().reset_index(drop=True)
    pos = df[df["label"] == 1]
    neg = df[df["label"] == 0]

    if len(pos) == 0 or len(neg) == 0:
        raise RuntimeError("Steering sampling requires both classes 0 and 1 in the train split.")

    n = int(per_class)
    if len(pos) < n or len(neg) < n:
        raise RuntimeError(
            f"Steering sampling impossible: {n} examples required per class, "
            f"available label1={len(pos)} label0={len(neg)}."
        )

    out = pd.concat(
        [
            pos.sample(n=n, random_state=SEED),
            neg.sample(n=n, random_state=SEED),
        ],
        ignore_index=True,
    )
    return out.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

def _build_rewrite_prompt(t: str, variants: List[str], variant_id: int = 0) -> str:
    system = ""
    user = (
        f"Text:\n{t}\n\n"
        f"{variants[variant_id % len(variants)]}\n\n"
        "Rewrite:\n"
    )
    return build_llama32_prompt(system, user)


import importlib
import task_prompt_variants as task_prompt_registry


task_prompt_registry = importlib.reload(task_prompt_registry)
TASK_PROMPT_VARIANTS = task_prompt_registry.TASK_PROMPT_VARIANTS
_registry_validator = getattr(task_prompt_registry, "validate_task_prompt_variants", None)
if callable(_registry_validator):
    _registry_validator()
else:
    _expected_tasks = set(getattr(task_prompt_registry, "EXPECTED_TASK_NAMES", TASK_IDS))
    _implicit_count = int(getattr(task_prompt_registry, "IMPLICIT_PROMPTS_PER_TASK", 50))
    if set(TASK_PROMPT_VARIANTS) != _expected_tasks:
        raise ValueError("Prompt registry task names do not match the expected tasks.")
    _all_variants = []
    for _task_name, _variants in TASK_PROMPT_VARIANTS.items():
        if len(_variants) != 100 or not 0 < _implicit_count < len(_variants):
            raise ValueError(f"[{_task_name}] invalid prompt variant counts.")
        if any(not isinstance(_item, str) or not _item.strip() for _item in _variants):
            raise ValueError(f"[{_task_name}] prompt variants must be non-empty strings.")
        _all_variants.extend(_variants)
    if len(set(_all_variants)) != len(_all_variants):
        raise ValueError("Prompt variants must be globally unique.")
    print("Prompt registry validated with the compatibility validator.")
PROMPT_VARIANT_COUNT = 100
PROMPT_SPLIT_FRACTIONS = {
    "train": 0.70,
    "validation": 0.10,
    "test": 0.20,
}


def make_libritts_normalization_prompt(t: str, variant_id: int = 0) -> str:
    return _build_rewrite_prompt(
        t,
        TASK_PROMPT_VARIANTS["libritts_text_original_to_normalized"],
        variant_id,
    )


def make_positive_sentiment_prompt(t: str, variant_id: int = 0) -> str:
    return _build_rewrite_prompt(
        t,
        TASK_PROMPT_VARIANTS["imdb_positive_sentiment"],
        variant_id,
    )


def make_polite_prompt(t: str, variant_id: int = 0) -> str:
    return _build_rewrite_prompt(
        t,
        TASK_PROMPT_VARIANTS["stanford_politeness_polite"],
        variant_id,
    )


def make_woolf_style_prompt(t: str, variant_id: int = 0) -> str:
    return _build_rewrite_prompt(
        t,
        TASK_PROMPT_VARIANTS["authormix_woolf_style"],
        variant_id,
    )


def make_objective_prompt(t: str, variant_id: int = 0) -> str:
    return _build_rewrite_prompt(
        t,
        TASK_PROMPT_VARIANTS["setfit_subj_objective"],
        variant_id,
    )


def make_non_toxic_prompt(t: str, variant_id: int = 0) -> str:
    return _build_rewrite_prompt(
        t,
        TASK_PROMPT_VARIANTS["paradetox_non_toxic"],
        variant_id,
    )


TASK_PROMPT_FN = {
    "libritts_text_original_to_normalized": make_libritts_normalization_prompt,
    "paradetox_non_toxic": make_non_toxic_prompt,
    "imdb_positive_sentiment": make_positive_sentiment_prompt,
    "stanford_politeness_polite": make_polite_prompt,
    "authormix_woolf_style": make_woolf_style_prompt,
    "setfit_subj_objective": make_objective_prompt,
}


PROMPT_SPLIT_ORDER = ("train", "validation", "test")
PROMPT_IMPLICIT_COUNT = int(
    getattr(task_prompt_registry, "IMPLICIT_PROMPTS_PER_TASK", PROMPT_VARIANT_COUNT // 2)
)


def _allocate_prompt_split_counts(total_count: int) -> Dict[str, int]:
    total_count = int(total_count)
    fractions = {name: float(PROMPT_SPLIT_FRACTIONS[name]) for name in PROMPT_SPLIT_ORDER}

    if total_count <= 0:
        raise ValueError("total_count must be positive.")
    if not math.isclose(sum(fractions.values()), 1.0, rel_tol=0.0, abs_tol=1e-12):
        raise ValueError(f"Prompt split fractions must sum to 1.0, found {fractions}.")
    if any(value <= 0.0 for value in fractions.values()):
        raise ValueError(f"Prompt split fractions must all be positive, found {fractions}.")

    raw_counts = {name: total_count * fractions[name] for name in PROMPT_SPLIT_ORDER}
    counts = {name: int(math.floor(raw_counts[name])) for name in PROMPT_SPLIT_ORDER}
    remainder = total_count - sum(counts.values())
    remainder_order = sorted(
        PROMPT_SPLIT_ORDER,
        key=lambda name: (
            -(raw_counts[name] - counts[name]),
            PROMPT_SPLIT_ORDER.index(name),
        ),
    )
    for name in remainder_order[:remainder]:
        counts[name] += 1

    if sum(counts.values()) != total_count or any(count <= 0 for count in counts.values()):
        raise ValueError(f"Invalid prompt split counts for total={total_count}: {counts}.")
    return counts


PROMPT_SPLIT_COUNTS = _allocate_prompt_split_counts(PROMPT_VARIANT_COUNT)
PROMPT_IMPLICIT_SPLIT_COUNTS = _allocate_prompt_split_counts(PROMPT_IMPLICIT_COUNT)
PROMPT_EXPLICIT_SPLIT_COUNTS = {
    name: PROMPT_SPLIT_COUNTS[name] - PROMPT_IMPLICIT_SPLIT_COUNTS[name]
    for name in PROMPT_SPLIT_ORDER
}


def _build_prompt_variant_splits() -> Dict[str, Dict[str, Tuple[int, ...]]]:
    if not (0 < PROMPT_IMPLICIT_COUNT < PROMPT_VARIANT_COUNT):
        raise ValueError(
            f"PROMPT_IMPLICIT_COUNT must be in (0, {PROMPT_VARIANT_COUNT}), "
            f"found {PROMPT_IMPLICIT_COUNT}."
        )

    splits_by_task: Dict[str, Dict[str, Tuple[int, ...]]] = {}
    for task_name, task_id in TASK_IDS.items():
        variants = TASK_PROMPT_VARIANTS[task_name]
        if len(variants) != PROMPT_VARIANT_COUNT:
            raise ValueError(
                f"[{task_name}] expected {PROMPT_VARIANT_COUNT} prompt variants, found {len(variants)}."
            )

        rng = np.random.default_rng(SEED + 10_000 + int(task_id))
        implicit_ids = rng.permutation(np.arange(PROMPT_IMPLICIT_COUNT, dtype=int)).tolist()
        explicit_ids = rng.permutation(
            np.arange(PROMPT_IMPLICIT_COUNT, PROMPT_VARIANT_COUNT, dtype=int)
        ).tolist()

        task_splits: Dict[str, Tuple[int, ...]] = {}
        implicit_start = 0
        explicit_start = 0
        for split_name in PROMPT_SPLIT_ORDER:
            implicit_stop = implicit_start + PROMPT_IMPLICIT_SPLIT_COUNTS[split_name]
            explicit_stop = explicit_start + PROMPT_EXPLICIT_SPLIT_COUNTS[split_name]
            split_ids = implicit_ids[implicit_start:implicit_stop] + explicit_ids[explicit_start:explicit_stop]
            task_splits[split_name] = tuple(
                int(variant_id) for variant_id in rng.permutation(np.asarray(split_ids, dtype=int))
            )
            implicit_start = implicit_stop
            explicit_start = explicit_stop

        splits_by_task[task_name] = task_splits

    return splits_by_task


PROMPT_VARIANT_SPLITS = _build_prompt_variant_splits()


PROMPT_VARIANT_SPLIT_SUMMARY = pd.DataFrame(
    [
        {
            "split": split_name,
            "variants_per_task": PROMPT_SPLIT_COUNTS[split_name],
            "percentage": 100.0 * PROMPT_SPLIT_COUNTS[split_name] / PROMPT_VARIANT_COUNT,
            "implicit_variants": PROMPT_IMPLICIT_SPLIT_COUNTS[split_name],
            "explicit_variants": PROMPT_EXPLICIT_SPLIT_COUNTS[split_name],
        }
        for split_name in PROMPT_SPLIT_ORDER
    ]
)
print("Prompt-variant partition (disjoint within each task):")
display(PROMPT_VARIANT_SPLIT_SUMMARY)


def _canonical_prompt_split(split: str) -> str:
    aliases = {
        "train": "train",
        "steering_train": "train",
        "validation": "validation",
        "val": "validation",
        "test": "test",
    }
    if split not in aliases:
        raise KeyError(f"Unsupported prompt split: {split}")
    return aliases[split]


def _prompt_variant_ids(task_name: str, split: str) -> Tuple[int, ...]:
    if task_name not in PROMPT_VARIANT_SPLITS:
        raise KeyError(f"Task not available for prompt variants: {task_name}")
    return PROMPT_VARIANT_SPLITS[task_name][_canonical_prompt_split(split)]

def _prepare_rewrite_pool_df(base_df: pd.DataFrame, task_name: str) -> pd.DataFrame:
    base = base_df.copy().reset_index(drop=True)
    base["raw_text"] = base["text"].astype(str)
    base["dataset"] = task_name
    base["task_id"] = TASK_IDS[task_name]
    base["gold_label"] = base["label"].astype(int)
    base["target_label"] = REWRITE_TARGET_LABEL
    base["reference"] = int(REWRITE_TARGET_LABEL)
    return base[["raw_text", "dataset", "task_id", "gold_label", "target_label", "reference"]].reset_index(drop=True)

def _assert_split_has_both_labels(df: pd.DataFrame, task_name: str, split_name: str) -> None:
    if df.empty:
        raise RuntimeError(f"[{task_name}] {split_name} is empty.")
    labels = set(df["gold_label"].astype(int).unique().tolist())
    if labels != {0, 1}:
        raise RuntimeError(f"[{task_name}] {split_name}: expected classes {{0, 1}}, found {sorted(labels)}")

def _assert_rewrite_eval_df(df: pd.DataFrame, task_name: str, split_name: str) -> None:
    _assert_split_has_both_labels(df, task_name, split_name)

    ref_values = set(df["reference"].astype(int).unique().tolist())
    if ref_values != {int(REWRITE_TARGET_LABEL)}:
        raise RuntimeError(
            f"[{task_name}] {split_name}: expected reference={REWRITE_TARGET_LABEL}, found {sorted(ref_values)}"
        )

def _assert_paired_target_df(df: pd.DataFrame, task_name: str, split_name: str) -> None:
    if df.empty:
        raise RuntimeError(f"[{task_name}] {split_name} is empty.")

    required = ["raw_text", "target_text"]
    for col in required:
        if col not in df.columns:
            raise RuntimeError(f"[{task_name}] {split_name}: missing column {col}.")
        if df[col].isna().any() or df[col].astype(str).str.len().eq(0).any():
            raise RuntimeError(f"[{task_name}] {split_name}: {col} contains empty values.")

def _paired_split_to_rewrite_eval_df(pair_df: pd.DataFrame, task_name: str) -> pd.DataFrame:
    if pair_df is None or pair_df.empty:
        raise RuntimeError(f"[{task_name}] paired split is empty.")

    out = pair_df[["raw_text", "target_text", "dataset", "task_id"]].copy()
    out["raw_text"] = out["raw_text"].astype(str).map(_truncate)
    out["target_text"] = out["target_text"].astype(str).map(_truncate)

    bad = out["raw_text"].str.len().eq(0) | out["target_text"].str.len().eq(0)
    out = out.loc[~bad].reset_index(drop=True)

    if out.empty:
        raise RuntimeError(f"[{task_name}] paired split has no valid raw->target pairs.")

    return out

def _build_steering_subset_from_training_split(
    training_split_df: pd.DataFrame,
    *,
    task_name: str,
    per_class_train: int,
) -> pd.DataFrame:
    sampled = _sample_per_class(
        training_split_df.rename(columns={"raw_text": "text", "gold_label": "label"})[["text", "label"]],
        per_class_train,
    )
    sampled["raw_text"] = sampled["text"].astype(str)
    sampled["dataset"] = task_name
    sampled["task_id"] = TASK_IDS[task_name]
    sampled["gold_label"] = sampled["label"].astype(int)
    sampled["target_label"] = REWRITE_TARGET_LABEL
    sampled["reference"] = int(REWRITE_TARGET_LABEL)
    return sampled[["raw_text", "dataset", "task_id", "gold_label", "target_label", "reference"]].reset_index(drop=True)

def _assign_prompt_variants(
        df: pd.DataFrame,
        task_name: str,
        split: str,
) -> pd.DataFrame:
    out = df.copy().reset_index(drop=True)
    canonical_split = _canonical_prompt_split(split)
    allowed_variant_ids = _prompt_variant_ids(task_name, canonical_split)

    if len(out) == 0:
        out["prompt_variant_id"] = pd.Series(dtype=int)
        out["prompt_split"] = pd.Series(dtype=str)
        out["prompt"] = pd.Series(dtype=str)
        return out

    variant_ids = np.resize(
        np.asarray(allowed_variant_ids, dtype=int),
        len(out),
    ).tolist()
    texts = out["raw_text"].astype(str).tolist()
    out["prompt_variant_id"] = variant_ids
    out["prompt_split"] = canonical_split
    out["prompt"] = [
        TASK_PROMPT_FN[task_name](text, variant_id=variant_id)
        for text, variant_id in zip(texts, variant_ids)
    ]

    unexpected_ids = set(out["prompt_variant_id"].astype(int)) - set(allowed_variant_ids)
    if unexpected_ids:
        raise RuntimeError(
            f"[{task_name}] {canonical_split}: unexpected prompt variants {sorted(unexpected_ids)}."
        )
    return out

if SHARED_TASK_DATA is None:
    SHARED_TASK_DATA = prepare_shared_task_data()

AVAILABLE_TASK_NAMES = [
    task_name
    for task_name in TASK_IDS.keys()
    if task_name in SHARED_TASK_DATA and task_name in TASK_PROMPT_FN
]

STEERING_EXAMPLES_PER_CLASS = int(STEERING_TRAIN_PER_CLASS)

experiment_train_by_task = {
    task_name: _prepare_rewrite_pool_df(SHARED_TASK_DATA[task_name]["df_train"], task_name)
    for task_name in AVAILABLE_TASK_NAMES
}

validation_by_task = {}
test_by_task = {}
steering_train_by_task = {}

for task_name in AVAILABLE_TASK_NAMES:
    shared_train_df = experiment_train_by_task[task_name]
    _assert_split_has_both_labels(shared_train_df, task_name, "train")

    if task_name in PAIRED_EVAL_TASKS:
        validation_eval_df = _paired_split_to_rewrite_eval_df(
            SHARED_TASK_DATA[task_name]["df_paired_val"],
            task_name,
        )
        test_eval_df = _paired_split_to_rewrite_eval_df(
            SHARED_TASK_DATA[task_name]["df_paired_test"],
            task_name,
        )
        _assert_paired_target_df(validation_eval_df, task_name, "validation")
        _assert_paired_target_df(test_eval_df, task_name, "test")
    else:
        validation_eval_df = _prepare_rewrite_pool_df(SHARED_TASK_DATA[task_name]["df_val"], task_name)
        test_eval_df = _prepare_rewrite_pool_df(SHARED_TASK_DATA[task_name]["df_test"], task_name)
        _assert_rewrite_eval_df(validation_eval_df, task_name, "validation")
        _assert_rewrite_eval_df(test_eval_df, task_name, "test")

    validation_by_task[task_name] = _assign_prompt_variants(
        validation_eval_df, task_name, split="validation"
    )
    test_by_task[task_name] = _assign_prompt_variants(test_eval_df, task_name, split="test")

    steering_subset_df = _build_steering_subset_from_training_split(
        shared_train_df,
        task_name=task_name,
        per_class_train=STEERING_EXAMPLES_PER_CLASS,
    )
    steering_train_by_task[task_name] = _assign_prompt_variants(
        steering_subset_df, task_name, split="train"
    )

    print(
        f"[{task_name}] shared splits -> "
        f"train={len(SHARED_TASK_DATA[task_name]['df_train'])} | "
        f"validation={len(SHARED_TASK_DATA[task_name]['df_val'])} | "
        f"test={len(SHARED_TASK_DATA[task_name]['df_test'])} | "
        f"steering_sample={len(steering_subset_df)} "
        f"(label0={int((steering_subset_df.gold_label == 0).sum())}, "
        f"label1={int((steering_subset_df.gold_label == 1).sum())})"
    )

def build_task_df(task_name: str, split: str) -> pd.DataFrame:
    if task_name not in AVAILABLE_TASK_NAMES:
        raise KeyError(f"Task not available: {task_name}")

    split_alias = {
        "train": "train",
        "validation": "validation",
        "val": "validation",
        "test": "test",
        "steering_train": "steering_train",
    }
    if split not in split_alias:
        raise KeyError(f"Unsupported split: {split}")

    canonical_split = split_alias[split]
    lookup = {
        "train": experiment_train_by_task,
        "validation": validation_by_task,
        "test": test_by_task,
        "steering_train": steering_train_by_task,
    }
    return lookup[canonical_split][task_name].copy().reset_index(drop=True)

def to_teacher_forcing_df(
        df_task: pd.DataFrame,
        task_name: str,
        split: str = "train",
) -> pd.DataFrame:
    out = df_task.copy().reset_index(drop=True)
    out = _assign_prompt_variants(out, task_name, split=split)
    out["response"] = out["raw_text"].astype(str)
    out["task_id"] = int(TASK_IDS[task_name])
    out["reference"] = out["gold_label"].astype(float)
    return out[["prompt", "response", "reference", "task_id"]]

steering_train_by_task_tf = {
    task_name: to_teacher_forcing_df(steering_train_by_task[task_name], task_name, split="train")
    for task_name in steering_train_by_task.keys()
}

all_steering_train_tf = pd.concat(list(steering_train_by_task_tf.values()), ignore_index=True)

# Build every BERT split from the complete formatted prompts, then train/load the router.
task_router_train_df = _build_task_router_frame("train")
task_router_val_df = _build_task_router_frame("validation")
task_router_test_df = _build_task_router_frame("test")
task_router_train_counts = {
    task_name: int((task_router_train_df["task_name"] == task_name).sum())
    for task_name in TASK_IDS
}
if any(count <= 0 for count in task_router_train_counts.values()):
    raise RuntimeError(f"Every task must have training data, found {task_router_train_counts}.")
print("BERT task-router training rows by task:", task_router_train_counts)
task_router = train_or_load_task_router()
task_router_metrics_signature = {
    "router_artifact_dir": task_router["artifact_dir"],
    "input_representation": "full_formatted_prompt",
    "validation_rows": len(task_router_val_df),
    "test_rows": len(task_router_test_df),
}
task_router_metrics_by_split = load_experiment_checkpoint(
    "task_router_metrics", {}, task_router_metrics_signature
)
task_router["model"].to("cuda" if torch.cuda.is_available() else "cpu")
for split_name, frame in (("validation", task_router_val_df), ("test", task_router_test_df)):
    if split_name in task_router_metrics_by_split:
        print(f"[checkpoint] skipping BERT task-router metrics: {split_name}")
        continue
    predicted = predict_task_ids(frame["text"].astype(str).tolist())
    labels = frame["label"].astype(int).tolist()
    task_router_metrics_by_split[split_name] = {
        "split": split_name,
        "rows": len(frame),
        "accuracy": accuracy_score(labels, predicted),
        "f1_macro": f1_score(labels, predicted, average="macro", zero_division=0),
    }
    save_experiment_checkpoint(
        "task_router_metrics", task_router_metrics_by_split, task_router_metrics_signature
    )
task_router_metrics = [task_router_metrics_by_split[name] for name in ("validation", "test")]
display(pd.DataFrame(task_router_metrics))
task_router["model"].to("cpu").eval()
del task_router_train_df, task_router_val_df, task_router_test_df
cleanup_memory()


Prompt registry validated with the compatibility validator.
Prompt-variant partition (disjoint within each task):


,split,variants_per_task,percentage,implicit_variants,explicit_variants
0,train,70,70.0,35,35
1,validation,10,10.0,5,5
2,test,20,20.0,10,10


[libritts_text_original_to_normalized] shared splits -> train=100502 | validation=800 | test=25336 | steering_sample=100 (label0=50, label1=50)
[paradetox_non_toxic] shared splits -> train=18994 | validation=400 | test=4848 | steering_sample=100 (label0=50, label1=50)


[imdb_positive_sentiment] shared splits -> train=39266 | validation=400 | test=9916 | steering_sample=100 (label0=50, label1=50)
[stanford_politeness_polite] shared splits -> train=652 | validation=400 | test=263 | steering_sample=100 (label0=50, label1=50)
[authormix_woolf_style] shared splits -> train=4877 | validation=400 | test=1319 | steering_sample=100 (label0=50, label1=50)
[setfit_subj_objective] shared splits -> train=7600 | validation=400 | test=2000 | steering_sample=100 (label0=50, label1=50)


BERT task-router training rows by task: {'libritts_text_original_to_normalized': 100502, 'paradetox_non_toxic': 18994, 'imdb_positive_sentiment': 39266, 'stanford_politeness_polite': 652, 'authormix_woolf_style': 4877, 'setfit_subj_objective': 7600}
Loaded cached BERT task router: /home/ubuntu/ATAS/.atas_cache/atas_artifacts/bert_task_router/b3adddbafa3104aa
[checkpoint] loaded task_router_metrics <- /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/2143919937f89e4b/task_router_metrics.pkl


[checkpoint] skipping BERT task-router metrics: validation
[checkpoint] skipping BERT task-router metrics: test


,split,rows,accuracy,f1_macro
0,validation,2800,0.999643,0.999583
1,test,43682,1.000000,1.000000


# Loading model

In [14]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, use_fast=True, local_files_only=True, cache_dir=os.path.expanduser("~/.cache/huggingface/hub"))
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

if hasattr(tokenizer, "truncation_side"):
    tokenizer.truncation_side = "left"

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA-capable GPU is required to run this notebook.")

active_gpu_name = torch.cuda.get_device_name()
if active_gpu_name != TARGET_GPU_NAME:
    raise RuntimeError(
        f"Expected {TARGET_GPU_NAME!r}, but CUDA exposed {active_gpu_name!r}. "
        "Restart the kernel and run the notebook from the first cell."
    )

major, minor = torch.cuda.get_device_capability()
dtype = torch.bfloat16 if major >= 8 and torch.cuda.is_bf16_supported() else torch.float16

# Fast SDPA on supported GPUs; RTX 8000 uses memory-efficient SDPA with math fallback.
flash_sdp_supported = major >= 8
torch.backends.cuda.enable_flash_sdp(flash_sdp_supported)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)
print(f"GPU: {active_gpu_name} | capability={major}.{minor} | flash_sdp={flash_sdp_supported}")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    torch_dtype=dtype,
    device_map="auto",
    attn_implementation="sdpa",
    local_files_only=True,
    cache_dir=os.path.expanduser("~/.cache/huggingface/hub"),
)

model.config.pad_token_id = tokenizer.pad_token_id
if hasattr(model, "generation_config"):
    model.generation_config.pad_token_id = tokenizer.pad_token_id
model.eval()
model_dtype = getattr(model, "dtype", torch.float32)


GPU: Quadro RTX 8000 | capability=7.5 | flash_sdp=False


In [15]:
all_layers = ModelUtils.find_transformer_layers(model)
NUM_LAYERS = len(all_layers)
LLAMA_ROBUSTNESS_RHO = 0.75
if any(layer < 0 or layer >= NUM_LAYERS for layer in LAYERS_TO_EXTRACT):
    raise ValueError(f"Steering layers {LAYERS_TO_EXTRACT} are outside [0, {NUM_LAYERS - 1}]")

print(f"Total transformer layers: {NUM_LAYERS}")
print(f"Layers used for steering: {LAYERS_TO_EXTRACT}")

BASE_ALPHAS = [0, 0.2, 0.4, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0]

ATAS_SIMILARITY_THRESHOLD = LLAMA_ROBUSTNESS_RHO

Total transformer layers: 16
Layers used for steering: [7, 8, 9]


In [16]:
def _normalize_wer_text(text: str) -> List[str]:
    text = "" if text is None else str(text)
    text = " ".join(text.strip().split())
    return text.split() if text else []

def _edit_distance_tokens(ref_tokens: List[str], hyp_tokens: List[str]) -> int:
    m, n = len(ref_tokens), len(hyp_tokens)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev = dp[0]
        dp[0] = i
        for j in range(1, n + 1):
            cur = dp[j]
            if ref_tokens[i - 1] == hyp_tokens[j - 1]:
                dp[j] = prev
            else:
                dp[j] = 1 + min(prev, dp[j], dp[j - 1])
            prev = cur
    return dp[n]

def compute_corpus_wer(preds: List[str], refs: List[str]) -> float:
    if len(preds) != len(refs):
        raise ValueError("preds and refs must have the same length.")

    total_words = 0
    total_errs = 0

    for pred, ref in zip(preds, refs):
        ref_toks = _normalize_wer_text(ref)
        hyp_toks = _normalize_wer_text(pred)
        total_words += len(ref_toks)
        total_errs += _edit_distance_tokens(ref_toks, hyp_toks)

    return 0.0 if total_words == 0 else float(total_errs) / float(total_words)

def evaluate_rewriting_with_classifier(
        steered_model: torch.nn.Module,
        tokenizer_obj: AutoTokenizer,
        df: pd.DataFrame,
        clf_key: str,
        raw_text_col: str = "raw_text",
        label_col: str = "reference",
        max_new_tokens: int = MAX_NEW_TOKENS,
        batch_size: int = EVAL_GENERATION_BATCH_SIZE,
        forced_task_ids: Optional[List[int]] = None,
) -> Dict[str, Any]:
    batch_size = max(int(EVAL_GENERATION_BATCH_SIZE), int(batch_size))
    prompts = df["prompt"].tolist()
    rewrites = batched_generate(
        steered_model, tokenizer_obj, prompts,
        batch_size=batch_size,
        max_new_tokens=max_new_tokens,
        generation_kwargs=GENERATION_KWARGS,
        forced_task_ids=forced_task_ids,
        empty_cache_every=0, gc_every=50,
    )
    raw_texts = df[raw_text_col].tolist()

    preds_orig = predict_local_labels(raw_texts, clf_key=clf_key, batch_size=128)
    preds_rew = predict_local_labels(rewrites, clf_key=clf_key, batch_size=128)
    refusal_mask = [
        " ".join(str(rewrite).split()) == ATAS_REFUSAL_TEXT
        for rewrite in rewrites
    ]
    refusal_count = int(sum(refusal_mask))
    bert_f1 = compute_bertscore(rewrites, raw_texts)

    if label_col in df.columns:
        gold = df[label_col].astype(int).tolist()
    else:
        gold = [REWRITE_TARGET_LABEL] * len(preds_rew)

    # A refusal remains in the denominator and is always an error,
    # regardless of the label predicted for its stock refusal text.
    correct = sum(
        int((not refused) and g == p)
        for g, p, refused in zip(gold, preds_rew, refusal_mask)
    )
    acc = correct / max(1, len(preds_rew))

    source_acc = None
    label_flip_rate = None
    if "gold_label" in df.columns:
        source_gold = df["gold_label"].astype(int).tolist()
        source_correct = sum(
            int((not refused) and g == p)
            for g, p, refused in zip(source_gold, preds_rew, refusal_mask)
        )
        source_acc = source_correct / max(1, len(preds_rew))
        label_flip_rate = sum(
            int((not refused) and s != p)
            for s, p, refused in zip(source_gold, preds_rew, refusal_mask)
        ) / max(1, len(preds_rew))

    return {
        "accuracy": acc,
        "target_success_rate": acc,
        "source_label_accuracy": source_acc,
        "label_flip_rate": label_flip_rate,
        "refusal_count": refusal_count,
        "refusal_rate": refusal_count / max(1, len(preds_rew)),
        "bertscore_f1": bert_f1,
        "bertscore_f1_all": bert_f1,
        "rewrites": rewrites,
        "pred_orig": preds_orig,
        "pred_rewrite": preds_rew,
        "selection_metric_name": "accuracy",
        "selection_metric_value": acc,
        "selection_score": acc,
        "metric_direction": "higher_is_better",
    }

def evaluate_rewriting_with_target_wer(
        steered_model: torch.nn.Module,
        tokenizer_obj: AutoTokenizer,
        df: pd.DataFrame,
        raw_text_col: str = "raw_text",
        target_text_col: str = "target_text",
        max_new_tokens: int = MAX_NEW_TOKENS,
        batch_size: int = EVAL_GENERATION_BATCH_SIZE,
        forced_task_ids: Optional[List[int]] = None,
) -> Dict[str, Any]:
    if target_text_col not in df.columns:
        raise KeyError(f"Missing column {target_text_col} for WER evaluation.")

    batch_size = max(int(EVAL_GENERATION_BATCH_SIZE), int(batch_size))
    prompts = df["prompt"].tolist()
    rewrites = batched_generate(
        steered_model, tokenizer_obj, prompts,
        batch_size=batch_size,
        max_new_tokens=max_new_tokens,
        generation_kwargs=GENERATION_KWARGS,
        forced_task_ids=forced_task_ids,
        empty_cache_every=0, gc_every=50,
    )
    raw_texts = df[raw_text_col].tolist()
    target_texts = df[target_text_col].tolist()

    target_wer = compute_corpus_wer(rewrites, target_texts)
    bert_f1_source = compute_bertscore(rewrites, raw_texts)
    bert_f1_target = compute_bertscore(rewrites, target_texts)

    return {
        "accuracy": float("nan"),
        "target_success_rate": float("nan"),
        "source_label_accuracy": None,
        "label_flip_rate": None,
        "bertscore_f1": bert_f1_source,
        "bertscore_f1_all": bert_f1_source,
        "bertscore_f1_target": bert_f1_target,
        "target_wer": target_wer,
        "rewrites": rewrites,
        "target_texts": target_texts,
        "selection_metric_name": "wer",
        "selection_metric_value": target_wer,
        "selection_score": -target_wer,
        "metric_direction": "lower_is_better",
    }

def evaluate_rewriting(
        steered_model: torch.nn.Module,
        tokenizer_obj: AutoTokenizer,
        df: pd.DataFrame,
        task_name: str,
        max_new_tokens: int = MAX_NEW_TOKENS,
        batch_size: int = EVAL_GENERATION_BATCH_SIZE,
        forced_task_ids: Optional[List[int]] = None,
) -> Dict[str, Any]:
    if task_name in PAIRED_EVAL_TASKS:
        return evaluate_rewriting_with_target_wer(
            steered_model=steered_model,
            tokenizer_obj=tokenizer_obj,
            df=df,
            max_new_tokens=max_new_tokens,
            batch_size=batch_size,
            forced_task_ids=forced_task_ids,
        )

    return evaluate_rewriting_with_classifier(
        steered_model=steered_model,
        tokenizer_obj=tokenizer_obj,
        df=df,
        clf_key=task_name,
        batch_size=batch_size,
        max_new_tokens=max_new_tokens,
        forced_task_ids=forced_task_ids,
    )

def _task_requires_classifier_eval(task_name: str) -> bool:
    return task_name not in PAIRED_EVAL_TASKS

def _attach_primary_metrics(pruned: Dict[str, Any], res: Dict[str, Any]) -> Dict[str, Any]:
    for key in [
        "accuracy",
        "target_success_rate",
        "source_label_accuracy",
        "label_flip_rate",
        "bertscore_f1",
        "bertscore_f1_all",
        "bertscore_f1_target",
        "target_wer",
        "selection_metric_name",
        "selection_metric_value",
        "selection_score",
        "metric_direction",
    ]:
        if key in res:
            pruned[key] = res[key]
    return pruned

def _metric_value(res: Dict[str, Any]) -> float:
    return float(res["selection_metric_value"])

def _metric_name(res: Dict[str, Any]) -> str:
    return str(res["selection_metric_name"])

def _metric_direction(res: Dict[str, Any]) -> str:
    return str(res.get("metric_direction", "higher_is_better"))

def _metric_is_better(candidate_res: Dict[str, Any], best_score: float) -> bool:
    return float(candidate_res["selection_score"]) > float(best_score)


def _select_best_alpha_from_scores(
    alpha_scores: List[Tuple[float, float]],
    *,
    atol: float = 1e-12,
) -> Tuple[float, float]:
    if not alpha_scores:
        raise ValueError("alpha_scores cannot be empty.")

    best_score = max(float(score) for _, score in alpha_scores)
    tied_alphas = sorted(
        float(alpha)
        for alpha, score in alpha_scores
        if math.isclose(float(score), float(best_score), abs_tol=float(atol))
    )

    selected_alpha = tied_alphas[1] if len(tied_alphas) >= 2 else tied_alphas[0]
    return float(selected_alpha), float(best_score)


def _baseline_improvement(current_res: Dict[str, Any], baseline_res: Dict[str, Any]) -> float:
    cur = float(current_res["selection_metric_value"])
    base = float(baseline_res["selection_metric_value"])
    if _metric_direction(current_res) == "lower_is_better":
        return base - cur
    return cur - base

def _metric_print_label(metric_name: str) -> str:
    return "WER" if metric_name == "wer" else metric_name

#  Baseline on test set

In [17]:
baseline_checkpoint_signature = {"tasks": list(test_by_task), "kind": "unsteered_test"}
baseline_results: Dict[str, Dict[str, Any]] = load_experiment_checkpoint(
    "baseline_results", {}, baseline_checkpoint_signature
)

for task_name, df_test in test_by_task.items():
        if task_name in baseline_results:
            print(f"[checkpoint] skipping completed baseline task: {task_name}")
            continue
        _assert_no_forward_hooks(model, where=f"(baseline test: {task_name})")

        clf = None
        orig_device = None
        moved_clf = False

        if _task_requires_classifier_eval(task_name):
            clf = local_classifiers[task_name]["model"]
            orig_device = get_model_device(clf)

            if torch.cuda.is_available() and orig_device.type != "cuda":
                clf.to("cuda")
                moved_clf = True

        try:
            res = evaluate_rewriting(
                steered_model=model,
                tokenizer_obj=tokenizer,
                df=df_test,
                task_name=task_name,
                batch_size=EVAL_GENERATION_BATCH_SIZE,
                max_new_tokens=GENERATION_KWARGS["max_new_tokens"],
            )

            pruned = prune_eval_result(res)
            pruned = _attach_primary_metrics(pruned, res)
            baseline_results[task_name] = pruned
            save_experiment_checkpoint(
                "baseline_results", baseline_results, baseline_checkpoint_signature
            )

            metric_name = _metric_name(res)
            metric_value = _metric_value(res)
            print(
                f"  => BASELINE {task_name}: "
                f"{_metric_print_label(metric_name)}={metric_value:.3f} | "
                f"bertF1={res.get('bertscore_f1', float('nan')):.3f}"
            )

            del res
            gc.collect()

        finally:
            if moved_clf and clf is not None and orig_device is not None:
                clf.to(orig_device)
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

print("baseline tasks:", list(baseline_results.keys()))


[checkpoint] loaded baseline_results <- /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/fac4310a3e3aa135/baseline_results.pkl
[checkpoint] skipping completed baseline task: libritts_text_original_to_normalized
[checkpoint] skipping completed baseline task: paradetox_non_toxic


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

[checkpoint] saved baseline_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/fac4310a3e3aa135/baseline_results.pkl
  => BASELINE imdb_positive_sentiment: accuracy=0.907 | bertF1=0.926


[checkpoint] saved baseline_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/fac4310a3e3aa135/baseline_results.pkl
  => BASELINE stanford_politeness_polite: accuracy=0.935 | bertF1=0.920


[checkpoint] saved baseline_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/fac4310a3e3aa135/baseline_results.pkl
  => BASELINE authormix_woolf_style: accuracy=0.574 | bertF1=0.918


[checkpoint] saved baseline_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/fac4310a3e3aa135/baseline_results.pkl
  => BASELINE setfit_subj_objective: accuracy=0.496 | bertF1=0.913


baseline tasks: ['libritts_text_original_to_normalized', 'paradetox_non_toxic', 'imdb_positive_sentiment', 'stanford_politeness_polite', 'authormix_woolf_style', 'setfit_subj_objective']


# Steering with CMD

In [17]:
cmd_checkpoint_signature = {
    "tasks": list(steering_train_by_task_tf),
    "alphas": [float(alpha) for alpha in BASE_ALPHAS],
    "mediator": "CMD",
}
cmd_state = load_experiment_checkpoint(
    "cmd_results",
    {
        "val_results": {float(alpha): {} for alpha in BASE_ALPHAS},
        "test_results": {},
        "best_alpha": {},
    },
    cmd_checkpoint_signature,
)
cmd_val_results: Dict[float, Dict[str, Dict[str, Any]]] = cmd_state["val_results"]
cmd_results: Dict[str, Dict[str, Any]] = cmd_state["test_results"]
cmd_best_alpha: Dict[str, float] = cmd_state["best_alpha"]

def _save_cmd_state() -> None:
    save_experiment_checkpoint("cmd_results", cmd_state, cmd_checkpoint_signature)

In [19]:
for task_name, steering_train_tf in steering_train_by_task_tf.items():
    if task_name in cmd_results:
        print(f"[checkpoint] skipping completed CMD task: {task_name}")
        continue
    val_df_task = validation_by_task[task_name]
    test_df_task = test_by_task[task_name]
    print(f"\n=== CMD TRAIN for {task_name} | train={len(steering_train_tf)} | val={len(val_df_task)} | test={len(test_df_task)} ===")
    train_df_vec = steering_train_tf.copy()
    executor_mm = Executor(
        model=model,
        tokenizer=tokenizer,
        train_df=train_df_vec,
        apply_from_token=-1,
        apply_from_mode=ApplyFromModeEnum.PROMPT_END,
        task_type=TaskTypeEnum.UNSUPERVISED,
        mediator_type=MediatorTypeEnum.CMD,
        layers_to_extract=LAYERS_TO_EXTRACT,
        batch_size=4,
        alpha=BASE_ALPHAS[0],
        use_last_token_for_response=True,
        tokens_window=-1,
        generation_kwargs=GENERATION_KWARGS,
        padding_side=tokenizer.padding_side,
        truncation_side=getattr(tokenizer, "truncation_side", "left"),
        dtype=getattr(model, "dtype", torch.float32),
    )

    _assert_no_forward_hooks(model, where=f"(before CMD steering: {task_name})")

    with executor_mm.representation_extractor() as steered_mm:
        clf = None
        orig_device = None
        moved_clf = False
        if _task_requires_classifier_eval(task_name):
            clf = local_classifiers[task_name]["model"]
            orig_device = get_model_device(clf)
            if torch.cuda.is_available() and orig_device.type != "cuda":
                clf.to("cuda")
                moved_clf = True

        try:
            alpha_scores: List[Tuple[float, float]] = []

            for alpha in BASE_ALPHAS:
                cached_val = cmd_val_results[float(alpha)].get(task_name)
                if cached_val is not None:
                    print(f"  -> checkpoint val alpha={alpha} on {task_name}")
                    alpha_scores.append((float(alpha), float(cached_val["selection_score"])))
                    continue
                steered_mm.alpha = alpha
                print(f"  -> val alpha={alpha} on {task_name}")

                res = evaluate_rewriting(
                    steered_model=steered_mm,
                    tokenizer_obj=tokenizer,
                    df=val_df_task,
                    task_name=task_name,
                    batch_size=EVAL_GENERATION_BATCH_SIZE,
                    max_new_tokens=GENERATION_KWARGS["max_new_tokens"],
                )

                pruned = prune_eval_result(res)
                pruned = _attach_primary_metrics(pruned, res)
                cmd_val_results[alpha][task_name] = pruned
                _save_cmd_state()

                metric_name = _metric_name(res)
                metric_value = _metric_value(res)
                print(
                    f"     val {_metric_print_label(metric_name)}={metric_value:.3f} | "
                    f"bertF1={res.get('bertscore_f1', float('nan')):.3f}"
                )

                alpha_scores.append((float(alpha), float(res["selection_score"])))

                del res
                gc.collect()

            best_alpha, best_score = _select_best_alpha_from_scores(alpha_scores)
            cmd_best_alpha[task_name] = float(best_alpha)
            _save_cmd_state()
            steered_mm.alpha = best_alpha
            print(f"  => best alpha for {task_name}: {best_alpha}")

            res = evaluate_rewriting(
                steered_model=steered_mm,
                tokenizer_obj=tokenizer,
                df=test_df_task,
                task_name=task_name,
                batch_size=EVAL_GENERATION_BATCH_SIZE,
                max_new_tokens=GENERATION_KWARGS["max_new_tokens"],
            )

            pruned = prune_eval_result(res)
            pruned = _attach_primary_metrics(pruned, res)
            pruned["selected_alpha"] = float(best_alpha)

            base = baseline_results[task_name]
            pruned["baseline_metric_name"] = base["selection_metric_name"]
            pruned["baseline_metric_value"] = float(base["selection_metric_value"])
            pruned["baseline_bertscore_f1"] = float(base.get("bertscore_f1", float("nan")))
            pruned["delta_vs_baseline_primary_metric"] = (
                float(res["selection_metric_value"]) - float(base["selection_metric_value"])
            )
            pruned["improvement_vs_baseline"] = _baseline_improvement(res, base)

            cmd_results[task_name] = pruned
            _save_cmd_state()

            metric_name = _metric_name(res)
            metric_value = _metric_value(res)
            baseline_metric_value = float(base["selection_metric_value"])
            delta = pruned["delta_vs_baseline_primary_metric"]

            print(
                f"  => TEST {task_name}: alpha={best_alpha} | "
                f"{_metric_print_label(metric_name)}={metric_value:.3f} | "
                f"baseline_{metric_name}={baseline_metric_value:.3f} | "
                f"delta={delta:+.3f} | "
                f"bertF1={res.get('bertscore_f1', float('nan')):.3f}"
            )

            del res
            gc.collect()

        finally:
            if moved_clf and clf is not None and orig_device is not None:
                clf.to(orig_device)
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    _assert_no_forward_hooks(model, where=f"(after CMD steering: {task_name})")

    del executor_mm
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("forward_hooks:", _count_forward_hooks(model))
print("cmd_best_alpha:", cmd_best_alpha)


=== CMD TRAIN for libritts_text_original_to_normalized | train=100 | val=400 | test=12672 ===


Extracting activations:   0%|          | 0/25 [00:00<?, ?it/s]

  -> val alpha=0 on libritts_text_original_to_normalized


[checkpoint] saved cmd_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/fac4310a3e3aa135/cmd_results.pkl
     val WER=0.953 | bertF1=0.965
  -> val alpha=0.2 on libritts_text_original_to_normalized


[checkpoint] saved cmd_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/fac4310a3e3aa135/cmd_results.pkl
     val WER=0.901 | bertF1=0.966
  -> val alpha=0.4 on libritts_text_original_to_normalized


...

[checkpoint] saved cmd_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/fac4310a3e3aa135/cmd_results.pkl
     val accuracy=0.672 | bertF1=0.910
  -> val alpha=1.8 on setfit_subj_objective


[checkpoint] saved cmd_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/fac4310a3e3aa135/cmd_results.pkl
     val accuracy=0.715 | bertF1=0.904
  -> val alpha=2.0 on setfit_subj_objective


[checkpoint] saved cmd_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/fac4310a3e3aa135/cmd_results.pkl
     val accuracy=0.728 | bertF1=0.895
[checkpoint] saved cmd_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/fac4310a3e3aa135/cmd_results.pkl
  => best alpha for setfit_subj_objective: 2.0


[checkpoint] saved cmd_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/fac4310a3e3aa135/cmd_results.pkl
  => TEST setfit_subj_objective: alpha=2.0 | accuracy=0.749 | baseline_accuracy=0.496 | delta=+0.253 | bertF1=0.890


forward_hooks: 0
cmd_best_alpha: {'libritts_text_original_to_normalized': 0.8, 'paradetox_non_toxic': 2.0, 'imdb_positive_sentiment': 1.8, 'stanford_politeness_polite': 2.0, 'authormix_woolf_style': 2.0, 'setfit_subj_objective': 2.0}


# Steering with CPCA

In [20]:
cpca_checkpoint_signature = {
    "tasks": list(steering_train_by_task_tf),
    "alphas": [float(alpha) for alpha in BASE_ALPHAS],
    "mediator": "CPCA",
}
cpca_state = load_experiment_checkpoint(
    "cpca_results",
    {
        "val_results": {float(alpha): {} for alpha in BASE_ALPHAS},
        "test_results": {},
        "best_alpha": {},
    },
    cpca_checkpoint_signature,
)
cpca_val_results: Dict[float, Dict[str, Dict[str, Any]]] = cpca_state["val_results"]
cpca_results: Dict[str, Dict[str, Any]] = cpca_state["test_results"]
cpca_best_alpha: Dict[str, float] = cpca_state["best_alpha"]

def _save_cpca_state() -> None:
    save_experiment_checkpoint("cpca_results", cpca_state, cpca_checkpoint_signature)

for task_name, steering_train_tf in steering_train_by_task_tf.items():
    if task_name in cpca_results:
        print(f"[checkpoint] skipping completed CPCA task: {task_name}")
        continue
    val_df_task = validation_by_task[task_name]
    test_df_task = test_by_task[task_name]
    print(f"\n=== CPCA TRAIN for {task_name} | train={len(steering_train_tf)} | val={len(val_df_task)} | test={len(test_df_task)} ===")
    train_df_vec = steering_train_tf.copy()
    executor_mm = Executor(
        model=model,
        tokenizer=tokenizer,
        train_df=train_df_vec,
        apply_from_token=-1,
        apply_from_mode=ApplyFromModeEnum.PROMPT_END,
        task_type=TaskTypeEnum.UNSUPERVISED,
        mediator_type=MediatorTypeEnum.CPCA,
        layers_to_extract=LAYERS_TO_EXTRACT,
        batch_size=4,
        alpha=BASE_ALPHAS[0],
        use_last_token_for_response=True,
        tokens_window=-1,
        generation_kwargs=GENERATION_KWARGS,
        padding_side=tokenizer.padding_side,
        truncation_side=getattr(tokenizer, "truncation_side", "left"),
        dtype=getattr(model, "dtype", torch.float32),
    )

    _assert_no_forward_hooks(model, where=f"(before CPCA steering: {task_name})")

    with executor_mm.representation_extractor() as steered_mm:
        clf = None
        orig_device = None
        moved_clf = False
        if _task_requires_classifier_eval(task_name):
            clf = local_classifiers[task_name]["model"]
            orig_device = get_model_device(clf)
            if torch.cuda.is_available() and orig_device.type != "cuda":
                clf.to("cuda")
                moved_clf = True

        try:
            alpha_scores: List[Tuple[float, float]] = []

            for alpha in BASE_ALPHAS:
                cached_val = cpca_val_results[float(alpha)].get(task_name)
                if cached_val is not None:
                    print(f"  -> checkpoint val alpha={alpha} on {task_name}")
                    alpha_scores.append((float(alpha), float(cached_val["selection_score"])))
                    continue
                steered_mm.alpha = alpha
                print(f"  -> val alpha={alpha} on {task_name}")

                res = evaluate_rewriting(
                    steered_model=steered_mm,
                    tokenizer_obj=tokenizer,
                    df=val_df_task,
                    task_name=task_name,
                    batch_size=EVAL_GENERATION_BATCH_SIZE,
                    max_new_tokens=GENERATION_KWARGS["max_new_tokens"],
                )

                pruned = prune_eval_result(res)
                pruned = _attach_primary_metrics(pruned, res)
                cpca_val_results[alpha][task_name] = pruned
                _save_cpca_state()

                metric_name = _metric_name(res)
                metric_value = _metric_value(res)
                print(
                    f"     val {_metric_print_label(metric_name)}={metric_value:.3f} | "
                    f"bertF1={res.get('bertscore_f1', float('nan')):.3f}"
                )

                alpha_scores.append((float(alpha), float(res["selection_score"])))

                del res
                gc.collect()

            best_alpha, best_score = _select_best_alpha_from_scores(alpha_scores)
            cpca_best_alpha[task_name] = float(best_alpha)
            _save_cpca_state()
            steered_mm.alpha = best_alpha
            print(f"  => best alpha for {task_name}: {best_alpha}")

            res = evaluate_rewriting(
                steered_model=steered_mm,
                tokenizer_obj=tokenizer,
                df=test_df_task,
                task_name=task_name,
                batch_size=EVAL_GENERATION_BATCH_SIZE,
                max_new_tokens=GENERATION_KWARGS["max_new_tokens"],
            )

            pruned = prune_eval_result(res)
            pruned = _attach_primary_metrics(pruned, res)
            pruned["selected_alpha"] = float(best_alpha)

            base = baseline_results[task_name]
            pruned["baseline_metric_name"] = base["selection_metric_name"]
            pruned["baseline_metric_value"] = float(base["selection_metric_value"])
            pruned["baseline_bertscore_f1"] = float(base.get("bertscore_f1", float("nan")))
            pruned["delta_vs_baseline_primary_metric"] = (
                float(res["selection_metric_value"]) - float(base["selection_metric_value"])
            )
            pruned["improvement_vs_baseline"] = _baseline_improvement(res, base)
            cpca_results[task_name] = pruned
            _save_cpca_state()

            metric_name = _metric_name(res)
            metric_value = _metric_value(res)
            baseline_metric_value = float(base["selection_metric_value"])
            delta = pruned["delta_vs_baseline_primary_metric"]

            print(
                f"  => TEST {task_name}: alpha={best_alpha} | "
                f"{_metric_print_label(metric_name)}={metric_value:.3f} | "
                f"baseline_{metric_name}={baseline_metric_value:.3f} | "
                f"delta={delta:+.3f} | "
                f"bertF1={res.get('bertscore_f1', float('nan')):.3f}"
            )

            del res
            gc.collect()

        finally:
            if moved_clf and clf is not None and orig_device is not None:
                clf.to(orig_device)
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    _assert_no_forward_hooks(model, where=f"(after CPCA steering: {task_name})")

    del executor_mm
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("forward_hooks:", _count_forward_hooks(model))
print("cpca_best_alpha:", cpca_best_alpha)


=== CPCA TRAIN for libritts_text_original_to_normalized | train=100 | val=400 | test=12672 ===


Extracting activations:   0%|          | 0/25 [00:00<?, ?it/s]

  -> val alpha=0 on libritts_text_original_to_normalized


[checkpoint] saved cpca_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/fac4310a3e3aa135/cpca_results.pkl
     val WER=0.953 | bertF1=0.965
  -> val alpha=0.2 on libritts_text_original_to_normalized


[checkpoint] saved cpca_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/fac4310a3e3aa135/cpca_results.pkl
     val WER=0.966 | bertF1=0.965
  -> val alpha=0.4 on libritts_text_original_to_normalized


...

[checkpoint] saved cpca_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/fac4310a3e3aa135/cpca_results.pkl
     val accuracy=0.610 | bertF1=0.927
  -> val alpha=1.8 on setfit_subj_objective


[checkpoint] saved cpca_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/fac4310a3e3aa135/cpca_results.pkl
     val accuracy=0.618 | bertF1=0.924
  -> val alpha=2.0 on setfit_subj_objective


[checkpoint] saved cpca_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/fac4310a3e3aa135/cpca_results.pkl
     val accuracy=0.627 | bertF1=0.918
[checkpoint] saved cpca_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/fac4310a3e3aa135/cpca_results.pkl
  => best alpha for setfit_subj_objective: 2.0


[checkpoint] saved cpca_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/fac4310a3e3aa135/cpca_results.pkl
  => TEST setfit_subj_objective: alpha=2.0 | accuracy=0.633 | baseline_accuracy=0.496 | delta=+0.137 | bertF1=0.911


forward_hooks: 0
cpca_best_alpha: {'libritts_text_original_to_normalized': 0.0, 'paradetox_non_toxic': 0.2, 'imdb_positive_sentiment': 0.0, 'stanford_politeness_polite': 1.6, 'authormix_woolf_style': 2.0, 'setfit_subj_objective': 2.0}


# Steering with ATAS-CMD

In [18]:
def make_profile_gen_kwargs(base_kwargs: dict, k: int) -> dict:
    kw = dict(base_kwargs)

    kw.pop("max_new_tokens", None)
    kw.pop("min_new_tokens", None)
    kw.pop("max_length", None)
    kw.pop("min_length", None)
    kw.pop("max_time", None)

    kw["do_sample"] = False
    kw["num_beams"] = 1
    kw["min_new_tokens"] = int(k)

    kw["eos_token_id"] = None
    kw["forced_eos_token_id"] = None

    kw.pop("stopping_criteria", None)
    kw.pop("stop_strings", None)

    return kw


def _resolve_profile_device(model_obj) -> torch.device:
    for candidate in (
        model_obj,
        getattr(model_obj, "model", None),
        getattr(model_obj, "base_model", None),
    ):
        if candidate is None:
            continue
        try:
            return get_model_device(candidate)
        except Exception:
            pass
    if torch.cuda.is_available():
        return torch.device(f"cuda:{torch.cuda.current_device()}")
    return torch.device("cpu")


def _gpu_inference_info(model_obj=None) -> dict:
    if not torch.cuda.is_available():
        return {"device": "cpu"}

    if model_obj is not None:
        dev = _resolve_profile_device(model_obj)
        if dev.type == "cuda":
            idx = dev.index if dev.index is not None else torch.cuda.current_device()
        else:
            idx = torch.cuda.current_device()
    else:
        idx = torch.cuda.current_device()

    prop = torch.cuda.get_device_properties(idx)
    return {
        "device": f"cuda:{idx}",
        "name": prop.name,
        "cc": f"{prop.major}.{prop.minor}",
        "total_mem_gib": float(prop.total_memory) / (1024 ** 3),
        "torch_cuda": str(torch.version.cuda),
        "torch": str(torch.__version__),
    }
@contextlib.contextmanager
def _active_generate_wrapper(model_obj):
    if getattr(model_obj, "_hooks_active", False):
        yield model_obj
        return

    strat = None
    for name in ("strategy", "_strategy"):
        try:
            strat = getattr(model_obj, name)
            break
        except Exception:
            pass

    if strat is not None and hasattr(model_obj, "__enter__") and hasattr(model_obj, "__exit__"):
        with model_obj as active_model:
            yield active_model
    else:
        yield model_obj

def _format_bytes(n: int) -> str:
    n = int(n)
    if n < 1024:
        return f"{n} B"
    if n < 1024 ** 2:
        return f"{n / 1024:.2f} KiB"
    if n < 1024 ** 3:
        return f"{n / 1024 ** 2:.2f} MiB"
    return f"{n / 1024 ** 3:.2f} GiB"


def _reset_cuda_runtime(model_obj) -> None:
    device = _resolve_profile_device(model_obj)
    gc.collect()
    if device.type == "cuda" and torch.cuda.is_available():
        torch.cuda.synchronize(device)
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
        torch.cuda.synchronize(device)


def _drop_strategy_gpu_caches(strategy) -> None:
    if strategy is None:
        return

    for name in ("_tensor_cache", "_alpha_dev_cache"):
        if hasattr(strategy, name):
            try:
                cur = getattr(strategy, name)
                if isinstance(cur, dict):
                    setattr(strategy, name, {})
                else:
                    setattr(strategy, name, None)
            except Exception:
                pass

    gc.collect()


@contextlib.contextmanager
def _atas_fully_disabled_for_profile(wrapper):
    old_strategy = getattr(wrapper, "strategy", None)
    had_strategy = old_strategy is not None

    try:
        if had_strategy:
            _drop_strategy_gpu_caches(old_strategy)
            wrapper.strategy = None
            _reset_cuda_runtime(wrapper)
        yield wrapper
    finally:
        if had_strategy:
            wrapper.strategy = old_strategy
            if hasattr(old_strategy, "prepare_alpha"):
                try:
                    old_strategy.prepare_alpha(wrapper.alpha)
                except Exception:
                    pass


def _effective_special_ids(model_obj, tokenizer_obj):
    pad_id = getattr(tokenizer_obj, "pad_token_id", None)
    if pad_id is None:
        gen_cfg = getattr(model_obj, "generation_config", None)
        if gen_cfg is not None:
            pad_id = getattr(gen_cfg, "pad_token_id", None)
    if pad_id is None:
        cfg = getattr(model_obj, "config", None)
        if cfg is not None:
            pad_id = getattr(cfg, "pad_token_id", None)

    eos_id = getattr(tokenizer_obj, "eos_token_id", None)
    if eos_id is None:
        gen_cfg = getattr(model_obj, "generation_config", None)
        if gen_cfg is not None:
            eos_id = getattr(gen_cfg, "eos_token_id", None)
    if eos_id is None:
        cfg = getattr(model_obj, "config", None)
        if cfg is not None:
            eos_id = getattr(cfg, "eos_token_id", None)

    return {
        "pad_id": None if pad_id is None else int(pad_id),
        "eos_id": None if eos_id is None else int(eos_id),
    }


def _count_new_tokens(out_ids: torch.Tensor, prompt_len: int, pad_id, eos_id=None) -> int:
    comp = out_ids[:, prompt_len:]
    if comp.numel() == 0:
        return 0

    if pad_id is None:
        return int(comp.numel())

    if eos_id is None or pad_id != eos_id:
        return int((comp != int(pad_id)).sum().item())

    total = 0
    eos_id = int(eos_id)

    for row in comp:
        eos_pos = (row == eos_id).nonzero(as_tuple=False)
        if eos_pos.numel() > 0:
            total += int(eos_pos[0].item()) + 1
        else:
            total += int(row.numel())

    return total


def _tensor_bytes(t: torch.Tensor) -> int:
    return int(t.numel()) * int(t.element_size())


def _sum_tensors_bytes_numel(obj, *, seen: set[int]) -> tuple[int, int]:
    if torch.is_tensor(obj):
        tid = id(obj)
        if tid in seen:
            return 0, 0
        seen.add(tid)
        return _tensor_bytes(obj), int(obj.numel())

    if isinstance(obj, dict):
        total_b, total_n = 0, 0
        for v in obj.values():
            b, n = _sum_tensors_bytes_numel(v, seen=seen)
            total_b += b
            total_n += n
        return total_b, total_n

    if isinstance(obj, (list, tuple)):
        total_b, total_n = 0, 0
        for v in obj:
            b, n = _sum_tensors_bytes_numel(v, seen=seen)
            total_b += b
            total_n += n
        return total_b, total_n

    return 0, 0


def steering_weight_stats(wrapper, *, include_gpu_cache: bool = True) -> dict:
    strat = getattr(wrapper, "strategy", None)
    if strat is None:
        return {
            "steering_bytes_cpu": 0,
            "steering_bytes_gpu_cache": 0,
            "steering_bytes_total": 0,
            "steering_mib_total": 0.0,
            "steering_params_cpu_numel": 0,
        }

    seen_cpu: set[int] = set()
    seen_gpu: set[int] = set()
    cpu_bytes = 0
    cpu_numel = 0

    cpu_objs = [
        getattr(strat, "_pc_t_cpu", None),
        getattr(strat, "_dv_n_cpu", None),
        getattr(strat, "_alpha_table_cpu", None),
        getattr(strat, "correction_vectors", None),
        getattr(strat, "prompt_centroids", None),
        getattr(strat, "correct_centroids", None),
        getattr(strat, "incorrect_centroids", None),
    ]

    for obj in cpu_objs:
        b, n = _sum_tensors_bytes_numel(obj, seen=seen_cpu)
        cpu_bytes += b
        cpu_numel += n

    gpu_cache_bytes = 0
    if include_gpu_cache:
        gpu_objs = [
            getattr(strat, "_tensor_cache", None),
            getattr(strat, "_alpha_dev_cache", None),
        ]
        for obj in gpu_objs:
            b, _ = _sum_tensors_bytes_numel(obj, seen=seen_gpu)
            gpu_cache_bytes += b

    total = int(cpu_bytes + gpu_cache_bytes)

    return {
        "steering_bytes_cpu": int(cpu_bytes),
        "steering_bytes_gpu_cache": int(gpu_cache_bytes),
        "steering_bytes_total": int(total),
        "steering_mib_total": float(total) / (1024 ** 2),
        "steering_params_cpu_numel": int(cpu_numel),
    }

def _map_tensors(obj, fn):
    if torch.is_tensor(obj):
        return fn(obj)
    if isinstance(obj, dict):
        return {k: _map_tensors(v, fn) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_map_tensors(v, fn) for v in obj]
    if isinstance(obj, tuple):
        return tuple(_map_tensors(v, fn) for v in obj)
    return obj


def atas_minimal_replication_payload(
    wrapper,
    *,
    use_fp16_tensors: bool = True,
    include_provenance: bool = False,
) -> dict:
    strat = getattr(wrapper, "strategy", None)
    if strat is None:
        return {"format": "atas_minimal_replication_v1", "strategy": None, "wrapper": None}

    if strat.__class__.__name__ != "AtasSteeringStrategy":
        raise TypeError(f"Expected AtasSteeringStrategy, got {type(strat).__name__}.")

    if hasattr(strat, "export_params"):
        routing_centroids, correction_vectors, task_ids = strat.export_params()
    else:
        pc_t = getattr(strat, "_pc_t_cpu", None)
        dv_n = getattr(strat, "_dv_n_cpu", None)

        if not torch.is_tensor(pc_t):
            raise RuntimeError("Missing _pc_t_cpu in strategy.")
        if not isinstance(dv_n, dict) or not dv_n:
            raise RuntimeError("Missing _dv_n_cpu in strategy.")

        routing_centroids = {
            int(strat.master_prompt_layer): pc_t.transpose(0, 1).detach().cpu().contiguous()
        }
        correction_vectors = {
            int(layer): t.detach().cpu().contiguous()
            for layer, t in dv_n.items()
            if torch.is_tensor(t)
        }
        task_ids = None if getattr(strat, "task_ids", None) is None else [int(t) for t in strat.task_ids]

    payload = {
        "format": "atas_minimal_replication_v1",
        "strategy": {
            "class": "AtasSteeringStrategy",
            "master_prompt_layer": int(strat.master_prompt_layer),
            "target_layers": [int(x) for x in getattr(strat, "target_layers", [])],
            "task_ids": None if task_ids is None else [int(t) for t in task_ids],
            "similarity_threshold": float(getattr(strat, "similarity_threshold", 0.0)),
            "use_last_token_for_prompt": bool(getattr(strat, "use_last_token_for_prompt", True)),
            "normalize_eps": float(getattr(strat, "_normalize_eps", 1e-12)),
            "routing_centroids": {
                int(k): v.detach().cpu().contiguous()
                for k, v in routing_centroids.items()
                if torch.is_tensor(v)
            },
            "correction_vectors": {
                int(k): v.detach().cpu().contiguous()
                for k, v in correction_vectors.items()
                if torch.is_tensor(v)
            },
        },
        "wrapper": {
            "alpha": (
                {int(k): float(v) for k, v in wrapper.alpha.items()}
                if isinstance(wrapper.alpha, dict)
                else float(wrapper.alpha)
            ),
            "apply_from_token": int(wrapper.apply_from_token),
            "apply_from_mode": wrapper.apply_from_mode.name,
            "tokens_window": int(wrapper.tokens_window),
            "padding_side": str(wrapper.padding_side),
        },
    }

    if include_provenance:
        base_model = getattr(wrapper, "base_model", None)
        tok = getattr(wrapper, "_tokenizer", None)
        cfg = getattr(base_model, "config", None) if base_model is not None else None

        payload["provenance"] = {
            "model_class": None if base_model is None else type(base_model).__name__,
            "tokenizer_class": None if tok is None else type(tok).__name__,
            "pad_token_id": None if tok is None else getattr(tok, "pad_token_id", None),
            "eos_token_id": None if tok is None else getattr(tok, "eos_token_id", None),
            "hidden_size": None if cfg is None else getattr(cfg, "hidden_size", None),
            "num_hidden_layers": None if cfg is None else getattr(cfg, "num_hidden_layers", None),
        }

    if use_fp16_tensors:
        def _to_storage_dtype(x):
            if torch.is_tensor(x) and x.is_floating_point():
                return x.to(torch.float16)
            return x
        payload = _map_tensors(payload, _to_storage_dtype)

    return payload


def atas_network_params_and_disk(
    wrapper,
    *,
    use_fp16_on_disk: bool = True,
    include_provenance: bool = False,
) -> dict:
    strat = getattr(wrapper, "strategy", None)
    if strat is None:
        return {
            "net_numel": 0,
            "disk_bytes": 0,
        }

    payload = atas_minimal_replication_payload(
        wrapper,
        use_fp16_tensors=use_fp16_on_disk,
        include_provenance=include_provenance,
    )

    seen_net: set[int] = set()
    _, net_numel = _sum_tensors_bytes_numel(payload, seen=seen_net)

    buf = io.BytesIO()
    torch.save(payload, buf)
    disk_bytes = buf.tell()

    return {
        "net_numel": int(net_numel),
        "disk_bytes": int(disk_bytes),
    }


def warmup_generate(
    model_obj,
    tokenizer_obj,
    prompts,
    *,
    batch_size: int,
    max_new_tokens: int,
    max_prompt_tokens=None,
    warmup_batches: int = 1,
    **gen_kwargs,
) -> None:
    if warmup_batches <= 0 or len(prompts) == 0:
        return

    device = _resolve_profile_device(model_obj)
    w_prompts = prompts[: min(len(prompts), batch_size * warmup_batches)]
    enc = tokenizer_obj(
        w_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_prompt_tokens,
        add_special_tokens=False,
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.inference_mode():
        _ = model_obj.generate(**enc, max_new_tokens=max_new_tokens, **gen_kwargs)

    del enc, _
    if device.type == "cuda" and torch.cuda.is_available():
        torch.cuda.synchronize(device)


def timed_generate_profile(
    model_obj,
    tokenizer_obj,
    prompts,
    *,
    batch_size: int,
    max_new_tokens: int,
    max_prompt_tokens=None,
    fixed_new_tokens: int | None = None,
    **gen_kwargs,
) -> dict:
    device = _resolve_profile_device(model_obj)
    use_cuda = (device.type == "cuda") and torch.cuda.is_available()

    total_wall_s = 0.0
    total_gpu_ms = 0.0
    total_new_tokens = 0
    total_decode_steps = 0
    peak_alloc = 0
    peak_reserved = 0

    special_ids = _effective_special_ids(model_obj, tokenizer_obj)
    pad_id = special_ids["pad_id"]
    eos_id = special_ids["eos_id"]

    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        enc = tokenizer_obj(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_prompt_tokens,
            add_special_tokens=False,
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        prompt_len = int(enc["input_ids"].shape[1])

        if use_cuda:
            torch.cuda.synchronize(device)
            torch.cuda.reset_peak_memory_stats(device)

            start_ev = torch.cuda.Event(enable_timing=True)
            end_ev = torch.cuda.Event(enable_timing=True)

            start_ev.record()
            t0 = time.perf_counter()
            with torch.inference_mode():
                out = model_obj.generate(**enc, max_new_tokens=max_new_tokens, **gen_kwargs)
            end_ev.record()
            torch.cuda.synchronize(device)
            t1 = time.perf_counter()

            wall = t1 - t0
            gpu_ms = float(start_ev.elapsed_time(end_ev))

            peak_alloc = max(peak_alloc, int(torch.cuda.max_memory_allocated(device)))
            peak_reserved = max(peak_reserved, int(torch.cuda.max_memory_reserved(device)))
        else:
            t0 = time.perf_counter()
            with torch.inference_mode():
                out = model_obj.generate(**enc, max_new_tokens=max_new_tokens, **gen_kwargs)
            t1 = time.perf_counter()
            wall = t1 - t0
            gpu_ms = float("nan")

        total_wall_s += float(wall)
        if not (isinstance(gpu_ms, float) and math.isnan(gpu_ms)):
            total_gpu_ms += float(gpu_ms)

        gen_len = int(out.shape[1] - prompt_len)
        total_decode_steps += gen_len

        if fixed_new_tokens is not None:
            if gen_len != int(fixed_new_tokens):
                raise RuntimeError(
                    f"Expected exactly {fixed_new_tokens} new tokens, got {gen_len}"
                )
            total_new_tokens += int(out.shape[0] * gen_len)
        else:
            total_new_tokens += _count_new_tokens(out, prompt_len, pad_id, eos_id)

        del enc, out

    tok_per_s = (total_new_tokens / total_wall_s) if total_wall_s > 0 else float("nan")
    ms_per_tok = (1000.0 * total_wall_s / total_new_tokens) if total_new_tokens > 0 else float("nan")
    ms_per_step = (1000.0 * total_wall_s / total_decode_steps) if total_decode_steps > 0 else float("nan")

    return {
        "device": str(device),
        "wall_s": float(total_wall_s),
        "gpu_ms": float(total_gpu_ms) if use_cuda else float("nan"),
        "new_tokens": int(total_new_tokens),
        "decode_steps": int(total_decode_steps),
        "tok_per_s": float(tok_per_s),
        "ms_per_tok": float(ms_per_tok),
        "ms_per_step": float(ms_per_step),
        "peak_alloc_bytes": int(peak_alloc),
        "peak_reserved_bytes": int(peak_reserved),
    }


def _safe_wall_ratio(base_stats: dict, atas_stats: dict, tol: float = 0.05) -> float:
    b = int(base_stats.get("new_tokens", 0))
    a = int(atas_stats.get("new_tokens", 0))
    if b <= 0:
        return float("nan")
    rel_diff = abs(a - b) / b
    if rel_diff > tol:
        return float("nan")
    return float(atas_stats["wall_s"] / base_stats["wall_s"]) if base_stats["wall_s"] > 0 else float("nan")


def _profile_sample_mean_std(values) -> Tuple[float, float]:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return float("nan"), float("nan")
    return float(arr.mean()), float(arr.std(ddof=1)) if arr.size > 1 else 0.0


def _summarize_profile_runs(runs: List[dict]) -> dict:
    if not runs:
        raise ValueError("At least one profiling run is required.")
    summary = {"device": str(runs[0]["device"]), "repeats": len(runs)}
    for metric in (
        "wall_s", "gpu_ms", "new_tokens", "decode_steps", "tok_per_s",
        "ms_per_tok", "ms_per_step", "peak_alloc_bytes", "peak_reserved_bytes",
    ):
        mean, std = _profile_sample_mean_std([run[metric] for run in runs])
        summary[metric] = mean
        summary[f"{metric}_std"] = std
    return summary


_gpu_info = _gpu_inference_info(model)
print("=== Inference device info ===")
print(_gpu_info)

PROFILE_N = 100
PROFILE_REPEATS = 5
_atas_profile_rows = []
_profile_cache = {}

all_steering_train_tf = pd.concat(list(steering_train_by_task_tf.values()), ignore_index=True)

id_to_task_name = {int(v): str(k) for k, v in TASK_IDS.items()}
atas_task_ids = sorted(int(x) for x in all_steering_train_tf["task_id"].dropna().unique())

missing_task_ids = [tid for tid in atas_task_ids if tid not in id_to_task_name]
if missing_task_ids:
    raise ValueError(f"task_ids present in all_steering_train_tf but missing from TASK_IDS: {missing_task_ids}")

cmd_alpha_by_task_id = {
    int(task_id): float(cmd_best_alpha[task_name])
    for task_name, task_id in TASK_IDS.items()
}
if sorted(cmd_alpha_by_task_id) != atas_task_ids:
    raise ValueError("ATAS-CMD alpha mapping must cover every routed task exactly.")
# Each routed sample receives the validation-selected alpha of the task inferred by ATAS.
print("atas_task_id_mapping =", {tid: id_to_task_name[tid] for tid in atas_task_ids})


print("=== ATAS TRAIN on all tasks ===")
print("all_steering_train_tf size =", len(all_steering_train_tf))

executor_atas = Executor(
    model=model,
    tokenizer=tokenizer,
    train_df=all_steering_train_tf,
    apply_from_token=-1,
    apply_from_mode=ApplyFromModeEnum.PROMPT_END,
    task_type=TaskTypeEnum.UNSUPERVISED,
    mediator_type=MediatorTypeEnum.ATAS_CMD,
    layers_to_extract=LAYERS_TO_EXTRACT,
    batch_size=4,
    use_last_token_for_response=True,
    alpha=dict(cmd_alpha_by_task_id),
    tokens_window=-1,
    generation_kwargs=GENERATION_KWARGS,
    padding_side=tokenizer.padding_side,
    truncation_side=getattr(tokenizer, "truncation_side", "left"),
    dtype=getattr(model, "dtype", torch.float32),
    atas_use_last_token_for_prompt=ATAS_USE_LAST_TOKEN_FOR_PROMPT,
    decode_chunk_max=1,
    free_training_artifacts_after_build=True,
)

executor_atas.atas_similarity_threshold = ATAS_SIMILARITY_THRESHOLD

atas_cmd_checkpoint_signature = {
    "tasks": list(test_by_task),
    "best_alpha": dict(cmd_best_alpha),
    "similarity_threshold": float(ATAS_SIMILARITY_THRESHOLD),
    "mediator": "ATAS_CMD",
    "alpha_policy": "validation_selected_alpha_of_atas_inferred_task",
    "profile_repeats": int(PROFILE_REPEATS),
}
atas_cmd_state = load_experiment_checkpoint(
    "atas_cmd_results",
    {"test_results": {}, "ood_results": {}, "profile_rows": []},
    atas_cmd_checkpoint_signature,
)
atas_cmd_results: Dict[str, Dict[str, Any]] = atas_cmd_state["test_results"]
atas_cmd_ood_results: Dict[str, Dict[str, Any]] = atas_cmd_state["ood_results"]
_atas_profile_rows = list(atas_cmd_state.get("profile_rows", []))
_profile_cache = {str(row["task"]): row for row in _atas_profile_rows}

_assert_no_forward_hooks(model, where="(before ATAS steering context)")

t0 = time.perf_counter()
with executor_atas.representation_extractor() as atas_steered_model:
    t_extract = time.perf_counter() - t0
    print(f"[ATAS] estrazione vettori completata in {t_extract/60:.2f} min")

    for task_name, df_test in test_by_task.items():
        if task_name in atas_cmd_results:
            print(f"[checkpoint] skipping completed ATAS-CMD task: {task_name}")
            continue
        task_alpha = float(cmd_best_alpha[task_name])
        print(f"\n=== ATAS eval task={task_name} | nominal_task_alpha={task_alpha} | alpha_policy=inferred_task ===")

        df_val = validation_by_task[task_name]
        prof_prompts = df_val["prompt"].tolist()[: min(PROFILE_N, len(df_val))]

        moved_clf = False
        clf = None
        orig_device = None

        try:
            if (task_name not in _profile_cache) and len(prof_prompts) > 0:
                PROFILE_K = 100
                gen_kw = make_profile_gen_kwargs(GENERATION_KWARGS, PROFILE_K)
                max_new = PROFILE_K
                bs_prof = min(8, len(prof_prompts))

                with _atas_fully_disabled_for_profile(atas_steered_model):
                    warmup_generate(
                        atas_steered_model,
                        tokenizer,
                        prof_prompts,
                        batch_size=bs_prof,
                        max_new_tokens=max_new,
                        max_prompt_tokens=executor_atas.max_prompt_tokens,
                        warmup_batches=1,
                        **gen_kw,
                    )
                    base_stats = timed_generate_profile(
                        atas_steered_model,
                        tokenizer,
                        prof_prompts,
                        batch_size=bs_prof,
                        max_new_tokens=max_new,
                        max_prompt_tokens=executor_atas.max_prompt_tokens,
                        fixed_new_tokens=max_new,
                        **gen_kw,
                    )
                atas_steered_model.alpha = dict(cmd_alpha_by_task_id)
                _reset_cuda_runtime(atas_steered_model)
                
                with _active_generate_wrapper(atas_steered_model) as prof_model:
                    warmup_generate(
                        prof_model,
                        tokenizer,
                        prof_prompts,
                        batch_size=bs_prof,
                        max_new_tokens=max_new,
                        max_prompt_tokens=executor_atas.max_prompt_tokens,
                        warmup_batches=1,
                        **gen_kw,
                    )
                    atas_stats = timed_generate_profile(
                        prof_model,
                        tokenizer,
                        prof_prompts,
                        batch_size=bs_prof,
                        max_new_tokens=max_new,
                        max_prompt_tokens=executor_atas.max_prompt_tokens,
                        fixed_new_tokens=max_new,
                        **gen_kw,
                    )

                baseline_profile_runs = [base_stats]
                atas_profile_runs = [atas_stats]
                for profile_repeat in range(1, PROFILE_REPEATS):
                    with _atas_fully_disabled_for_profile(atas_steered_model):
                        _reset_cuda_runtime(atas_steered_model)
                        repeat_base_stats = timed_generate_profile(
                            atas_steered_model, tokenizer, prof_prompts,
                            batch_size=bs_prof, max_new_tokens=max_new,
                            max_prompt_tokens=executor_atas.max_prompt_tokens,
                            fixed_new_tokens=max_new, **gen_kw,
                        )
                    atas_steered_model.alpha = dict(cmd_alpha_by_task_id)
                    _reset_cuda_runtime(atas_steered_model)
                    with _active_generate_wrapper(atas_steered_model) as prof_model:
                        repeat_atas_stats = timed_generate_profile(
                            prof_model, tokenizer, prof_prompts,
                            batch_size=bs_prof, max_new_tokens=max_new,
                            max_prompt_tokens=executor_atas.max_prompt_tokens,
                            fixed_new_tokens=max_new, **gen_kw,
                        )
                    baseline_profile_runs.append(repeat_base_stats)
                    atas_profile_runs.append(repeat_atas_stats)

                paired_wall_ratios = [
                    _safe_wall_ratio(base_run, atas_run, tol=0.05)
                    for base_run, atas_run in zip(baseline_profile_runs, atas_profile_runs)
                ]
                paired_per_tok_ratios = [
                    atas_run["ms_per_tok"] / base_run["ms_per_tok"]
                    if base_run["ms_per_tok"] > 0 else float("nan")
                    for base_run, atas_run in zip(baseline_profile_runs, atas_profile_runs)
                ]
                paired_per_step_ratios = [
                    atas_run["ms_per_step"] / base_run["ms_per_step"]
                    if base_run["ms_per_step"] > 0 else float("nan")
                    for base_run, atas_run in zip(baseline_profile_runs, atas_profile_runs)
                ]
                base_stats = _summarize_profile_runs(baseline_profile_runs)
                atas_stats = _summarize_profile_runs(atas_profile_runs)

                w_stats = steering_weight_stats(atas_steered_model, include_gpu_cache=True)
                netdisk = atas_network_params_and_disk(atas_steered_model, use_fp16_on_disk=True)

                slowdown_wall, slowdown_wall_std = _profile_sample_mean_std(paired_wall_ratios)
                slowdown_per_tok, slowdown_per_tok_std = _profile_sample_mean_std(paired_per_tok_ratios)
                slowdown_per_step, slowdown_per_step_std = _profile_sample_mean_std(paired_per_step_ratios)

                wall_ratio_str = f"{slowdown_wall:.2f}x" if not math.isnan(slowdown_wall) else "nan(token_mismatch)"

                print(
                    "  [PROFILE][task=%s][alpha=%.3f] repeats=%d N=%d | baseline wall=%.3f+/-%.3fs, ms/tok=%.2f+/-%.2f | "
                    "ATAS wall=%.3f+/-%.3fs, ms/tok=%.2f+/-%.2f | wall_ratio=%.2f+/-%.2fx | slowdown_per_tok=%.2f+/-%.2fx | slowdown_per_step=%.2f+/-%.2fx"
                    % (
                        task_name,
                        task_alpha,
                        PROFILE_REPEATS,
                        len(prof_prompts),
                        base_stats["wall_s"], base_stats["wall_s_std"],
                        base_stats["ms_per_tok"], base_stats["ms_per_tok_std"],
                        atas_stats["wall_s"], atas_stats["wall_s_std"],
                        atas_stats["ms_per_tok"], atas_stats["ms_per_tok_std"],
                        slowdown_wall, slowdown_wall_std,
                        slowdown_per_tok, slowdown_per_tok_std,
                        slowdown_per_step, slowdown_per_step_std,
                    )
                )

                print(
                    "  [STEERING] cpu=%s | gpu_cache=%s | total=%s (%.2f MiB) | cpu_numel=%d"
                    % (
                        _format_bytes(w_stats["steering_bytes_cpu"]),
                        _format_bytes(w_stats["steering_bytes_gpu_cache"]),
                        _format_bytes(w_stats["steering_bytes_total"]),
                        w_stats["steering_mib_total"],
                        w_stats["steering_params_cpu_numel"],
                    )
                )

                print("NET numel:", netdisk["net_numel"])
                print("DISK bytes:", netdisk["disk_bytes"])

                row = {
                    "task": str(task_name),
                    "profile_repeats": int(PROFILE_REPEATS),
                    "alpha": float(task_alpha),
                    "device": str(base_stats["device"]),
                    "baseline_wall_s": float(base_stats["wall_s"]),
                    "baseline_wall_s_std": float(base_stats["wall_s_std"]),
                    "baseline_ms_per_tok": float(base_stats["ms_per_tok"]),
                    "baseline_ms_per_tok_std": float(base_stats["ms_per_tok_std"]),
                    "baseline_tok_per_s": float(base_stats["tok_per_s"]),
                    "baseline_tok_per_s_std": float(base_stats["tok_per_s_std"]),
                    "baseline_new_tokens": int(base_stats["new_tokens"]),
                    "baseline_new_tokens_std": float(base_stats["new_tokens_std"]),
                    "baseline_peak_alloc_bytes": int(base_stats["peak_alloc_bytes"]),
                    "baseline_peak_alloc_bytes_std": float(base_stats["peak_alloc_bytes_std"]),
                    "atas_wall_s": float(atas_stats["wall_s"]),
                    "atas_wall_s_std": float(atas_stats["wall_s_std"]),
                    "atas_ms_per_tok": float(atas_stats["ms_per_tok"]),
                    "atas_ms_per_tok_std": float(atas_stats["ms_per_tok_std"]),
                    "atas_tok_per_s": float(atas_stats["tok_per_s"]),
                    "atas_tok_per_s_std": float(atas_stats["tok_per_s_std"]),
                    "atas_new_tokens": int(atas_stats["new_tokens"]),
                    "atas_new_tokens_std": float(atas_stats["new_tokens_std"]),
                    "atas_peak_alloc_bytes": int(atas_stats["peak_alloc_bytes"]),
                    "atas_peak_alloc_bytes_std": float(atas_stats["peak_alloc_bytes_std"]),
                    "wall_ratio_x": float(slowdown_wall),
                    "wall_ratio_x_std": float(slowdown_wall_std),
                    "slowdown_per_tok_x": float(slowdown_per_tok),
                    "slowdown_per_tok_x_std": float(slowdown_per_tok_std),
                    "steering_mib_total": float(w_stats["steering_mib_total"]),
                    "steering_bytes_cpu": int(w_stats["steering_bytes_cpu"]),
                    "steering_bytes_gpu_cache": int(w_stats["steering_bytes_gpu_cache"]),
                    "steering_params_cpu_numel": int(w_stats["steering_params_cpu_numel"]),
                    "net_numel": int(netdisk["net_numel"]),
                    "disk_bytes": int(netdisk["disk_bytes"]),
                    "baseline_ms_per_step": float(base_stats["ms_per_step"]),
                    "baseline_ms_per_step_std": float(base_stats["ms_per_step_std"]),
                    "atas_ms_per_step": float(atas_stats["ms_per_step"]),
                    "atas_ms_per_step_std": float(atas_stats["ms_per_step_std"]),
                    "slowdown_per_step_x": float(slowdown_per_step),
                    "slowdown_per_step_x_std": float(slowdown_per_step_std),
                }

                _profile_cache[task_name] = row
                _atas_profile_rows.append(row)

            if _task_requires_classifier_eval(task_name):
                clf = local_classifiers[task_name]["model"]
                orig_device = get_model_device(clf)
                if torch.cuda.is_available() and orig_device.type != "cuda":
                    clf.to("cuda")
                    moved_clf = True

            atas_steered_model.alpha = dict(cmd_alpha_by_task_id)


            with _active_generate_wrapper(atas_steered_model) as active_model:
                res = evaluate_rewriting(
                    steered_model=active_model,
                    tokenizer_obj=tokenizer,
                    df=df_test,
                    task_name=task_name,
                    batch_size=EVAL_GENERATION_BATCH_SIZE,
                    max_new_tokens=GENERATION_KWARGS["max_new_tokens"],
                )

            pruned = prune_eval_result(res)
            pruned = _attach_primary_metrics(pruned, res)
            pruned["eval_task_alpha"] = float(task_alpha)
            pruned["alpha_policy"] = "validation_selected_alpha_of_atas_inferred_task"
            pruned["selected_alpha_by_task_id"] = dict(cmd_alpha_by_task_id)

            base = baseline_results[task_name]
            pruned["baseline_metric_name"] = base["selection_metric_name"]
            pruned["baseline_metric_value"] = float(base["selection_metric_value"])
            pruned["baseline_bertscore_f1"] = float(base.get("bertscore_f1", float("nan")))
            pruned["delta_vs_baseline_primary_metric"] = (
                float(res["selection_metric_value"]) - float(base["selection_metric_value"])
            )
            pruned["improvement_vs_baseline"] = _baseline_improvement(res, base)

            atas_cmd_results[task_name] = pruned
            atas_cmd_state["profile_rows"] = list(_atas_profile_rows)
            save_experiment_checkpoint(
                "atas_cmd_results", atas_cmd_state, atas_cmd_checkpoint_signature
            )

            metric_name = _metric_name(res)
            metric_value = _metric_value(res)
            baseline_metric_value = float(base["selection_metric_value"])
            delta = pruned["delta_vs_baseline_primary_metric"]
            print(
                f"     {task_name:34s} {_metric_print_label(metric_name)}={metric_value:.3f} | "
                f"baseline_{metric_name}={baseline_metric_value:.3f} | "
                f"delta={delta:+.3f} | "
                f"bertF1={res.get('bertscore_f1', float('nan')):.3f}"
            )

            del res
            gc.collect()

        finally:
            if moved_clf and clf is not None and orig_device is not None:
                clf.to(orig_device)
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

_assert_no_forward_hooks(model, where="(after ATAS steering context)")

del executor_atas
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("forward_hooks:", _count_forward_hooks(model))

print("\n=== Profiling summary (per-task means and sample standard deviations over repetitions) ===")
print(f"Each task uses {PROFILE_REPEATS} repetitions; every *_std column uses ddof=1.")
if len(_atas_profile_rows) == 0:
    print("No profiling rows collected.")
else:
    prof_df = pd.DataFrame(_atas_profile_rows)
    mean_row = prof_df[[
        "baseline_ms_per_tok", "atas_ms_per_tok", "slowdown_per_tok_x", "wall_ratio_x",
        "baseline_wall_s", "atas_wall_s",
        "baseline_new_tokens", "atas_new_tokens",
        "steering_mib_total", "steering_bytes_cpu", "steering_bytes_gpu_cache",
        "steering_params_cpu_numel",
        "net_numel", "disk_bytes",
    ]].mean(numeric_only=True)
    print(mean_row.to_string())
    display(prof_df)

=== Inference device info ===
{'device': 'cuda:0', 'name': 'Quadro RTX 8000', 'cc': '7.5', 'total_mem_gib': 47.26666259765625, 'torch_cuda': '12.1', 'torch': '2.5.1+cu121'}
atas_task_id_mapping = {0: 'libritts_text_original_to_normalized', 1: 'paradetox_non_toxic', 2: 'imdb_positive_sentiment', 3: 'stanford_politeness_polite', 4: 'authormix_woolf_style', 5: 'setfit_subj_objective'}
=== ATAS TRAIN on all tasks ===
all_steering_train_tf size = 600


Extracting activations:   0%|          | 0/150 [00:00<?, ?it/s]

Executor teacher forcing capped 14/600 responses at 512 tokens (longest before truncation: 926).


[ATAS] estrazione vettori completata in 0.23 min

=== ATAS eval task=libritts_text_original_to_normalized | nominal_task_alpha=0.8 | alpha_policy=inferred_task ===


  [PROFILE][task=libritts_text_original_to_normalized][alpha=0.800] repeats=5 N=100 | baseline wall=33.038+/-0.570s, ms/tok=3.30+/-0.06 | ATAS wall=36.597+/-0.768s, ms/tok=3.66+/-0.08 | wall_ratio=1.11+/-0.02x | slowdown_per_tok=1.11+/-0.02x | slowdown_per_step=1.11+/-0.02x
  [STEERING] cpu=240.02 KiB | gpu_cache=192.02 KiB | total=432.05 KiB (0.42 MiB) | cpu_numel=86022
NET numel: 49152
DISK bytes: 100624


...


=== ATAS eval task=setfit_subj_objective | nominal_task_alpha=2.0 | alpha_policy=inferred_task ===


  [PROFILE][task=setfit_subj_objective][alpha=2.000] repeats=5 N=100 | baseline wall=33.620+/-0.878s, ms/tok=3.36+/-0.09 | ATAS wall=36.657+/-0.386s, ms/tok=3.67+/-0.04 | wall_ratio=1.09+/-0.03x | slowdown_per_tok=1.09+/-0.03x | slowdown_per_step=1.09+/-0.03x
  [STEERING] cpu=240.02 KiB | gpu_cache=192.02 KiB | total=432.05 KiB (0.42 MiB) | cpu_numel=86022
NET numel: 49152
DISK bytes: 100624


[checkpoint] saved atas_cmd_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/2143919937f89e4b/atas_cmd_results.pkl
     setfit_subj_objective              accuracy=0.747 | baseline_accuracy=0.496 | delta=+0.250 | bertF1=0.890


forward_hooks: 0

=== Profiling summary (per-task means and sample standard deviations over repetitions) ===
Each task uses 5 repetitions; every *_std column uses ddof=1.
baseline_ms_per_tok               3.370377
atas_ms_per_tok                   3.716028
slowdown_per_tok_x                1.103125
wall_ratio_x                      1.103125
baseline_wall_s                  33.703769
atas_wall_s                      37.160278
baseline_new_tokens           10000.000000
atas_new_tokens               10000.000000
steering_mib_total                0.421921
steering_bytes_cpu           245784.000000
steering_bytes_gpu_cache     196632.000000
steering_params_cpu_numel     86022.000000
net_numel                     49152.000000
disk_bytes                   100624.000000


,task,profile_repeats,alpha,device,baseline_wall_s,baseline_wall_s_std,baseline_ms_per_tok,baseline_ms_per_tok_std,baseline_tok_per_s,baseline_tok_per_s_std,...,steering_bytes_gpu_cache,steering_params_cpu_numel,net_numel,disk_bytes,baseline_ms_per_step,baseline_ms_per_step_std,atas_ms_per_step,atas_ms_per_step_std,slowdown_per_step_x,slowdown_per_step_x_std
0,libritts_text_original_to_normalized,5,0.8,cuda:0,33.038491,0.569574,3.303849,0.056957,302.749806,5.260050,...,196632,86022,49152,100624,25.414224,0.438134,28.151512,0.590480,1.107805,0.020938
1,paradetox_non_toxic,5,2.0,cuda:0,33.192932,1.022100,3.319293,0.102210,301.489538,8.955096,...,196632,86022,49152,100624,25.533025,0.786231,27.802524,0.448974,1.089803,0.041095
2,imdb_positive_sentiment,5,1.8,cuda:0,36.096384,1.146492,3.609638,0.114649,277.251687,8.486834,...,196632,86022,49152,100624,27.766449,0.881917,30.677132,0.330464,1.105437,0.024735
3,stanford_politeness_polite,5,2.0,cuda:0,32.712696,0.553284,3.271270,0.055328,305.762254,5.214221,...,196632,86022,49152,100624,25.163613,0.425603,28.309053,0.470365,1.125309,0.029117
4,authormix_woolf_style,5,2.0,cuda:0,33.561650,0.958161,3.356165,0.095816,298.147720,8.258696,...,196632,86022,49152,100624,25.816654,0.737047,28.371274,0.472522,1.099568,0.032642
5,setfit_subj_objective,5,2.0,cuda:0,33.620461,0.877527,3.362046,0.087753,297.595690,7.556478,...,196632,86022,49152,100624,25.861893,0.675020,28.197483,0.296870,1.090827,0.027240


# Steering with ATAS-CPCA

In [19]:
all_steering_train_tf = pd.concat(list(steering_train_by_task_tf.values()), ignore_index=True)

id_to_task_name = {int(v): str(k) for k, v in TASK_IDS.items()}
atas_task_ids = sorted(int(x) for x in all_steering_train_tf["task_id"].dropna().unique())

missing_task_ids = [tid for tid in atas_task_ids if tid not in id_to_task_name]
if missing_task_ids:
    raise ValueError(
        f"task_ids present in all_steering_train_tf but missing from TASK_IDS: {missing_task_ids}"
    )

atas_task_names = [id_to_task_name[tid] for tid in atas_task_ids]

cpca_alpha_by_task_id = {
    int(TASK_IDS[task_name]): float(cpca_best_alpha[task_name])
    for task_name in atas_task_names
}

expected_task_ids = atas_task_ids
got_task_ids = sorted(cpca_alpha_by_task_id.keys())
if got_task_ids != expected_task_ids:
    raise ValueError(
        "cpca_alpha_by_task_id does not cover every ATAS CPCA task exactly.\n"
        f"expected_task_ids={expected_task_ids}\n"
        f"got_task_ids={got_task_ids}"
    )

print("alpha_policy = alpha_of_atas_selected_task")
print("cpca_alpha_by_task_id =", cpca_alpha_by_task_id)

print("=== ATAS TRAIN CPCA on all tasks ===")
print("all_steering_train_tf size =", len(all_steering_train_tf))

executor_atas = Executor(
    model=model,
    tokenizer=tokenizer,
    train_df=all_steering_train_tf,
    apply_from_token=-1,
    apply_from_mode=ApplyFromModeEnum.PROMPT_END,
    task_type=TaskTypeEnum.UNSUPERVISED,
    mediator_type=MediatorTypeEnum.ATAS_CPCA,
    layers_to_extract=LAYERS_TO_EXTRACT,
    batch_size=4,
    use_last_token_for_response=True,
    alpha=cpca_alpha_by_task_id,
    tokens_window=-1,
    generation_kwargs=GENERATION_KWARGS,
    padding_side=tokenizer.padding_side,
    truncation_side=getattr(tokenizer, "truncation_side", "left"),
    dtype=getattr(model, "dtype", torch.float32),
    atas_use_last_token_for_prompt=ATAS_USE_LAST_TOKEN_FOR_PROMPT,
    free_training_artifacts_after_build=True,
)

executor_atas.atas_similarity_threshold = ATAS_SIMILARITY_THRESHOLD

atas_cpca_checkpoint_signature = {
    "tasks": list(test_by_task),
    "best_alpha": dict(cpca_best_alpha),
    "similarity_threshold": float(ATAS_SIMILARITY_THRESHOLD),
    "mediator": "ATAS_CPCA",
}
atas_cpca_state = load_experiment_checkpoint(
    "atas_cpca_results",
    {"test_results": {}, "ood_results": {}},
    atas_cpca_checkpoint_signature,
)
atas_cpca_results: Dict[str, Dict[str, Any]] = atas_cpca_state["test_results"]
atas_cpca_ood_results: Dict[str, Dict[str, Any]] = atas_cpca_state["ood_results"]

_assert_no_forward_hooks(model, where="(before ATAS steering context)")

with executor_atas.representation_extractor() as atas_steered_model:
    for task_name, df_test in test_by_task.items():
        if task_name in atas_cpca_results:
            print(f"[checkpoint] skipping completed ATAS-CPCA task: {task_name}")
            continue
        task_alpha = float(cpca_best_alpha[task_name])
        print(
            f"\n=== ATAS eval task={task_name} | "
            f"eval_task_alpha={task_alpha} | alpha_policy=alpha_of_atas_selected_task ==="
        )

        clf = None
        orig_device = None
        moved_clf = False

        if _task_requires_classifier_eval(task_name):
            clf = local_classifiers[task_name]["model"]
            orig_device = get_model_device(clf)
            if torch.cuda.is_available() and orig_device.type != "cuda":
                clf.to("cuda")
                moved_clf = True

        try:
            atas_steered_model.alpha = cpca_alpha_by_task_id

            res = evaluate_rewriting(
                steered_model=atas_steered_model,
                tokenizer_obj=tokenizer,
                df=df_test,
                task_name=task_name,
                batch_size=EVAL_GENERATION_BATCH_SIZE,
                max_new_tokens=GENERATION_KWARGS["max_new_tokens"],
            )

            pruned = prune_eval_result(res)
            pruned = _attach_primary_metrics(pruned, res)

            pruned["eval_task_alpha"] = float(task_alpha)
            pruned["alpha_policy"] = "alpha_of_atas_selected_task"
            pruned["alpha_by_task_id"] = dict(cpca_alpha_by_task_id)

            base = baseline_results[task_name]
            pruned["baseline_metric_name"] = base["selection_metric_name"]
            pruned["baseline_metric_value"] = float(base["selection_metric_value"])
            pruned["baseline_bertscore_f1"] = float(base.get("bertscore_f1", float("nan")))
            pruned["delta_vs_baseline_primary_metric"] = (
                float(res["selection_metric_value"]) - float(base["selection_metric_value"])
            )
            pruned["improvement_vs_baseline"] = _baseline_improvement(res, base)

            atas_cpca_results[task_name] = pruned
            save_experiment_checkpoint(
                "atas_cpca_results", atas_cpca_state, atas_cpca_checkpoint_signature
            )

            metric_name = _metric_name(res)
            metric_value = _metric_value(res)
            baseline_metric_value = float(base["selection_metric_value"])
            delta = pruned["delta_vs_baseline_primary_metric"]

            print(
                f"     {task_name:34s} {_metric_print_label(metric_name)}={metric_value:.3f} | "
                f"baseline_{metric_name}={baseline_metric_value:.3f} | "
                f"delta={delta:+.3f} | "
                f"bertF1={res.get('bertscore_f1', float('nan')):.3f}"
            )

            del res
            gc.collect()

        finally:
            if moved_clf and clf is not None and orig_device is not None:
                clf.to(orig_device)
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

_assert_no_forward_hooks(model, where="(after atas steering context)")

del executor_atas
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("forward_hooks:", _count_forward_hooks(model))

alpha_policy = alpha_of_atas_selected_task
cpca_alpha_by_task_id = {0: 0.0, 1: 0.2, 2: 0.0, 3: 1.6, 4: 2.0, 5: 2.0}
=== ATAS TRAIN CPCA on all tasks ===
all_steering_train_tf size = 600


Extracting activations:   0%|          | 0/150 [00:00<?, ?it/s]

Executor teacher forcing capped 14/600 responses at 512 tokens (longest before truncation: 926).



=== ATAS eval task=libritts_text_original_to_normalized | eval_task_alpha=0.0 | alpha_policy=alpha_of_atas_selected_task ===


[checkpoint] saved atas_cpca_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/2143919937f89e4b/atas_cpca_results.pkl
     libritts_text_original_to_normalized WER=1.434 | baseline_wer=1.444 | delta=-0.010 | bertF1=0.952

=== ATAS eval task=paradetox_non_toxic | eval_task_alpha=0.2 | alpha_policy=alpha_of_atas_selected_task ===


...


=== ATAS eval task=authormix_woolf_style | eval_task_alpha=2.0 | alpha_policy=alpha_of_atas_selected_task ===


[checkpoint] saved atas_cpca_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/2143919937f89e4b/atas_cpca_results.pkl
     authormix_woolf_style              accuracy=0.802 | baseline_accuracy=0.574 | delta=+0.228 | bertF1=0.869



=== ATAS eval task=setfit_subj_objective | eval_task_alpha=2.0 | alpha_policy=alpha_of_atas_selected_task ===


[checkpoint] saved atas_cpca_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/2143919937f89e4b/atas_cpca_results.pkl
     setfit_subj_objective              accuracy=0.631 | baseline_accuracy=0.496 | delta=+0.135 | bertF1=0.911


forward_hooks: 0


# Steering with ATAS-CMD - Random baseline

In [23]:
ATAS_RANDOM_ROUTING_SEED = SEED

id_to_task_name = {int(v): str(k) for k, v in TASK_IDS.items()}
atas_task_ids = sorted(int(x) for x in all_steering_train_tf["task_id"].dropna().unique())

missing_task_ids = [tid for tid in atas_task_ids if tid not in id_to_task_name]
if missing_task_ids:
    raise ValueError(f"task_ids present in all_steering_train_tf but missing from TASK_IDS: {missing_task_ids}")

atas_task_names = [id_to_task_name[tid] for tid in atas_task_ids]

cmd_alpha_by_task_id = {
    int(TASK_IDS[task_name]): float(cmd_best_alpha[task_name])
    for task_name in atas_task_names
}

expected_task_ids = atas_task_ids
got_task_ids = sorted(cmd_alpha_by_task_id.keys())
if got_task_ids != expected_task_ids:
    raise ValueError(
        "cmd_alpha_by_task_id does not cover every ATAS task exactly.\n"
        f"expected_task_ids={expected_task_ids}\n"
        f"got_task_ids={got_task_ids}"
    )

print("=== ATAS-RANDOM CMD TRAIN on all tasks ===")
print("all_steering_train_tf size =", len(all_steering_train_tf))
print("random_routing_seed =", ATAS_RANDOM_ROUTING_SEED)
print("alpha_policy = alpha_of_random_selected_task")
print("cmd_alpha_by_task_id =", cmd_alpha_by_task_id)

atas_random_cmd_checkpoint_signature = {
    "tasks": list(test_by_task),
    "alpha_by_task_id": dict(cmd_alpha_by_task_id),
    "base_seed": int(ATAS_RANDOM_ROUTING_SEED),
    "mediator": "ATAS_RANDOM_CMD",
}
atas_random_cmd_results: Dict[str, Dict[str, Any]] = load_experiment_checkpoint(
    "atas_random_cmd_results", {}, atas_random_cmd_checkpoint_signature
)

executor_atas_random_cmd = Executor(
    model=model,
    tokenizer=tokenizer,
    train_df=all_steering_train_tf,
    apply_from_token=-1,
    apply_from_mode=ApplyFromModeEnum.PROMPT_END,
    task_type=TaskTypeEnum.UNSUPERVISED,
    mediator_type=MediatorTypeEnum.ATAS_CMD,
    layers_to_extract=LAYERS_TO_EXTRACT,
    batch_size=4,
    use_last_token_for_response=True,
    alpha=cmd_alpha_by_task_id,
    tokens_window=-1,
    generation_kwargs=GENERATION_KWARGS,
    padding_side=tokenizer.padding_side,
    truncation_side=getattr(tokenizer, "truncation_side", "left"),
    dtype=getattr(model, "dtype", torch.float32),
    atas_use_last_token_for_prompt=ATAS_USE_LAST_TOKEN_FOR_PROMPT,
    decode_chunk_max=1,
    free_training_artifacts_after_build=True,
    atas_random_routing_seed=ATAS_RANDOM_ROUTING_SEED,
)

executor_atas_random_cmd.atas_similarity_threshold = 0.0

_assert_no_forward_hooks(model, where="(before ATAS-RANDOM CMD steering context)")

with executor_atas_random_cmd.representation_extractor() as atas_random_cmd_model:
    for task_name, df_test in test_by_task.items():
        if task_name in atas_random_cmd_results:
            print(f"[checkpoint] skipping completed ATAS-RANDOM CMD task: {task_name}")
            continue
        task_random_seed = int(ATAS_RANDOM_ROUTING_SEED + TASK_IDS[task_name])
        atas_random_cmd_model.strategy.reseed_random_routing(task_random_seed)
        eval_task_alpha = float(cmd_best_alpha[task_name])
        print(
            f"\n=== ATAS-RANDOM CMD eval task={task_name} | "
            f"eval_task_alpha={eval_task_alpha} | alpha_policy=random_selected_task ==="
        )

        clf = None
        orig_device = None
        moved_clf = False

        if _task_requires_classifier_eval(task_name):
            clf = local_classifiers[task_name]["model"]
            orig_device = get_model_device(clf)
            if torch.cuda.is_available() and orig_device.type != "cuda":
                clf.to("cuda")
                moved_clf = True

        try:
            with _active_generate_wrapper(atas_random_cmd_model) as active_model:
                res = evaluate_rewriting(
                    steered_model=active_model,
                    tokenizer_obj=tokenizer,
                    df=df_test,
                    task_name=task_name,
                    batch_size=EVAL_GENERATION_BATCH_SIZE,
                    max_new_tokens=GENERATION_KWARGS["max_new_tokens"],
                )

            pruned = prune_eval_result(res)
            pruned = _attach_primary_metrics(pruned, res)

            pruned["eval_task_alpha"] = float(eval_task_alpha)
            pruned["alpha_policy"] = "alpha_of_random_selected_task"
            pruned["alpha_by_task_id"] = dict(cmd_alpha_by_task_id)
            pruned["random_routing_seed"] = int(ATAS_RANDOM_ROUTING_SEED)
            pruned["task_random_routing_seed"] = task_random_seed

            base = baseline_results[task_name]
            pruned["baseline_metric_name"] = base["selection_metric_name"]
            pruned["baseline_metric_value"] = float(base["selection_metric_value"])
            pruned["baseline_bertscore_f1"] = float(base.get("bertscore_f1", float("nan")))
            pruned["delta_vs_baseline_primary_metric"] = (
                float(res["selection_metric_value"]) - float(base["selection_metric_value"])
            )
            pruned["improvement_vs_baseline"] = _baseline_improvement(res, base)

            atas_random_cmd_results[task_name] = pruned
            save_experiment_checkpoint(
                "atas_random_cmd_results",
                atas_random_cmd_results,
                atas_random_cmd_checkpoint_signature,
            )

            metric_name = _metric_name(res)
            metric_value = _metric_value(res)
            baseline_metric_value = float(base["selection_metric_value"])
            delta = pruned["delta_vs_baseline_primary_metric"]

            print(
                f"     {task_name:34s} {_metric_print_label(metric_name)}={metric_value:.3f} | "
                f"baseline_{metric_name}={baseline_metric_value:.3f} | "
                f"delta={delta:+.3f} | "
                f"bertF1={res.get('bertscore_f1', float('nan')):.3f}"
            )

            del res
            gc.collect()

        finally:
            if moved_clf and clf is not None and orig_device is not None:
                clf.to(orig_device)
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

_assert_no_forward_hooks(model, where="(after ATAS-RANDOM CMD steering context)")

del executor_atas_random_cmd
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("forward_hooks:", _count_forward_hooks(model))

=== ATAS-RANDOM CMD TRAIN on all tasks ===
all_steering_train_tf size = 600
random_routing_seed = 42
alpha_policy = alpha_of_random_selected_task
cmd_alpha_by_task_id = {0: 0.8, 1: 2.0, 2: 1.8, 3: 2.0, 4: 2.0, 5: 2.0}


Extracting activations:   0%|          | 0/150 [00:00<?, ?it/s]

Executor teacher forcing capped 14/600 responses at 512 tokens (longest before truncation: 926).



=== ATAS-RANDOM CMD eval task=libritts_text_original_to_normalized | eval_task_alpha=0.8 | alpha_policy=random_selected_task ===


[checkpoint] saved atas_random_cmd_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/fac4310a3e3aa135/atas_random_cmd_results.pkl
     libritts_text_original_to_normalized WER=2.130 | baseline_wer=1.444 | delta=+0.686 | bertF1=0.929


...


=== ATAS-RANDOM CMD eval task=authormix_woolf_style | eval_task_alpha=2.0 | alpha_policy=random_selected_task ===


[checkpoint] saved atas_random_cmd_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/fac4310a3e3aa135/atas_random_cmd_results.pkl
     authormix_woolf_style              accuracy=0.567 | baseline_accuracy=0.574 | delta=-0.007 | bertF1=0.898



=== ATAS-RANDOM CMD eval task=setfit_subj_objective | eval_task_alpha=2.0 | alpha_policy=random_selected_task ===


[checkpoint] saved atas_random_cmd_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/fac4310a3e3aa135/atas_random_cmd_results.pkl
     setfit_subj_objective              accuracy=0.606 | baseline_accuracy=0.496 | delta=+0.109 | bertF1=0.900


forward_hooks: 0


# Steering with ATAS-CPCA - Random baseline

In [24]:
ATAS_RANDOM_ROUTING_SEED = SEED

id_to_task_name = {int(v): str(k) for k, v in TASK_IDS.items()}
atas_task_ids = sorted(int(x) for x in all_steering_train_tf["task_id"].dropna().unique())

missing_task_ids = [tid for tid in atas_task_ids if tid not in id_to_task_name]
if missing_task_ids:
    raise ValueError(f"task_ids present in all_steering_train_tf but missing from TASK_IDS: {missing_task_ids}")

atas_task_names = [id_to_task_name[tid] for tid in atas_task_ids]

cpca_alpha_by_task_id = {
    int(TASK_IDS[task_name]): float(cpca_best_alpha[task_name])
    for task_name in atas_task_names
}

expected_task_ids = atas_task_ids
got_task_ids = sorted(cpca_alpha_by_task_id.keys())
if got_task_ids != expected_task_ids:
    raise ValueError(
        "cpca_alpha_by_task_id does not cover every ATAS task exactly.\n"
        f"expected_task_ids={expected_task_ids}\n"
        f"got_task_ids={got_task_ids}"
    )

print("=== ATAS-RANDOM CPCA TRAIN on all tasks ===")
print("all_steering_train_tf size =", len(all_steering_train_tf))
print("random_routing_seed =", ATAS_RANDOM_ROUTING_SEED)
print("alpha_policy = alpha_of_random_selected_task")
print("cpca_alpha_by_task_id =", cpca_alpha_by_task_id)

atas_random_cpca_checkpoint_signature = {
    "tasks": list(test_by_task),
    "alpha_by_task_id": dict(cpca_alpha_by_task_id),
    "base_seed": int(ATAS_RANDOM_ROUTING_SEED),
    "mediator": "ATAS_RANDOM_CPCA",
}
atas_random_cpca_results: Dict[str, Dict[str, Any]] = load_experiment_checkpoint(
    "atas_random_cpca_results", {}, atas_random_cpca_checkpoint_signature
)

executor_atas_random_cpca = Executor(
    model=model,
    tokenizer=tokenizer,
    train_df=all_steering_train_tf,
    apply_from_token=-1,
    apply_from_mode=ApplyFromModeEnum.PROMPT_END,
    task_type=TaskTypeEnum.UNSUPERVISED,
    mediator_type=MediatorTypeEnum.ATAS_CPCA,
    layers_to_extract=LAYERS_TO_EXTRACT,
    batch_size=4,
    use_last_token_for_response=True,
    alpha=cpca_alpha_by_task_id,
    tokens_window=-1,
    generation_kwargs=GENERATION_KWARGS,
    padding_side=tokenizer.padding_side,
    truncation_side=getattr(tokenizer, "truncation_side", "left"),
    dtype=getattr(model, "dtype", torch.float32),
    atas_use_last_token_for_prompt=ATAS_USE_LAST_TOKEN_FOR_PROMPT,
    decode_chunk_max=1,
    free_training_artifacts_after_build=True,
    atas_random_routing_seed=ATAS_RANDOM_ROUTING_SEED,
)

executor_atas_random_cpca.atas_similarity_threshold = 0.0

_assert_no_forward_hooks(model, where="(before ATAS-RANDOM CPCA steering context)")

with executor_atas_random_cpca.representation_extractor() as atas_random_cpca_model:
    for task_name, df_test in test_by_task.items():
        if task_name in atas_random_cpca_results:
            print(f"[checkpoint] skipping completed ATAS-RANDOM CPCA task: {task_name}")
            continue
        task_random_seed = int(ATAS_RANDOM_ROUTING_SEED + TASK_IDS[task_name])
        atas_random_cpca_model.strategy.reseed_random_routing(task_random_seed)
        eval_task_alpha = float(cpca_best_alpha[task_name])
        print(
            f"\n=== ATAS-RANDOM CPCA eval task={task_name} | "
            f"eval_task_alpha={eval_task_alpha} | alpha_policy=random_selected_task ==="
        )

        clf = None
        orig_device = None
        moved_clf = False

        if _task_requires_classifier_eval(task_name):
            clf = local_classifiers[task_name]["model"]
            orig_device = get_model_device(clf)
            if torch.cuda.is_available() and orig_device.type != "cuda":
                clf.to("cuda")
                moved_clf = True

        try:
            with _active_generate_wrapper(atas_random_cpca_model) as active_model:
                res = evaluate_rewriting(
                    steered_model=active_model,
                    tokenizer_obj=tokenizer,
                    df=df_test,
                    task_name=task_name,
                    batch_size=EVAL_GENERATION_BATCH_SIZE,
                    max_new_tokens=GENERATION_KWARGS["max_new_tokens"],
                )

            pruned = prune_eval_result(res)
            pruned = _attach_primary_metrics(pruned, res)

            pruned["eval_task_alpha"] = float(eval_task_alpha)
            pruned["alpha_policy"] = "alpha_of_random_selected_task"
            pruned["alpha_by_task_id"] = dict(cpca_alpha_by_task_id)
            pruned["random_routing_seed"] = int(ATAS_RANDOM_ROUTING_SEED)
            pruned["task_random_routing_seed"] = task_random_seed

            base = baseline_results[task_name]
            pruned["baseline_metric_name"] = base["selection_metric_name"]
            pruned["baseline_metric_value"] = float(base["selection_metric_value"])
            pruned["baseline_bertscore_f1"] = float(base.get("bertscore_f1", float("nan")))
            pruned["delta_vs_baseline_primary_metric"] = (
                float(res["selection_metric_value"]) - float(base["selection_metric_value"])
            )
            pruned["improvement_vs_baseline"] = _baseline_improvement(res, base)
            atas_random_cpca_results[task_name] = pruned
            save_experiment_checkpoint(
                "atas_random_cpca_results",
                atas_random_cpca_results,
                atas_random_cpca_checkpoint_signature,
            )

            metric_name = _metric_name(res)
            metric_value = _metric_value(res)
            baseline_metric_value = float(base["selection_metric_value"])
            delta = pruned["delta_vs_baseline_primary_metric"]

            print(
                f"     {task_name:34s} {_metric_print_label(metric_name)}={metric_value:.3f} | "
                f"baseline_{metric_name}={baseline_metric_value:.3f} | "
                f"delta={delta:+.3f} | "
                f"bertF1={res.get('bertscore_f1', float('nan')):.3f}"
            )

            del res
            gc.collect()

        finally:
            if moved_clf and clf is not None and orig_device is not None:
                clf.to(orig_device)
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

_assert_no_forward_hooks(model, where="(after ATAS-RANDOM CPCA steering context)")

del executor_atas_random_cpca
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("forward_hooks:", _count_forward_hooks(model))

=== ATAS-RANDOM CPCA TRAIN on all tasks ===
all_steering_train_tf size = 600
random_routing_seed = 42
alpha_policy = alpha_of_random_selected_task
cpca_alpha_by_task_id = {0: 0.0, 1: 0.2, 2: 0.0, 3: 1.6, 4: 2.0, 5: 2.0}


Extracting activations:   0%|          | 0/150 [00:00<?, ?it/s]

Executor teacher forcing capped 14/600 responses at 512 tokens (longest before truncation: 926).



=== ATAS-RANDOM CPCA eval task=libritts_text_original_to_normalized | eval_task_alpha=0.0 | alpha_policy=random_selected_task ===


[checkpoint] saved atas_random_cpca_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/fac4310a3e3aa135/atas_random_cpca_results.pkl
     libritts_text_original_to_normalized WER=1.708 | baseline_wer=1.444 | delta=+0.264 | bertF1=0.939


...


=== ATAS-RANDOM CPCA eval task=authormix_woolf_style | eval_task_alpha=2.0 | alpha_policy=random_selected_task ===


[checkpoint] saved atas_random_cpca_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/fac4310a3e3aa135/atas_random_cpca_results.pkl
     authormix_woolf_style              accuracy=0.568 | baseline_accuracy=0.574 | delta=-0.006 | bertF1=0.909



=== ATAS-RANDOM CPCA eval task=setfit_subj_objective | eval_task_alpha=2.0 | alpha_policy=random_selected_task ===


[checkpoint] saved atas_random_cpca_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/fac4310a3e3aa135/atas_random_cpca_results.pkl
     setfit_subj_objective              accuracy=0.510 | baseline_accuracy=0.496 | delta=+0.014 | bertF1=0.906


forward_hooks: 0


# ATAS with supervised BERT routing

The BERT router predicts one task from each complete held-out model input prompt. That prediction directly chooses both the correction vector and the validation-selected alpha associated with the predicted task. ATAS centroid routing and similarity rejection are bypassed. Both CMD and CPCA variants are evaluated.

In [20]:
id_to_task_name = {int(task_id): task_name for task_name, task_id in TASK_IDS.items()}
atas_task_ids = sorted(int(value) for value in all_steering_train_tf["task_id"].dropna().unique())
if atas_task_ids != sorted(id_to_task_name):
    raise ValueError(
        f"ATAS task IDs and task-router labels differ: ATAS={atas_task_ids}, "
        f"router={sorted(id_to_task_name)}."
    )

bert_router_prediction_signature = {
    "router_artifact_dir": task_router["artifact_dir"],
    "input_representation": "full_formatted_prompt",
    "split_sizes": {
        split_name: {task_name: len(frame) for task_name, frame in frames.items()}
        for split_name, frames in (("validation", validation_by_task), ("test", test_by_task))
    },
}
bert_router_state = load_experiment_checkpoint(
    "bert_router_predictions",
    {"predictions": {"validation": {}, "test": {}}, "diagnostics": []},
    bert_router_prediction_signature,
)
bert_router_predictions = bert_router_state["predictions"]
bert_router_diagnostics = bert_router_state["diagnostics"]
task_router["model"].to("cuda" if torch.cuda.is_available() else "cpu").eval()
for split_name, frames_by_task in (("validation", validation_by_task), ("test", test_by_task)):
    for true_task_name, frame in frames_by_task.items():
        if true_task_name in bert_router_predictions[split_name]:
            print(f"[checkpoint] skipping BERT router predictions: {split_name}/{true_task_name}")
            continue
        predicted_ids, confidences = predict_task_ids(
            frame["prompt"].astype(str).tolist(), return_conf=True
        )
        bert_router_predictions[split_name][true_task_name] = predicted_ids
        true_task_id = int(TASK_IDS[true_task_name])
        bert_router_diagnostics.append({
            "split": split_name,
            "task": true_task_name,
            "rows": len(frame),
            "routing_accuracy": float(np.mean(np.asarray(predicted_ids) == true_task_id)),
            "mean_confidence": float(np.mean(confidences)),
        })
        save_experiment_checkpoint(
            "bert_router_predictions", bert_router_state, bert_router_prediction_signature
        )
task_router["model"].to("cpu").eval()
cleanup_memory()
bert_router_diagnostics_df = pd.DataFrame(bert_router_diagnostics)
display(bert_router_diagnostics_df)

def _routing_confusion_frame(split_name: str) -> pd.DataFrame:
    true_names, predicted_names = [], []
    frames_by_task = validation_by_task if split_name == "validation" else test_by_task
    for true_task_name, frame in frames_by_task.items():
        predicted_ids = bert_router_predictions[split_name][true_task_name]
        true_names.extend([true_task_name] * len(frame))
        predicted_names.extend(id_to_task_name[int(task_id)] for task_id in predicted_ids)
    return pd.crosstab(
        pd.Series(true_names, name="true_task"),
        pd.Series(predicted_names, name="predicted_task"),
        dropna=False,
    )

print("BERT task-router test confusion matrix:")
display(_routing_confusion_frame("test"))

def run_bert_routed_atas_baseline(
        mediator_type: MediatorTypeEnum,
        selected_alpha_by_task_id: Dict[int, float],
        label: str,
) -> Dict[str, Dict[str, Any]]:
    if sorted(int(key) for key in selected_alpha_by_task_id) != atas_task_ids:
        raise ValueError(f"{label}: alpha mapping does not cover every ATAS task exactly.")

    checkpoint_name = f"bert_routed_{mediator_type.value}_results"
    checkpoint_signature = {
        "tasks": list(test_by_task),
        "selected_alpha_by_task_id": dict(selected_alpha_by_task_id),
        "router_artifact_dir": task_router["artifact_dir"],
        "mediator": mediator_type.value,
        "alpha_policy": "validation_selected_alpha_of_bert_predicted_task",
        "input_representation": "full_formatted_prompt",
    }
    results: Dict[str, Dict[str, Any]] = load_experiment_checkpoint(
        checkpoint_name, {}, checkpoint_signature
    )
    if set(results) == set(test_by_task):
        print(f"[checkpoint] all {label} tasks already completed.")
        return results

    executor = Executor(
        model=model,
        tokenizer=tokenizer,
        train_df=all_steering_train_tf.copy(),
        apply_from_token=-1,
        apply_from_mode=ApplyFromModeEnum.PROMPT_END,
        task_type=TaskTypeEnum.UNSUPERVISED,
        mediator_type=mediator_type,
        layers_to_extract=LAYERS_TO_EXTRACT,
        atas_master_layer=LLAMA_ROBUSTNESS_LAYER,
        batch_size=4,
        alpha=dict(selected_alpha_by_task_id),
        use_last_token_for_response=True,
        tokens_window=-1,
        generation_kwargs=GENERATION_KWARGS,
        padding_side=tokenizer.padding_side,
        truncation_side=getattr(tokenizer, "truncation_side", "left"),
        dtype=getattr(model, "dtype", torch.float32),
        atas_similarity_threshold=0.0,
        atas_use_last_token_for_prompt=ATAS_USE_LAST_TOKEN_FOR_PROMPT,
    )
    executor.atas_similarity_threshold = 0.0
    _assert_no_forward_hooks(model, where=f"(before {label})")
    try:
        with executor.representation_extractor() as routed_model:
            strategy = getattr(routed_model, "strategy", None)
            if strategy is None or not hasattr(strategy, "set_forced_task_ids"):
                raise RuntimeError("The installed pysteer version lacks supervised task routing support.")

            for task_name, test_frame in test_by_task.items():
                if task_name in results:
                    print(f"[checkpoint] skipping completed {label} task: {task_name}")
                    continue
                predicted_task_ids = bert_router_predictions["test"][task_name]
                if len(predicted_task_ids) != len(test_frame):
                    raise RuntimeError(f"{task_name}: router predictions and test rows are misaligned.")

                true_task_id = int(TASK_IDS[task_name])
                nominal_task_alpha = float(selected_alpha_by_task_id[true_task_id])
                routed_model.alpha = dict(selected_alpha_by_task_id)
                applied_alphas = np.asarray([
                    selected_alpha_by_task_id[int(predicted_task_id)]
                    for predicted_task_id in predicted_task_ids
                ], dtype=float)
                routing_accuracy = float(np.mean(np.asarray(predicted_task_ids) == true_task_id))
                print(
                    f"\n=== {label} task={task_name} | "
                    f"nominal_task_alpha={nominal_task_alpha} | "
                    f"applied_alpha_mean={applied_alphas.mean():.3f} | "
                    f"supervised_routing_accuracy={routing_accuracy:.3f} ==="
                )

                clf = None
                original_device = None
                moved_classifier = False
                if _task_requires_classifier_eval(task_name):
                    clf = local_classifiers[task_name]["model"]
                    original_device = get_model_device(clf)
                    if torch.cuda.is_available() and original_device.type != "cuda":
                        clf.to("cuda")
                        moved_classifier = True

                try:
                    with _active_generate_wrapper(routed_model) as active_model:
                        result = evaluate_rewriting(
                            steered_model=active_model,
                            tokenizer_obj=tokenizer,
                            df=test_frame,
                            task_name=task_name,
                            batch_size=EVAL_GENERATION_BATCH_SIZE,
                            max_new_tokens=GENERATION_KWARGS["max_new_tokens"],
                            forced_task_ids=predicted_task_ids,
                        )

                    pruned = _attach_primary_metrics(prune_eval_result(result), result)
                    pruned["routing_policy"] = "supervised_bert_predicted_task"
                    pruned["routing_accuracy"] = routing_accuracy
                    pruned["alpha_policy"] = "validation_selected_alpha_of_bert_predicted_task"
                    pruned["eval_task_alpha"] = nominal_task_alpha
                    pruned["applied_alpha_mean"] = float(applied_alphas.mean())
                    pruned["applied_alpha_std"] = float(applied_alphas.std(ddof=1)) if len(applied_alphas) > 1 else 0.0
                    pruned["selected_alpha_by_task_id"] = dict(selected_alpha_by_task_id)
                    pruned["similarity_routing_bypassed"] = True
                    pruned["similarity_rejection_bypassed"] = True
                    base = baseline_results[task_name]
                    pruned["baseline_metric_name"] = base["selection_metric_name"]
                    pruned["baseline_metric_value"] = float(base["selection_metric_value"])
                    pruned["improvement_vs_baseline"] = _baseline_improvement(result, base)
                    results[task_name] = pruned
                    save_experiment_checkpoint(checkpoint_name, results, checkpoint_signature)

                    metric_name = _metric_name(result)
                    print(
                        f"  => TEST {_metric_print_label(metric_name)}={_metric_value(result):.3f} | "
                        f"nominal_task_alpha={nominal_task_alpha} | "
                        f"routing_accuracy={routing_accuracy:.3f} | "
                        f"bertF1={result.get('bertscore_f1', float('nan')):.3f}"
                    )
                    del result
                finally:
                    if moved_classifier and clf is not None and original_device is not None:
                        clf.to(original_device)
                    cleanup_memory()
    finally:
        del executor
        cleanup_memory()
        _assert_no_forward_hooks(model, where=f"(after {label})")

    return results

bert_cmd_alpha_by_task_id = {
    int(task_id): float(cmd_best_alpha[task_name])
    for task_name, task_id in TASK_IDS.items()
}
bert_cpca_alpha_by_task_id = {
    int(task_id): float(cpca_best_alpha[task_name])
    for task_name, task_id in TASK_IDS.items()
}
bert_routed_cmd_results = run_bert_routed_atas_baseline(
    MediatorTypeEnum.ATAS_CMD, bert_cmd_alpha_by_task_id, "BERT-ROUTED ATAS-CMD"
)
bert_routed_cpca_results = run_bert_routed_atas_baseline(
    MediatorTypeEnum.ATAS_CPCA, bert_cpca_alpha_by_task_id, "BERT-ROUTED ATAS-CPCA"
)
bert_routed_results_df = pd.DataFrame([
    {
        "method": method_name,
        "task": task_name,
        "metric": result["selection_metric_name"],
        "value": result["selection_metric_value"],
        "nominal_task_alpha": result["eval_task_alpha"],
        "applied_alpha_mean": result["applied_alpha_mean"],
        "applied_alpha_std": result["applied_alpha_std"],
        "routing_accuracy": result["routing_accuracy"],
        "improvement_vs_unsteered": result["improvement_vs_baseline"],
    }
    for method_name, method_results in (
        ("BERT-ROUTED ATAS-CMD", bert_routed_cmd_results),
        ("BERT-ROUTED ATAS-CPCA", bert_routed_cpca_results),
    )
    for task_name, result in method_results.items()
])
display(bert_routed_results_df)
print("forward_hooks:", _count_forward_hooks(model))

steering_result_groups = [
    ("UNSTEERED", baseline_results),
    ("CMD", cmd_results),
    ("CPCA", cpca_results),
    ("ATAS-CMD", atas_cmd_results),
    ("ATAS-CPCA", atas_cpca_results),
    ("ATAS-RANDOM-CMD", atas_random_cmd_results),
    ("ATAS-RANDOM-CPCA", atas_random_cpca_results),
    ("BERT-ROUTED-ATAS-CMD", bert_routed_cmd_results),
    ("BERT-ROUTED-ATAS-CPCA", bert_routed_cpca_results),
]
expected_tasks = set(test_by_task)
incomplete_groups = {
    method: sorted(expected_tasks - set(method_results))
    for method, method_results in steering_result_groups
    if set(method_results) != expected_tasks
}
if incomplete_groups:
    raise RuntimeError(f"Cannot build the final table; incomplete results: {incomplete_groups}")

overall_rows = []
for method, method_results in steering_result_groups:
    for task_name in test_by_task:
        result = method_results[task_name]
        metric_value = float(result["selection_metric_value"])
        baseline_value = float(baseline_results[task_name]["selection_metric_value"])
        overall_rows.append({
            "result_type": "rewriting",
            "method": method,
            "split": "test",
            "task": task_name,
            "metric": result["selection_metric_name"],
            "value": metric_value,
            "selected_alpha": result.get("selected_alpha", result.get("eval_task_alpha")),
            "alpha_policy": result.get("alpha_policy"),
            "alpha_by_task_id": result.get("selected_alpha_by_task_id", result.get("alpha_by_task_id")),
            "applied_alpha_mean": result.get("applied_alpha_mean"),
            "applied_alpha_std": result.get("applied_alpha_std"),
            "routing_accuracy": result.get("routing_accuracy"),
            "bertscore_f1": result.get("bertscore_f1"),
            "refusal_count": result.get("refusal_count"),
            "refusal_rate": result.get("refusal_rate"),
            "baseline_value": baseline_value,
            "improvement_vs_unsteered": (
                0.0 if method == "UNSTEERED" else result.get("improvement_vs_baseline")
            ),
        })

for router_metric in task_router_metrics:
    overall_rows.append({
        "result_type": "routing",
        "method": "BERT-TASK-ROUTER",
        "split": router_metric["split"],
        "task": "ALL",
        "metric": "accuracy",
        "value": router_metric["accuracy"],
        "f1_macro": router_metric["f1_macro"],
        "rows": router_metric["rows"],
    })

for classifier_result in all_results:
    for metric_name in ("test_acc", "test_f1"):
        if metric_name not in classifier_result:
            continue
        overall_rows.append({
            "result_type": "classifier",
            "method": "DISTILBERT-TASK-EVALUATOR",
            "split": "test",
            "task": classifier_result["dataset"],
            "metric": metric_name.removeprefix("test_"),
            "value": classifier_result[metric_name],
        })

for diagnostic in bert_router_diagnostics:
    overall_rows.append({
        "result_type": "routing",
        "method": "BERT-TASK-ROUTER",
        "split": diagnostic["split"],
        "task": diagnostic["task"],
        "metric": "routing_accuracy",
        "value": diagnostic["routing_accuracy"],
        "mean_confidence": diagnostic["mean_confidence"],
        "rows": diagnostic["rows"],
    })

overall_results_table = pd.DataFrame(overall_rows)
overall_results_signature = {
    "methods": [method for method, _ in steering_result_groups],
    "tasks": list(test_by_task),
}
save_experiment_checkpoint(
    "overall_results_table", overall_results_table, overall_results_signature
)
overall_results_csv = CHECKPOINT_DIR / "overall_results.csv"
overall_results_csv_tmp = overall_results_csv.with_suffix(".csv.tmp")
with overall_results_csv_tmp.open("w", encoding="utf-8", newline="") as handle:
    overall_results_table.to_csv(handle, index=False)
    handle.flush()
    os.fsync(handle.fileno())
os.replace(overall_results_csv_tmp, overall_results_csv)
print("Overall results saved to:", overall_results_csv)
display(overall_results_table)

[checkpoint] saved bert_router_predictions -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/2143919937f89e4b/bert_router_predictions.pkl


[checkpoint] saved bert_router_predictions -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/2143919937f89e4b/bert_router_predictions.pkl


[checkpoint] saved bert_router_predictions -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/2143919937f89e4b/bert_router_predictions.pkl


[checkpoint] saved bert_router_predictions -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/2143919937f89e4b/bert_router_predictions.pkl


[checkpoint] saved bert_router_predictions -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/2143919937f89e4b/bert_router_predictions.pkl


...


=== BERT-ROUTED ATAS-CPCA task=setfit_subj_objective | nominal_task_alpha=2.0 | applied_alpha_mean=2.000 | supervised_routing_accuracy=1.000 ===


[checkpoint] saved bert_routed_atas_cpca_results -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/2143919937f89e4b/bert_routed_atas_cpca_results.pkl
  => TEST accuracy=0.631 | nominal_task_alpha=2.0 | routing_accuracy=1.000 | bertF1=0.911


,method,task,metric,value,nominal_task_alpha,applied_alpha_mean,applied_alpha_std,routing_accuracy,improvement_vs_unsteered
0,BERT-ROUTED ATAS-CMD,libritts_text_original_to_normalized,wer,1.115190,0.8,0.8,0.000000e+00,1.0,0.328856
1,BERT-ROUTED ATAS-CMD,paradetox_non_toxic,accuracy,0.984736,2.0,2.0,0.000000e+00,1.0,0.011964
2,BERT-ROUTED ATAS-CMD,imdb_positive_sentiment,accuracy,0.996571,1.8,1.8,2.220558e-16,1.0,0.089552
3,BERT-ROUTED ATAS-CMD,stanford_politeness_polite,accuracy,0.988593,2.0,2.0,0.000000e+00,1.0,0.053232
4,BERT-ROUTED ATAS-CMD,authormix_woolf_style,accuracy,0.899924,2.0,2.0,0.000000e+00,1.0,0.326005
5,BERT-ROUTED ATAS-CMD,setfit_subj_objective,accuracy,0.748500,2.0,2.0,0.000000e+00,1.0,0.252000
6,BERT-ROUTED ATAS-CPCA,libritts_text_original_to_normalized,wer,1.442208,0.0,0.0,0.000000e+00,1.0,0.001838
7,BERT-ROUTED ATAS-CPCA,paradetox_non_toxic,accuracy,0.968441,0.2,0.2,2.775844e-17,1.0,-0.004332
8,BERT-ROUTED ATAS-CPCA,imdb_positive_sentiment,accuracy,0.907019,0.0,0.0,0.000000e+00,1.0,0.000000
9,BERT-ROUTED ATAS-CPCA,stanford_politeness_polite,accuracy,0.992395,1.6,1.6,2.224680e-16,1.0,0.057034


forward_hooks: 0
[checkpoint] saved overall_results_table -> /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/2143919937f89e4b/overall_results_table.pkl
Overall results saved to: /home/ubuntu/ATAS/.atas_cache/atas_artifacts/experiment_checkpoints/2143919937f89e4b/overall_results.csv


,result_type,method,split,task,metric,value,selected_alpha,alpha_policy,alpha_by_task_id,applied_alpha_mean,applied_alpha_std,routing_accuracy,bertscore_f1,refusal_count,refusal_rate,baseline_value,improvement_vs_unsteered,f1_macro,rows,mean_confidence
0,rewriting,UNSTEERED,test,libritts_text_original_to_normalized,wer,1.444046,NaN,None,None,NaN,NaN,NaN,0.952517,NaN,NaN,1.444046,0.0,NaN,NaN,NaN
1,rewriting,UNSTEERED,test,paradetox_non_toxic,accuracy,0.972772,NaN,None,None,NaN,NaN,NaN,0.919803,NaN,NaN,0.972772,0.0,NaN,NaN,NaN
2,rewriting,UNSTEERED,test,imdb_positive_sentiment,accuracy,0.907019,NaN,None,None,NaN,NaN,NaN,0.926345,NaN,NaN,0.907019,0.0,NaN,NaN,NaN
3,rewriting,UNSTEERED,test,stanford_politeness_polite,accuracy,0.935361,NaN,None,None,NaN,NaN,NaN,0.920096,NaN,NaN,0.935361,0.0,NaN,NaN,NaN
4,rewriting,UNSTEERED,test,authormix_woolf_style,accuracy,0.573920,NaN,None,None,NaN,NaN,NaN,0.917810,NaN,NaN,0.573920,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,routing,BERT-TASK-ROUTER,test,paradetox_non_toxic,routing_accuracy,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4848.0,0.999764
76,routing,BERT-TASK-ROUTER,test,imdb_positive_sentiment,routing_accuracy,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9916.0,0.999872
77,routing,BERT-TASK-ROUTER,test,stanford_politeness_polite,routing_accuracy,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,263.0,0.997395
78,routing,BERT-TASK-ROUTER,test,authormix_woolf_style,routing_accuracy,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1319.0,0.999216
